# Land-source context prototype

This notebook assembles **source-side evidence** for the canonical H3 R7 land viewshed and publishes a prototype relative land reporting-opportunity index at daily H3 R6 grain. It does not claim true observer effort or detection probability.

Contract:

- The unique `source_h3` values in `LAND_STATIC_WEIGHTS_R7.parquet` define the complete left-hand universe.
- Exact-cell population, source-centered population catchments, and public-shore access remain separately inspectable evidence.
- An absent or unavailable row remains null and is never converted to an observed zero.
- Calendar context remains a separate one-row-per-date table and joins source context only by date.
- Daily HRRR weather and daylight map to land observer/source cells through explicit H3 parents; missing spatial support remains unavailable.
- Whale sightings are not loaded until after every pressure variant is constructed.
- Prototype outputs are written under `outputs/`, not the durable processed-data contract.

In [ ]:
from __future__ import annotations

import hashlib
import json
import time
from datetime import datetime, timezone
from pathlib import Path

import h3
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import requests
from IPython.display import display
from scipy import sparse
from scipy.spatial import cKDTree


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "configs/salish_sea.yaml").is_file():
            return candidate
    raise FileNotFoundError("Could not locate the Viewshed Toolkit repository root.")


REPO_ROOT = find_repo_root()
LAND_VIEWSHED_PATH = REPO_ROOT / "data/processed/domain/human/viewshed/RES7/LAND_STATIC_WEIGHTS_R7.parquet"
POPULATION_PATH = REPO_ROOT / "data/processed/domain/human/demography_and_presence/population/population_context_h3_r7.parquet"
PUBLIC_SHORE_PATH = REPO_ROOT / "data/processed/domain/human/accessibility/public_shore_access/public_shore_access_h3_r7.parquet"
CALENDAR_PATH = REPO_ROOT / "data/processed/domain/human/temporal_context/calendar/calendar_daily.parquet"
SURFACE_WEATHER_DIR = REPO_ROOT / "data/processed/domain/environmental_layer/meteorological/surface_weather/H3_SURFACE_WEATHER_DAILY_RES_5"
SURFACE_WEATHER_MANIFEST_PATH = REPO_ROOT / "data/processed/domain/environmental_layer/meteorological/surface_weather/MANIFEST.json"
DAYLIGHT_DIR = REPO_ROOT / "data/processed/domain/environmental_layer/meteorological/daylight/H3_DAYLIGHT_DAILY_RES_4"
DAYLIGHT_MANIFEST_PATH = REPO_ROOT / "data/processed/domain/environmental_layer/meteorological/daylight/MANIFEST.json"
SIGHTINGS_PATH = REPO_ROOT / "data/processed/domain/whale_layer/sightings/observations.parquet"
PROTOTYPE_DIR = REPO_ROOT / "outputs/effort/land_source_context"
PROTOTYPE_PATH = PROTOTYPE_DIR / "land_source_context_h3_r7_prototype.parquet"
COVERAGE_PATH = PROTOTYPE_DIR / "land_source_context_coverage.csv"
METADATA_PATH = PROTOTYPE_DIR / "land_source_context_h3_r7_prototype.metadata.json"
TARGET_PRESSURE_PATH = PROTOTYPE_DIR / "target_land_viewing_pressure_h3_r7_prototype.parquet"
TARGET_PRESSURE_METADATA_PATH = PROTOTYPE_DIR / "target_land_viewing_pressure_h3_r7_prototype.metadata.json"
SIGHTINGS_COMPARISON_PATH = PROTOTYPE_DIR / "sightings_vs_land_viewing_pressure_h3_r7.parquet"
SIGHTINGS_COMPARISON_METADATA_PATH = PROTOTYPE_DIR / "sightings_vs_land_viewing_pressure_h3_r7.metadata.json"
SIGHTINGS_DECILES_PATH = PROTOTYPE_DIR / "sightings_vs_land_viewing_pressure_deciles.csv"
SIGHTINGS_PLOT_PATH = PROTOTYPE_DIR / "sightings_vs_land_viewing_pressure.png"
POPULATION_CATCHMENT_SUMMARY_PATH = PROTOTYPE_DIR / "source_population_catchment_summary.csv"
PRESSURE_VARIANT_ASSOCIATIONS_PATH = PROTOTYPE_DIR / "target_pressure_variant_associations.csv"
TEMPORAL_HOLDOUT_ASSOCIATIONS_PATH = PROTOTYPE_DIR / "target_pressure_temporal_holdout_associations.csv"
SIGHTINGS_VARIANTS_PLOT_PATH = PROTOTYPE_DIR / "sightings_vs_land_viewing_pressure_variants.png"
TRANSPORT_ACCESS_DIR = PROTOTYPE_DIR / "transport_access"
TRANSPORT_ROUTING_CACHE_DIR = TRANSPORT_ACCESS_DIR / "osrm_table_cache"
SOURCE_TRANSPORT_ACCESS_PATH = TRANSPORT_ACCESS_DIR / "land_source_transport_access_h3_r7_prototype.parquet"
SOURCE_TRANSPORT_SUMMARY_PATH = TRANSPORT_ACCESS_DIR / "land_source_transport_access_summary.csv"
SOURCE_TRANSPORT_METADATA_PATH = TRANSPORT_ACCESS_DIR / "land_source_transport_access_h3_r7_prototype.metadata.json"
POPULATION_TRAVEL_DIR = PROTOTYPE_DIR / "population_travel_time"
POPULATION_TRAVEL_ROUTING_CACHE_DIR = POPULATION_TRAVEL_DIR / "osrm_population_table_cache"
POPULATION_TRAVEL_ORIGINS_PATH = POPULATION_TRAVEL_DIR / "population_travel_origins_h3_r4_prototype.parquet"
SOURCE_POPULATION_TRAVEL_PATH = POPULATION_TRAVEL_DIR / "land_source_population_travel_demand_h3_r7_prototype.parquet"
SOURCE_POPULATION_TRAVEL_SUMMARY_PATH = POPULATION_TRAVEL_DIR / "land_source_population_travel_demand_summary.csv"
SOURCE_POPULATION_TRAVEL_METADATA_PATH = POPULATION_TRAVEL_DIR / "land_source_population_travel_demand_h3_r7_prototype.metadata.json"
DYNAMIC_VIEWABILITY_DIR = PROTOTYPE_DIR / "dynamic_viewability"
DAILY_OPPORTUNITY_PATH = DYNAMIC_VIEWABILITY_DIR / "daily_land_reporting_opportunity_vs_sightings_prototype.parquet"
DAILY_CORRELATIONS_PATH = DYNAMIC_VIEWABILITY_DIR / "daily_land_reporting_opportunity_correlations.csv"
DAILY_OPPORTUNITY_METADATA_PATH = DYNAMIC_VIEWABILITY_DIR / "daily_land_reporting_opportunity_vs_sightings_prototype.metadata.json"
DAILY_CORRELATION_PLOT_PATH = DYNAMIC_VIEWABILITY_DIR / "daily_land_reporting_opportunity_correlations.png"
LAND_EFFORT_GRID_PATH = DYNAMIC_VIEWABILITY_DIR / "land_based_effort_proxy_daily_h3_r6.parquet"
LAND_EFFORT_GRID_METADATA_PATH = DYNAMIC_VIEWABILITY_DIR / "land_based_effort_proxy_daily_h3_r6.metadata.json"

POPULATION_CAP_QUANTILE = 0.99
POPULATION_CATCHMENT_RADII_KM = (5, 10, 25, 50)
POPULATION_DECAY_RADII_KM = (10, 25, 50)
PRIMARY_POPULATION_VARIANT = "DECAYED_25_KM"
EARTH_MEAN_RADIUS_KM = 6371.0088
ROAD_DISTANCE_DECAY_KM = 5.0
CITY_TRAVEL_TIME_DECAY_MINUTES = 120.0
POPULATION_TRAVEL_ORIGIN_RESOLUTION = 4
POPULATION_TRAVEL_RETAINED_SHARE = 0.99
POPULATION_TRAVEL_LOCAL_ORIGIN_DISTANCE_KM = 25.0
POPULATION_TRAVEL_DECAY_MINUTES = (60, 120, 240)
POPULATION_TRAVEL_WITHIN_MINUTES = (60, 120, 240)
PRIMARY_POPULATION_TRAVEL_DECAY_MINUTES = 120
POPULATION_TRAVEL_MIN_ROUTED_SHARE = 0.99
VISIBILITY_TRANSITION_FRACTION = 0.20
VISIBILITY_MINIMUM_TRANSITION_KM = 1.0
WIND_SUPPORT_MIDPOINT_MS = 5.5
WIND_SUPPORT_SLOPE_MS = 1.5
PRECIP_SUPPORT_HALF_MM_DAY = 10.0
CONDITIONS_WIND_EXPONENT = 0.70
CONDITIONS_PRECIP_EXPONENT = 0.30
DAILY_DISTANCE_BIN_KM = 0.25
DAILY_ROLLING_WINDOW_DAYS = 28
LAND_EFFORT_TARGET_RESOLUTION = 6
LAND_EFFORT_REFERENCE_WEEKS = 104
LAND_EFFORT_SCALE_QUANTILE = 0.99
LAND_EFFORT_DATE_CHUNK_DAYS = 92
SIGHTINGS_START_DATE = pd.Timestamp("2020-01-01")
SIGHTINGS_END_DATE = pd.Timestamp("2026-06-25")

print(f"Repository: {REPO_ROOT}")

## Inputs and provenance

Every input must exist before assembly. Checksums are captured in the prototype metadata so a later production builder can reproduce the exact join.

In [ ]:
INPUT_PATHS = {
    "land_viewshed": LAND_VIEWSHED_PATH,
    "population_context": POPULATION_PATH,
    "public_shore_access": PUBLIC_SHORE_PATH,
    "calendar_context": CALENDAR_PATH,
    "surface_weather_manifest": SURFACE_WEATHER_MANIFEST_PATH,
    "daylight_manifest": DAYLIGHT_MANIFEST_PATH,
}

INPUT_DATASET_DIRS = {
    "surface_weather_daily": SURFACE_WEATHER_DIR,
    "daylight_daily": DAYLIGHT_DIR,
}

missing_inputs = [str(path) for path in INPUT_PATHS.values() if not path.is_file()]
missing_inputs.extend(
    str(path) for path in INPUT_DATASET_DIRS.values() if not path.is_dir()
)
if missing_inputs:
    raise FileNotFoundError(f"Missing required inputs: {missing_inputs}")


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


input_inventory = pd.DataFrame(
    [
        {
            "input": name,
            "path": str(path.relative_to(REPO_ROOT)),
            "bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        }
        for name, path in INPUT_PATHS.items()
    ]
)
display(input_inventory)

In [ ]:
land_pairs = pd.read_parquet(
    LAND_VIEWSHED_PATH,
    columns=["source_h3", "target_h3", "weight_static_viewability"],
)
population = pd.read_parquet(POPULATION_PATH)
public_shore = pd.read_parquet(PUBLIC_SHORE_PATH)
calendar = pd.read_parquet(CALENDAR_PATH)

required_columns = {
    "land_viewshed": {"source_h3", "target_h3", "weight_static_viewability"},
    "population_context": {
        "H3_INDEX",
        "H3_RESOLUTION",
        "POPULATION",
        "POPULATION_LOG1P",
        "POPULATION_CONTEXT_AVAILABLE",
        "MARINE_TRANSFER_APPLIED",
        "MEASUREMENT_STATUS",
    },
    "public_shore_access": {
        "H3_INDEX",
        "H3_RESOLUTION",
        "PUBLIC_ACCESSIBLE_SHORELINE_M",
        "ACCESSIBLE_WATERFRONT_FRACTION",
        "PUBLIC_ACCESS_STATE",
        "SOURCE_COVERAGE_COMPLETE",
        "FRACTION_DENOMINATOR_MISMATCH_QC",
        "MEASUREMENT_STATUS",
    },
    "calendar_context": {"date", "calendar_effort_multiplier", "calendar_effort_weight"},
}
frames = {
    "land_viewshed": land_pairs,
    "population_context": population,
    "public_shore_access": public_shore,
    "calendar_context": calendar,
}
for name, required in required_columns.items():
    missing = required - set(frames[name].columns)
    if missing:
        raise ValueError(f"{name} is missing required columns: {sorted(missing)}")

schema_summary = pd.DataFrame(
    [
        {"input": name, "rows": len(frame), "columns": len(frame.columns)}
        for name, frame in frames.items()
    ]
)
display(schema_summary)

## Canonical land-source universe

The pair table is reduced to one row per land source. Pair counts are retained only as viewshed-support diagnostics; they are not normalized into probabilities.

In [ ]:
duplicate_pairs = int(land_pairs.duplicated(["source_h3", "target_h3"]).sum())
if duplicate_pairs:
    raise ValueError(f"Canonical land viewshed contains {duplicate_pairs:,} duplicate source-target pairs.")

land_sources = (
    land_pairs.groupby("source_h3", as_index=False)
    .agg(
        VIEWSHED_TARGET_PAIR_COUNT=("target_h3", "size"),
        VIEWSHED_TARGET_COUNT=("target_h3", "nunique"),
    )
    .sort_values("source_h3")
    .reset_index(drop=True)
)
land_sources.insert(1, "H3_RESOLUTION", 7)

if not land_sources["source_h3"].is_unique:
    raise ValueError("Land source universe is not unique by source_h3.")
if not land_sources["source_h3"].map(h3.get_resolution).eq(7).all():
    raise ValueError("Land source universe contains a non-R7 H3 index.")
if not land_sources["VIEWSHED_TARGET_PAIR_COUNT"].eq(land_sources["VIEWSHED_TARGET_COUNT"]).all():
    raise ValueError("Source-target uniqueness failed during source aggregation.")

print(f"Land source cells: {len(land_sources):,}")
print(f"Canonical source-target pairs: {len(land_pairs):,}")
display(land_sources.head())

## Join population and mapped-access evidence

Both context tables must be unique by H3 cell. Match flags distinguish a joined row from a missing row. Public-shore coverage is partial, so an unmatched cell means **no mapped context row**, not no public access.

In [ ]:
if not population["H3_INDEX"].is_unique:
    raise ValueError("Population context is not unique by H3_INDEX.")
if not public_shore["H3_INDEX"].is_unique:
    raise ValueError("Public-shore context is not unique by H3_INDEX.")
if not population["H3_RESOLUTION"].eq(7).all():
    raise ValueError("Population context contains a non-R7 row.")
if not public_shore["H3_RESOLUTION"].eq(7).all():
    raise ValueError("Public-shore context contains a non-R7 row.")

population_join = (
    population.drop(columns=["H3_RESOLUTION"])
    .rename(
        columns={
            "H3_INDEX": "source_h3",
            "CONTEXT_SCOPE": "POPULATION_CONTEXT_SCOPE",
            "MEASUREMENT_STATUS": "POPULATION_MEASUREMENT_STATUS",
        }
    )
    .assign(_POPULATION_JOIN_ROW=True)
)
public_shore_join = (
    public_shore.drop(columns=["H3_RESOLUTION"])
    .rename(
        columns={
            "H3_INDEX": "source_h3",
            "MEASUREMENT_STATUS": "PUBLIC_SHORE_MEASUREMENT_STATUS",
        }
    )
    .assign(_PUBLIC_SHORE_JOIN_ROW=True)
)

land_source_context = land_sources.merge(
    population_join, on="source_h3", how="left", validate="one_to_one"
).merge(
    public_shore_join, on="source_h3", how="left", validate="one_to_one"
)
land_source_context["POPULATION_CONTEXT_MATCHED"] = land_source_context.pop(
    "_POPULATION_JOIN_ROW"
).eq(True)
land_source_context["PUBLIC_SHORE_CONTEXT_MATCHED"] = land_source_context.pop(
    "_PUBLIC_SHORE_JOIN_ROW"
).eq(True)

land_source_context = land_source_context.sort_values("source_h3").reset_index(drop=True)

In [ ]:
if len(land_source_context) != len(land_sources):
    raise ValueError("Context joins changed the land-source row count.")
if not land_source_context["source_h3"].is_unique:
    raise ValueError("Joined context is not unique by source_h3.")
if "target_h3" in land_source_context.columns:
    raise ValueError("Source context must not retain target_h3.")

population_missing = ~land_source_context["POPULATION_CONTEXT_MATCHED"]
shore_missing = ~land_source_context["PUBLIC_SHORE_CONTEXT_MATCHED"]
if not land_source_context.loc[population_missing, ["POPULATION", "POPULATION_LOG1P"]].isna().all().all():
    raise ValueError("Unmatched population cells were not preserved as null.")
if not land_source_context.loc[shore_missing, ["PUBLIC_ACCESSIBLE_SHORELINE_M", "ACCESSIBLE_WATERFRONT_FRACTION"]].isna().all().all():
    raise ValueError("Unmatched public-shore cells were not preserved as null.")

denominator_mismatch = land_source_context["FRACTION_DENOMINATOR_MISMATCH_QC"].eq(True)
if not land_source_context.loc[denominator_mismatch, "ACCESSIBLE_WATERFRONT_FRACTION"].isna().all():
    raise ValueError("Denominator-mismatch cells must retain null modeled fractions.")
population_matched = land_source_context["POPULATION_CONTEXT_MATCHED"]
if land_source_context.loc[population_matched, "MARINE_TRANSFER_APPLIED"].eq(True).any():
    raise ValueError("Population context unexpectedly contains marine transfer.")

print("Join and missingness contracts passed.")

In [ ]:
source_count = len(land_source_context)
coverage_summary = pd.DataFrame(
    [
        {"measure": "canonical_land_sources", "count": source_count},
        {"measure": "population_context_matched", "count": int(land_source_context["POPULATION_CONTEXT_MATCHED"].sum())},
        {"measure": "population_context_unmatched", "count": int((~land_source_context["POPULATION_CONTEXT_MATCHED"]).sum())},
        {"measure": "population_positive", "count": int(land_source_context["POPULATION"].gt(0).sum())},
        {"measure": "population_observed_zero", "count": int(land_source_context["POPULATION"].eq(0).sum())},
        {"measure": "public_shore_context_matched", "count": int(land_source_context["PUBLIC_SHORE_CONTEXT_MATCHED"].sum())},
        {"measure": "public_shore_context_unmatched", "count": int((~land_source_context["PUBLIC_SHORE_CONTEXT_MATCHED"]).sum())},
        {"measure": "public_access_state_observed", "count": int(land_source_context["PUBLIC_ACCESS_STATE"].eq("observed_access").sum())},
        {"measure": "public_access_state_unknown", "count": int(land_source_context["PUBLIC_ACCESS_STATE"].eq("unknown").sum())},
        {"measure": "shore_fraction_denominator_mismatch", "count": int(denominator_mismatch.sum())},
    ]
)
coverage_summary["fraction_of_land_sources"] = coverage_summary["count"] / source_count
display(coverage_summary)

In [ ]:
display_columns = [
    "source_h3",
    "H3_RESOLUTION",
    "VIEWSHED_TARGET_COUNT",
    "POPULATION",
    "POPULATION_LOG1P",
    "POPULATION_CONTEXT_MATCHED",
    "JURISDICTION",
    "PUBLIC_ACCESS_STATE",
    "PUBLIC_ACCESS_SITE_COUNT",
    "ACCESSIBLE_WATERFRONT_FRACTION",
    "PUBLIC_SHORE_CONTEXT_MATCHED",
]
display(land_source_context[display_columns].head(12))

print("Unmatched population source cells:")
display(land_source_context.loc[population_missing, display_columns])

print("Public-shore access states among matched source cells:")
display(
    land_source_context.loc[land_source_context["PUBLIC_SHORE_CONTEXT_MATCHED"], "PUBLIC_ACCESS_STATE"]
    .value_counts(dropna=False)
    .rename_axis("PUBLIC_ACCESS_STATE")
    .reset_index(name="source_cells")
)

## Calendar remains separate

Calendar is validated here as a one-row-per-date context table. It will be joined by date only when a daily or weekly reporting-opportunity component is assembled; this notebook does not create a date × H3 cross-product.

In [ ]:
if not calendar["date"].is_unique:
    raise ValueError("Calendar context is not unique by date.")
if any("H3" in column.upper() for column in calendar.columns):
    raise ValueError("Calendar context unexpectedly contains an H3 field.")

calendar_summary = pd.DataFrame(
    [
        {
            "rows": len(calendar),
            "minimum_date": calendar["date"].min(),
            "maximum_date": calendar["date"].max(),
            "duplicate_dates": int(calendar["date"].duplicated().sum()),
            "h3_columns": 0,
        }
    ]
)
display(calendar_summary)

## Connect existing weather and daylight context

The canonical HRRR daily weather collection is H3 R5 and the deterministic daylight collection is H3 R4. Each H3 R7 land observer/source cell is mapped to its containing R5 weather and R4 daylight cell. Calendar joins by date only. This validates the connection for one date without materializing the full source × date or source × target × date cube, and uncovered source cells remain unavailable rather than becoming zero.

In [ ]:
WEATHER_COLUMNS = [
    "H3_INDEX",
    "DATE",
    "VISIBILITY_KM_MEAN",
    "VISIBILITY_KM_MIN",
    "WIND_SPEED_10M_MS_MEAN",
    "WIND_SPEED_10M_MS_MAX",
    "TOTAL_CLOUD_COVER_PCT_MEAN",
    "PRECIP_MM_DAY_ESTIMATE",
    "SAMPLE_COVERAGE_FRAC",
    "QC_STATE",
]
DAYLIGHT_COLUMNS = [
    "H3_INDEX",
    "DATE",
    "DAYLIGHT_HOURS",
    "DAYLIGHT_FRACTION",
    "DAYLIGHT_WEIGHT",
    "SOLAR_ELEVATION_MAX_DEG",
    "LOW_SUN_DAYLIGHT_HOURS",
]

surface_weather_daily = pd.read_parquet(
    SURFACE_WEATHER_DIR, columns=WEATHER_COLUMNS
)
daylight_daily = pd.read_parquet(DAYLIGHT_DIR, columns=DAYLIGHT_COLUMNS)
surface_weather_daily["DATE"] = pd.to_datetime(
    surface_weather_daily["DATE"]
).dt.normalize()
daylight_daily["DATE"] = pd.to_datetime(daylight_daily["DATE"]).dt.normalize()
calendar_dynamic = calendar.rename(columns={"date": "DATE"}).copy()
calendar_dynamic["DATE"] = pd.to_datetime(calendar_dynamic["DATE"]).dt.normalize()

if surface_weather_daily.duplicated(["H3_INDEX", "DATE"]).any():
    raise ValueError("Surface weather is not unique by H3_INDEX and DATE.")
if daylight_daily.duplicated(["H3_INDEX", "DATE"]).any():
    raise ValueError("Daylight is not unique by H3_INDEX and DATE.")
if not calendar_dynamic["DATE"].is_unique:
    raise ValueError("Calendar context is not unique by DATE.")
weather_resolutions = {
    h3.get_resolution(cell)
    for cell in surface_weather_daily["H3_INDEX"].drop_duplicates()
}
daylight_resolutions = {
    h3.get_resolution(cell)
    for cell in daylight_daily["H3_INDEX"].drop_duplicates()
}
if weather_resolutions != {5}:
    raise ValueError(f"Expected only H3 R5 weather cells; found {weather_resolutions}.")
if daylight_resolutions != {4}:
    raise ValueError(f"Expected only H3 R4 daylight cells; found {daylight_resolutions}.")

surface_weather_manifest = json.loads(SURFACE_WEATHER_MANIFEST_PATH.read_text())
daylight_manifest = json.loads(DAYLIGHT_MANIFEST_PATH.read_text())
if surface_weather_manifest.get("source_completeness") != "complete":
    raise ValueError("Surface-weather manifest is not coverage-complete.")

dynamic_context_start_date = max(
    surface_weather_daily["DATE"].min(),
    daylight_daily["DATE"].min(),
    calendar_dynamic["DATE"].min(),
)
dynamic_context_end_date = min(
    surface_weather_daily["DATE"].max(),
    daylight_daily["DATE"].max(),
    calendar_dynamic["DATE"].max(),
)
if dynamic_context_start_date > dynamic_context_end_date:
    raise ValueError("Weather, daylight, and calendar have no shared date support.")


def load_land_source_dynamic_context_for_date(
    date_value: str | pd.Timestamp,
) -> pd.DataFrame:
    requested_date = pd.Timestamp(date_value).normalize()
    if not dynamic_context_start_date <= requested_date <= dynamic_context_end_date:
        raise ValueError(
            f"Requested date {requested_date.date()} is outside shared support "
            f"{dynamic_context_start_date.date()} to {dynamic_context_end_date.date()}."
        )

    source_context = land_sources[["source_h3"]].copy()
    source_context.insert(1, "H3_RESOLUTION", 7)
    source_context["WEATHER_H3_R5"] = source_context["source_h3"].map(
        lambda cell: h3.cell_to_parent(cell, 5)
    )
    source_context["DAYLIGHT_H3_R4"] = source_context["source_h3"].map(
        lambda cell: h3.cell_to_parent(cell, 4)
    )
    source_context["DATE"] = requested_date

    weather_for_date = surface_weather_daily.loc[
        surface_weather_daily["DATE"].eq(requested_date)
    ].rename(columns={"H3_INDEX": "WEATHER_H3_R5"})
    daylight_for_date = daylight_daily.loc[
        daylight_daily["DATE"].eq(requested_date)
    ].rename(columns={"H3_INDEX": "DAYLIGHT_H3_R4"})
    calendar_for_date = calendar_dynamic.loc[
        calendar_dynamic["DATE"].eq(requested_date)
    ]

    source_context = source_context.merge(
        weather_for_date,
        on=["WEATHER_H3_R5", "DATE"],
        how="left",
        validate="many_to_one",
    ).merge(
        daylight_for_date,
        on=["DAYLIGHT_H3_R4", "DATE"],
        how="left",
        validate="many_to_one",
    ).merge(
        calendar_for_date,
        on="DATE",
        how="left",
        validate="many_to_one",
    )
    source_context["WEATHER_CONTEXT_AVAILABLE"] = source_context[
        "SAMPLE_COVERAGE_FRAC"
    ].notna()
    source_context["WEATHER_SOURCE_COVERAGE_COMPLETE"] = (
        source_context["SAMPLE_COVERAGE_FRAC"].eq(1.0)
        & source_context["QC_STATE"].eq("COMPLETE")
    )
    source_context["DAYLIGHT_CONTEXT_AVAILABLE"] = source_context[
        "DAYLIGHT_FRACTION"
    ].notna()
    source_context["CALENDAR_CONTEXT_AVAILABLE"] = source_context[
        "calendar_effort_weight"
    ].notna()
    if len(source_context) != len(land_sources):
        raise ValueError("Dynamic-context joins changed the land-source universe.")
    if not source_context["source_h3"].is_unique:
        raise ValueError("Dynamic-context output is not unique by source_h3.")
    return source_context


dynamic_context_sample = load_land_source_dynamic_context_for_date(
    dynamic_context_end_date
)
dynamic_context_connection_summary = pd.DataFrame(
    [
        {
            "shared_start_date": dynamic_context_start_date,
            "shared_end_date": dynamic_context_end_date,
            "sample_date": dynamic_context_end_date,
            "land_source_cells": len(dynamic_context_sample),
            "weather_available_sources": int(
                dynamic_context_sample["WEATHER_CONTEXT_AVAILABLE"].sum()
            ),
            "weather_unavailable_sources": int(
                (~dynamic_context_sample["WEATHER_CONTEXT_AVAILABLE"]).sum()
            ),
            "daylight_available_sources": int(
                dynamic_context_sample["DAYLIGHT_CONTEXT_AVAILABLE"].sum()
            ),
            "daylight_unavailable_sources": int(
                (~dynamic_context_sample["DAYLIGHT_CONTEXT_AVAILABLE"]).sum()
            ),
            "calendar_available_sources": int(
                dynamic_context_sample["CALENDAR_CONTEXT_AVAILABLE"].sum()
            ),
        }
    ]
)
display(dynamic_context_connection_summary)
display(
    dynamic_context_sample[
        [
            "source_h3",
            "DATE",
            "WEATHER_H3_R5",
            "VISIBILITY_KM_MEAN",
            "PRECIP_MM_DAY_ESTIMATE",
            "DAYLIGHT_H3_R4",
            "DAYLIGHT_FRACTION",
            "calendar_effort_weight",
            "WEATHER_CONTEXT_AVAILABLE",
            "DAYLIGHT_CONTEXT_AVAILABLE",
        ]
    ].head()
)

## Source-centered population catchments

These features answer the source-centered question directly: how many residents are represented by census-allocated H3 cells whose **centroids are within X km of each land viewing source**? They are not the legacy `water_weighted_population_*` fields, which weight a population cell by that cell's distance to any project water.

Cumulative catchments are calculated at 5, 10, 25, and 50 km. Linear distance-decayed catchments are calculated at 10, 25, and 50 km using `population × max(1 - distance / radius, 0)`. Each radius retains support-cell counts, country codes, census vintages, and an availability flag. A supported zero remains zero; a radius with no census-support cells remains null.

In [ ]:
population_support = population.loc[
    population["POPULATION_CONTEXT_AVAILABLE"].eq(True)
    & population["POPULATION"].notna()
].copy()
if population_support.empty:
    raise ValueError("Population context has no available support cells.")
if not population_support["H3_INDEX"].is_unique:
    raise ValueError("Population catchment support is not unique by H3_INDEX.")


def h3_centroids_radians(cells: pd.Series) -> np.ndarray:
    return np.radians(
        np.asarray([h3.cell_to_latlng(cell) for cell in cells], dtype="float64")
    )


def radians_to_unit_sphere(latlng_radians: np.ndarray) -> np.ndarray:
    latitude = latlng_radians[:, 0]
    longitude = latlng_radians[:, 1]
    return np.column_stack(
        (
            np.cos(latitude) * np.cos(longitude),
            np.cos(latitude) * np.sin(longitude),
            np.sin(latitude),
        )
    )


population_xyz = radians_to_unit_sphere(
    h3_centroids_radians(population_support["H3_INDEX"])
)
source_xyz = radians_to_unit_sphere(h3_centroids_radians(land_sources["source_h3"]))
population_tree = cKDTree(population_xyz)
maximum_radius_km = max(POPULATION_CATCHMENT_RADII_KM)
maximum_chord = 2.0 * np.sin(
    (maximum_radius_km / EARTH_MEAN_RADIUS_KM) / 2.0
)
candidate_neighbors = population_tree.query_ball_point(source_xyz, r=maximum_chord)
population_values = population_support["POPULATION"].to_numpy(dtype="float64")

catchment_records = []
for source_h3, source_vector, candidate_indices in zip(
    land_sources["source_h3"], source_xyz, candidate_neighbors, strict=True
):
    record = {"source_h3": source_h3}
    candidate_indices = np.asarray(candidate_indices, dtype="int64")
    if candidate_indices.size:
        central_angles = np.arccos(
            np.clip(population_xyz[candidate_indices] @ source_vector, -1.0, 1.0)
        )
        distances_km = EARTH_MEAN_RADIUS_KM * central_angles
    else:
        distances_km = np.asarray([], dtype="float64")

    for radius_km in POPULATION_CATCHMENT_RADII_KM:
        within_radius = distances_km <= radius_km + 1e-9
        radius_indices = candidate_indices[within_radius]
        available = bool(radius_indices.size)
        suffix = f"{radius_km}_KM"
        record[f"POPULATION_CATCHMENT_AVAILABLE_{suffix}"] = available
        record[f"POPULATION_CATCHMENT_SUPPORT_CELL_COUNT_{suffix}"] = int(
            radius_indices.size
        )
        record[f"POPULATION_WITHIN_{suffix}"] = (
            float(population_values[radius_indices].sum()) if available else np.nan
        )
        if available:
            radius_rows = population_support.iloc[radius_indices]
            countries = sorted(
                {
                    code
                    for codes in radius_rows["COUNTRY_CODES"].dropna().astype(str)
                    for code in codes.split("|")
                    if code
                }
            )
            record[f"POPULATION_CATCHMENT_COUNTRY_CODES_{suffix}"] = "|".join(
                countries
            )
            record[f"POPULATION_CATCHMENT_CENSUS_YEAR_MIN_{suffix}"] = int(
                radius_rows["CENSUS_YEAR_MIN"].min()
            )
            record[f"POPULATION_CATCHMENT_CENSUS_YEAR_MAX_{suffix}"] = int(
                radius_rows["CENSUS_YEAR_MAX"].max()
            )
        else:
            record[f"POPULATION_CATCHMENT_COUNTRY_CODES_{suffix}"] = None
            record[f"POPULATION_CATCHMENT_CENSUS_YEAR_MIN_{suffix}"] = pd.NA
            record[f"POPULATION_CATCHMENT_CENSUS_YEAR_MAX_{suffix}"] = pd.NA

        if radius_km in POPULATION_DECAY_RADII_KM:
            record[f"POPULATION_DECAYED_{suffix}"] = (
                float(
                    (
                        population_values[radius_indices]
                        * (1.0 - distances_km[within_radius] / radius_km)
                    ).sum()
                )
                if available
                else np.nan
            )
    catchment_records.append(record)

population_catchments = pd.DataFrame.from_records(catchment_records)
population_catchments["POPULATION_CATCHMENT_DISTANCE_BASIS"] = (
    "h3_r7_centroid_great_circle"
)
population_catchments["POPULATION_CATCHMENT_DECAY_KERNEL"] = (
    "linear_truncated_at_radius"
)
if not population_catchments["source_h3"].is_unique:
    raise ValueError("Population catchments are not unique by source_h3.")
if len(population_catchments) != len(land_sources):
    raise ValueError("Population catchments changed the source universe.")

land_source_context = land_source_context.merge(
    population_catchments, on="source_h3", how="left", validate="one_to_one"
)
for radius_km in POPULATION_CATCHMENT_RADII_KM:
    availability_column = f"POPULATION_CATCHMENT_AVAILABLE_{radius_km}_KM"
    population_column = f"POPULATION_WITHIN_{radius_km}_KM"
    unavailable = ~land_source_context[availability_column].eq(True)
    if not land_source_context.loc[unavailable, population_column].isna().all():
        raise ValueError(f"Unavailable {radius_km} km catchments must remain null.")

catchment_summary_rows = []
for radius_km in POPULATION_CATCHMENT_RADII_KM:
    for feature_type, population_column in [
        ("cumulative", f"POPULATION_WITHIN_{radius_km}_KM"),
        *(
            [("linear_decay", f"POPULATION_DECAYED_{radius_km}_KM")]
            if radius_km in POPULATION_DECAY_RADII_KM
            else []
        ),
    ]:
        values = land_source_context[population_column].dropna()
        catchment_summary_rows.append(
            {
                "feature": population_column,
                "feature_type": feature_type,
                "radius_km": radius_km,
                "available_sources": len(values),
                "observed_zero_sources": int(values.eq(0.0).sum()),
                "median_population": float(values.median()),
                "p90_population": float(values.quantile(0.90)),
                "p99_population": float(values.quantile(0.99)),
                "maximum_population": float(values.max()),
            }
        )
population_catchment_summary = pd.DataFrame(catchment_summary_rows)
display(population_catchment_summary)

## Exploratory road and city travel access

The repository does not yet contain a land-road graph. For this notebook prototype, each land-source H3 centroid is sent to the public OSRM driving table service in bounded, checksum-keyed batches. The returned destination waypoint provides the distance from the H3 centroid to the nearest routable driving segment; the table provides road-network distance and duration from eight explicit regional city-center origins.

This is **routing evidence**, not legal shore access. A routable road near a source does not imply public access, parking, a trail, or a safe viewpoint. Source-centroid snapping can also overstate accessibility on islands and irregular coastal cells. Raw responses are cached under the prototype output directory, missing routes remain null, and no unreachable or unavailable value is converted to zero.

In [ ]:
OSRM_TABLE_ENDPOINT = "https://router.project-osrm.org/table/v1/driving"
OSRM_MAX_DESTINATIONS_PER_BATCH = 90
OSRM_REQUEST_TIMEOUT_SECONDS = 180
OSRM_REQUEST_ATTEMPTS = 3
REFRESH_TRANSPORT_ROUTING = False
transport_city_origins = pd.DataFrame(
    [
        {"CITY": "Seattle", "COUNTRY_CODE": "US", "LATITUDE": 47.6062, "LONGITUDE": -122.3321},
        {"CITY": "Tacoma", "COUNTRY_CODE": "US", "LATITUDE": 47.2529, "LONGITUDE": -122.4443},
        {"CITY": "Olympia", "COUNTRY_CODE": "US", "LATITUDE": 47.0379, "LONGITUDE": -122.9007},
        {"CITY": "Everett", "COUNTRY_CODE": "US", "LATITUDE": 47.9790, "LONGITUDE": -122.2021},
        {"CITY": "Bellingham", "COUNTRY_CODE": "US", "LATITUDE": 48.7519, "LONGITUDE": -122.4787},
        {"CITY": "Vancouver", "COUNTRY_CODE": "CA", "LATITUDE": 49.2827, "LONGITUDE": -123.1207},
        {"CITY": "Victoria", "COUNTRY_CODE": "CA", "LATITUDE": 48.4284, "LONGITUDE": -123.3656},
        {"CITY": "Nanaimo", "COUNTRY_CODE": "CA", "LATITUDE": 49.1659, "LONGITUDE": -123.9401},
    ]
)
transport_city_origins["ORIGIN_DEFINITION"] = "explicit_approximate_city_center"
if not transport_city_origins["CITY"].is_unique:
    raise ValueError("Transport city origins are not unique.")


def city_feature_slug(city: str) -> str:
    return "".join(character if character.isalnum() else "_" for character in city.upper())


TRANSPORT_ACCESS_DIR.mkdir(parents=True, exist_ok=True)
TRANSPORT_ROUTING_CACHE_DIR.mkdir(parents=True, exist_ok=True)
source_transport_base = land_sources[["source_h3"]].copy()
source_transport_latlng = np.asarray(
    [h3.cell_to_latlng(cell) for cell in source_transport_base["source_h3"]],
    dtype="float64",
)
source_transport_base["SOURCE_CENTROID_LATITUDE"] = source_transport_latlng[:, 0]
source_transport_base["SOURCE_CENTROID_LONGITUDE"] = source_transport_latlng[:, 1]
origin_coordinates = [
    (float(row.LONGITUDE), float(row.LATITUDE))
    for row in transport_city_origins.itertuples(index=False)
]
origin_count = len(origin_coordinates)
routing_records = []
routing_cache_paths = []
origin_snap_distances_by_city: dict[str, float] = {}

for batch_index, start_index in enumerate(
    range(0, len(source_transport_base), OSRM_MAX_DESTINATIONS_PER_BATCH)
):
    batch = source_transport_base.iloc[
        start_index : start_index + OSRM_MAX_DESTINATIONS_PER_BATCH
    ].copy()
    destination_coordinates = list(
        zip(
            batch["SOURCE_CENTROID_LONGITUDE"],
            batch["SOURCE_CENTROID_LATITUDE"],
            strict=True,
        )
    )
    all_coordinates = origin_coordinates + destination_coordinates
    request_contract = {
        "endpoint": OSRM_TABLE_ENDPOINT,
        "profile": "driving",
        "annotations": ["duration", "distance"],
        "city_origins": transport_city_origins[
            ["CITY", "COUNTRY_CODE", "LATITUDE", "LONGITUDE", "ORIGIN_DEFINITION"]
        ].to_dict(orient="records"),
        "source_h3": batch["source_h3"].tolist(),
        "source_centroid_coordinates": [
            [round(float(longitude), 7), round(float(latitude), 7)]
            for longitude, latitude in destination_coordinates
        ],
    }
    request_digest = hashlib.sha256(
        json.dumps(request_contract, sort_keys=True, separators=(",", ":")).encode("utf-8")
    ).hexdigest()
    cache_path = TRANSPORT_ROUTING_CACHE_DIR / (
        f"osrm_table_{batch_index:03d}_{request_digest[:16]}.json"
    )
    routing_cache_paths.append(cache_path)
    if cache_path.is_file() and not REFRESH_TRANSPORT_ROUTING:
        cache_wrapper = json.loads(cache_path.read_text(encoding="utf-8"))
        if cache_wrapper.get("request_sha256") != request_digest:
            raise ValueError(f"Routing cache contract mismatch: {cache_path}")
        payload = cache_wrapper["response"]
    else:
        coordinate_string = ";".join(
            f"{float(longitude):.7f},{float(latitude):.7f}"
            for longitude, latitude in all_coordinates
        )
        source_indexes = ";".join(str(index) for index in range(origin_count))
        destination_indexes = ";".join(
            str(index) for index in range(origin_count, len(all_coordinates))
        )
        request_url = f"{OSRM_TABLE_ENDPOINT}/{coordinate_string}"
        errors = []
        payload = None
        for attempt in range(OSRM_REQUEST_ATTEMPTS):
            try:
                response = requests.get(
                    request_url,
                    params={
                        "sources": source_indexes,
                        "destinations": destination_indexes,
                        "annotations": "duration,distance",
                    },
                    headers={
                        "User-Agent": "OrcaCastDataPrep/1.1 (+https://github.com/orcacast)",
                        "Accept": "application/json",
                    },
                    timeout=(30, OSRM_REQUEST_TIMEOUT_SECONDS),
                )
                response.raise_for_status()
                candidate_payload = response.json()
                if candidate_payload.get("code") != "Ok":
                    raise RuntimeError(
                        f"OSRM returned {candidate_payload.get('code')}: "
                        f"{candidate_payload.get('message')}"
                    )
                payload = candidate_payload
                break
            except Exception as exc:
                errors.append(f"attempt {attempt + 1}: {type(exc).__name__}: {exc}")
                if attempt + 1 < OSRM_REQUEST_ATTEMPTS:
                    time.sleep(min(2 ** attempt, 5))
        if payload is None:
            raise RuntimeError("OSRM routing batch failed: " + "; ".join(errors))
        cache_wrapper = {
            "schema_version": "0.1.0-prototype",
            "request_sha256": request_digest,
            "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
            "request_contract": request_contract,
            "response": payload,
        }
        temporary_cache_path = cache_path.with_suffix(".json.tmp")
        temporary_cache_path.write_text(
            json.dumps(cache_wrapper, separators=(",", ":"), allow_nan=False),
            encoding="utf-8",
        )
        temporary_cache_path.replace(cache_path)

    expected_destinations = len(batch)
    if len(payload.get("destinations", [])) != expected_destinations:
        raise ValueError(f"OSRM destination count mismatch in batch {batch_index}.")
    if len(payload.get("durations", [])) != origin_count:
        raise ValueError(f"OSRM duration-origin count mismatch in batch {batch_index}.")
    if len(payload.get("distances", [])) != origin_count:
        raise ValueError(f"OSRM distance-origin count mismatch in batch {batch_index}.")
    if any(len(row) != expected_destinations for row in payload["durations"]):
        raise ValueError(f"OSRM duration-destination count mismatch in batch {batch_index}.")
    if any(len(row) != expected_destinations for row in payload["distances"]):
        raise ValueError(f"OSRM distance-destination count mismatch in batch {batch_index}.")

    for origin_index, city_row in enumerate(
        transport_city_origins.itertuples(index=False)
    ):
        source_waypoint = payload["sources"][origin_index]
        origin_snap_distance = source_waypoint.get("distance")
        if origin_snap_distance is not None:
            previous = origin_snap_distances_by_city.get(city_row.CITY)
            if previous is not None and not np.isclose(
                previous, float(origin_snap_distance), atol=0.01
            ):
                raise ValueError(f"OSRM city-origin snap drift for {city_row.CITY}.")
            origin_snap_distances_by_city[city_row.CITY] = float(origin_snap_distance)

    for destination_index, source_row in enumerate(batch.itertuples(index=False)):
        waypoint = payload["destinations"][destination_index]
        snapped_location = waypoint.get("location") or [None, None]
        record = {
            "source_h3": source_row.source_h3,
            "SOURCE_CENTROID_LATITUDE": float(source_row.SOURCE_CENTROID_LATITUDE),
            "SOURCE_CENTROID_LONGITUDE": float(source_row.SOURCE_CENTROID_LONGITUDE),
            "ROAD_SNAP_DISTANCE_FROM_SOURCE_CENTROID_M": (
                float(waypoint["distance"])
                if waypoint.get("distance") is not None
                else np.nan
            ),
            "ROAD_SNAP_LONGITUDE": (
                float(snapped_location[0]) if snapped_location[0] is not None else np.nan
            ),
            "ROAD_SNAP_LATITUDE": (
                float(snapped_location[1]) if snapped_location[1] is not None else np.nan
            ),
            "ROAD_SNAP_NAME": waypoint.get("name") or None,
        }
        for origin_index, city_row in enumerate(
            transport_city_origins.itertuples(index=False)
        ):
            slug = city_feature_slug(city_row.CITY)
            duration_seconds = payload["durations"][origin_index][destination_index]
            distance_meters = payload["distances"][origin_index][destination_index]
            record[f"TRAVEL_TIME_FROM_{slug}_MIN"] = (
                float(duration_seconds) / 60.0
                if duration_seconds is not None
                else np.nan
            )
            record[f"TRAVEL_DISTANCE_FROM_{slug}_KM"] = (
                float(distance_meters) / 1000.0
                if distance_meters is not None
                else np.nan
            )
        routing_records.append(record)

source_transport_access = pd.DataFrame.from_records(routing_records)
if len(source_transport_access) != len(land_sources):
    raise ValueError("Transport routing changed the land-source universe.")
if not source_transport_access["source_h3"].is_unique:
    raise ValueError("Transport routing is not unique by source_h3.")
time_columns = [
    f"TRAVEL_TIME_FROM_{city_feature_slug(city)}_MIN"
    for city in transport_city_origins["CITY"]
]
distance_columns = [
    f"TRAVEL_DISTANCE_FROM_{city_feature_slug(city)}_KM"
    for city in transport_city_origins["CITY"]
]
source_transport_access["MIN_CITY_TRAVEL_TIME_MIN"] = source_transport_access[
    time_columns
].min(axis=1, skipna=True)
source_transport_access["MIN_CITY_TRAVEL_DISTANCE_KM"] = source_transport_access[
    distance_columns
].min(axis=1, skipna=True)
source_transport_access["NEAREST_CITY_BY_TRAVEL_TIME"] = (
    source_transport_access[time_columns]
    .idxmin(axis=1, skipna=True)
    .str.removeprefix("TRAVEL_TIME_FROM_")
    .str.removesuffix("_MIN")
)
source_transport_access["NEAREST_CITY_BY_TRAVEL_DISTANCE"] = (
    source_transport_access[distance_columns]
    .idxmin(axis=1, skipna=True)
    .str.removeprefix("TRAVEL_DISTANCE_FROM_")
    .str.removesuffix("_KM")
)
source_transport_access["ROUTED_CITY_ORIGIN_COUNT"] = source_transport_access[
    time_columns
].notna().sum(axis=1).astype("int64")
source_transport_access["CITY_ORIGIN_COUNT"] = origin_count
source_transport_access["ROAD_ROUTING_AVAILABLE"] = source_transport_access[
    "ROAD_SNAP_DISTANCE_FROM_SOURCE_CENTROID_M"
].notna()
source_transport_access["CITY_TRAVEL_ROUTING_AVAILABLE"] = source_transport_access[
    "MIN_CITY_TRAVEL_TIME_MIN"
].notna()
source_transport_access["TRANSPORT_CONTEXT_AVAILABLE"] = (
    source_transport_access["ROAD_ROUTING_AVAILABLE"]
    & source_transport_access["CITY_TRAVEL_ROUTING_AVAILABLE"]
)
source_transport_access["TRAVEL_ROUTING_STATUS"] = np.select(
    [
        source_transport_access["ROUTED_CITY_ORIGIN_COUNT"].eq(origin_count),
        source_transport_access["ROUTED_CITY_ORIGIN_COUNT"].gt(0),
    ],
    ["all_configured_city_origins_routed", "partial_city_origin_routing"],
    default="unavailable_no_city_route",
)
source_transport_access["ROUTING_PROFILE"] = "osrm_driving"
source_transport_access["ROAD_DISTANCE_BASIS"] = (
    "source_h3_centroid_to_osrm_snapped_driving_segment"
)
source_transport_access["CITY_ORIGIN_DEFINITION"] = (
    "explicit_approximate_city_centers"
)
source_transport_access["MEASUREMENT_STATUS"] = "derived_prototype"

numeric_transport_columns = [
    "ROAD_SNAP_DISTANCE_FROM_SOURCE_CENTROID_M",
    "MIN_CITY_TRAVEL_TIME_MIN",
    "MIN_CITY_TRAVEL_DISTANCE_KM",
    *time_columns,
    *distance_columns,
]
if any(
    source_transport_access[column].dropna().lt(0).any()
    for column in numeric_transport_columns
):
    raise ValueError("Transport routing produced a negative distance or duration.")

transport_city_origins["ORIGIN_ROAD_SNAP_DISTANCE_M"] = transport_city_origins[
    "CITY"
].map(origin_snap_distances_by_city)
routing_cache_inventory = pd.DataFrame(
    [
        {
            "path": str(path.relative_to(REPO_ROOT)),
            "bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        }
        for path in routing_cache_paths
    ]
)
land_source_context = land_source_context.merge(
    source_transport_access.drop(
        columns=["SOURCE_CENTROID_LATITUDE", "SOURCE_CENTROID_LONGITUDE", "MEASUREMENT_STATUS"]
    ),
    on="source_h3",
    how="left",
    validate="one_to_one",
)
if land_source_context["TRANSPORT_CONTEXT_AVAILABLE"].isna().any():
    raise ValueError("A land source is missing its transport routing row.")

transport_summary_rows = [
    {"measure": "land_source_cells", "value": len(source_transport_access), "unit": "source_cells"},
    {"measure": "transport_context_available", "value": int(source_transport_access["TRANSPORT_CONTEXT_AVAILABLE"].sum()), "unit": "source_cells"},
    {"measure": "routing_cache_batches", "value": len(routing_cache_inventory), "unit": "batches"},
    {"measure": "median_road_snap_distance", "value": float(source_transport_access["ROAD_SNAP_DISTANCE_FROM_SOURCE_CENTROID_M"].median()), "unit": "meters"},
    {"measure": "p90_road_snap_distance", "value": float(source_transport_access["ROAD_SNAP_DISTANCE_FROM_SOURCE_CENTROID_M"].quantile(0.90)), "unit": "meters"},
    {"measure": "p99_road_snap_distance", "value": float(source_transport_access["ROAD_SNAP_DISTANCE_FROM_SOURCE_CENTROID_M"].quantile(0.99)), "unit": "meters"},
    {"measure": "median_min_city_travel_time", "value": float(source_transport_access["MIN_CITY_TRAVEL_TIME_MIN"].median()), "unit": "minutes"},
    {"measure": "p90_min_city_travel_time", "value": float(source_transport_access["MIN_CITY_TRAVEL_TIME_MIN"].quantile(0.90)), "unit": "minutes"},
    {"measure": "median_min_city_travel_distance", "value": float(source_transport_access["MIN_CITY_TRAVEL_DISTANCE_KM"].median()), "unit": "kilometers"},
]
for city in transport_city_origins["CITY"]:
    slug = city_feature_slug(city)
    transport_summary_rows.append(
        {
            "measure": f"median_travel_time_from_{slug.lower()}",
            "value": float(source_transport_access[f"TRAVEL_TIME_FROM_{slug}_MIN"].median()),
            "unit": "minutes",
        }
    )
transport_source_summary = pd.DataFrame(transport_summary_rows)
display(transport_city_origins)
display(transport_source_summary)

## Composite land-viewing pressure by target cell

The composite is constructed before sightings are loaded:

1. Exact-cell, 5/10/25/50 km cumulative, and 10/25/50 km linearly decayed population features are each transformed with `log1p`, divided by their own source-universe 99th percentile, and clipped to `[0, 1]`.
2. Population-only target pressure retains the full census-supported source universe.
3. Access-gated target pressure multiplies each population component by valid `ACCESSIBLE_WATERFRONT_FRACTION`; unknown access and denominator-QC cells remain unavailable.
4. Every target variant is the sum of `weight_static_viewability × source component` over its usable source-target pairs.
5. `DECAYED_25_KM` is designated as the primary exploratory access-gated variant before sightings are loaded; sightings do not select or tune it.

The raw pressure is a partial-evidence index, not observer counts or detection probability. Every target retains context-coverage fields; targets with no usable source context retain null pressure.

In [ ]:
source_composite_available = (
    land_source_context["POPULATION_CONTEXT_MATCHED"]
    & land_source_context["PUBLIC_SHORE_CONTEXT_MATCHED"]
    & land_source_context["POPULATION_LOG1P"].notna()
    & land_source_context["ACCESSIBLE_WATERFRONT_FRACTION"].notna()
    & ~land_source_context["FRACTION_DENOMINATOR_MISMATCH_QC"].eq(True)
    & land_source_context["PUBLIC_ACCESS_STATE"].eq("observed_access")
)
population_cap = land_source_context.loc[
    source_composite_available, "POPULATION_LOG1P"
].quantile(POPULATION_CAP_QUANTILE)
if pd.isna(population_cap) or population_cap <= 0:
    raise ValueError("Usable source context did not produce a positive population cap.")

land_source_context["POPULATION_COMPONENT"] = (
    land_source_context["POPULATION_LOG1P"] / population_cap
).clip(lower=0.0, upper=1.0)
land_source_context["PUBLIC_SHORE_ACCESS_COMPONENT"] = land_source_context[
    "ACCESSIBLE_WATERFRONT_FRACTION"
].where(source_composite_available)
land_source_context["LAND_SOURCE_COMPOSITE_AVAILABLE"] = source_composite_available
land_source_context["LAND_SOURCE_COMPOSITE"] = (
    land_source_context["POPULATION_COMPONENT"]
    * land_source_context["PUBLIC_SHORE_ACCESS_COMPONENT"]
).where(source_composite_available)

if not land_source_context.loc[
    ~source_composite_available, "LAND_SOURCE_COMPOSITE"
].isna().all():
    raise ValueError("Unavailable source context must retain null composite values.")
if not land_source_context.loc[
    source_composite_available, "LAND_SOURCE_COMPOSITE"
].between(0.0, 1.0).all():
    raise ValueError("Source composite values must be bounded to [0, 1].")

source_composite_summary = pd.DataFrame(
    [
        {
            "land_source_cells": len(land_source_context),
            "usable_source_cells": int(source_composite_available.sum()),
            "unavailable_source_cells": int((~source_composite_available).sum()),
            "population_cap_quantile": POPULATION_CAP_QUANTILE,
            "population_log1p_cap": float(population_cap),
        }
    ]
)
coverage_summary = pd.concat(
    [
        coverage_summary,
        pd.DataFrame(
            [
                {"measure": "land_source_composite_available", "count": int(source_composite_available.sum())},
                {"measure": "land_source_composite_unavailable", "count": int((~source_composite_available).sum())},
            ]
        ),
    ],
    ignore_index=True,
)
coverage_summary["fraction_of_land_sources"] = coverage_summary["count"] / len(land_source_context)
display(source_composite_summary)

In [ ]:
land_source_context["POPULATION_EXACT_CELL_AVAILABLE"] = (
    land_source_context["POPULATION_CONTEXT_MATCHED"]
    & land_source_context["POPULATION_CONTEXT_AVAILABLE"].eq(True)
    & land_source_context["POPULATION"].notna()
)
population_variant_specs = {
    "EXACT_CELL": {
        "population_column": "POPULATION",
        "availability_column": "POPULATION_EXACT_CELL_AVAILABLE",
        "radius_km": None,
        "kernel": "exact_h3_cell",
    },
    **{
        f"WITHIN_{radius_km}_KM": {
            "population_column": f"POPULATION_WITHIN_{radius_km}_KM",
            "availability_column": f"POPULATION_CATCHMENT_AVAILABLE_{radius_km}_KM",
            "radius_km": radius_km,
            "kernel": "cumulative_hard_radius",
        }
        for radius_km in POPULATION_CATCHMENT_RADII_KM
    },
    **{
        f"DECAYED_{radius_km}_KM": {
            "population_column": f"POPULATION_DECAYED_{radius_km}_KM",
            "availability_column": f"POPULATION_CATCHMENT_AVAILABLE_{radius_km}_KM",
            "radius_km": radius_km,
            "kernel": "linear_truncated_at_radius",
        }
        for radius_km in POPULATION_DECAY_RADII_KM
    },
}
if PRIMARY_POPULATION_VARIANT not in population_variant_specs:
    raise ValueError("Primary population variant is not defined.")

public_shore_access_available = (
    land_source_context["PUBLIC_SHORE_CONTEXT_MATCHED"]
    & land_source_context["ACCESSIBLE_WATERFRONT_FRACTION"].notna()
    & ~land_source_context["FRACTION_DENOMINATOR_MISMATCH_QC"].eq(True)
    & land_source_context["PUBLIC_ACCESS_STATE"].eq("observed_access")
)
land_source_context["PUBLIC_SHORE_ACCESS_COMPONENT"] = land_source_context[
    "ACCESSIBLE_WATERFRONT_FRACTION"
].where(public_shore_access_available)

population_cap_rows = []
variant_coverage_rows = []
for variant, spec in population_variant_specs.items():
    population_column = spec["population_column"]
    population_available = land_source_context[spec["availability_column"]].eq(True)
    population_log1p = np.log1p(land_source_context[population_column]).where(
        population_available
    )
    population_cap = population_log1p.loc[population_available].quantile(
        POPULATION_CAP_QUANTILE
    )
    if pd.isna(population_cap) or population_cap <= 0:
        raise ValueError(f"{variant} did not produce a positive population cap.")

    component_column = f"POPULATION_COMPONENT_{variant}"
    population_source_available_column = f"POPULATION_SOURCE_AVAILABLE_{variant}"
    access_available_column = f"LAND_SOURCE_ACCESS_GATED_AVAILABLE_{variant}"
    access_component_column = f"LAND_SOURCE_ACCESS_GATED_COMPONENT_{variant}"
    land_source_context[population_source_available_column] = population_available
    land_source_context[component_column] = (population_log1p / population_cap).clip(
        lower=0.0, upper=1.0
    )
    access_available = population_available & public_shore_access_available
    land_source_context[access_available_column] = access_available
    land_source_context[access_component_column] = (
        land_source_context[component_column]
        * land_source_context["PUBLIC_SHORE_ACCESS_COMPONENT"]
    ).where(access_available)

    if not land_source_context.loc[
        population_available, component_column
    ].between(0.0, 1.0).all():
        raise ValueError(f"{variant} population components must be within [0, 1].")
    if not land_source_context.loc[
        ~access_available, access_component_column
    ].isna().all():
        raise ValueError(f"Unavailable {variant} access-gated components must be null.")

    population_cap_rows.append(
        {
            "variant": variant,
            "population_column": population_column,
            "radius_km": spec["radius_km"],
            "kernel": spec["kernel"],
            "population_log1p_cap_quantile": POPULATION_CAP_QUANTILE,
            "population_log1p_cap": float(population_cap),
            "population_supported_sources": int(population_available.sum()),
            "access_gated_sources": int(access_available.sum()),
        }
    )
    variant_coverage_rows.extend(
        [
            {
                "measure": f"population_source_available_{variant.lower()}",
                "count": int(population_available.sum()),
            },
            {
                "measure": f"access_gated_source_available_{variant.lower()}",
                "count": int(access_available.sum()),
            },
        ]
    )

population_cap_summary = pd.DataFrame(population_cap_rows)
coverage_summary = pd.concat(
    [coverage_summary, pd.DataFrame(variant_coverage_rows)], ignore_index=True
)
coverage_summary["fraction_of_land_sources"] = (
    coverage_summary["count"] / len(land_source_context)
)

primary_component_column = f"POPULATION_COMPONENT_{PRIMARY_POPULATION_VARIANT}"
primary_available_column = (
    f"LAND_SOURCE_ACCESS_GATED_AVAILABLE_{PRIMARY_POPULATION_VARIANT}"
)
primary_access_component_column = (
    f"LAND_SOURCE_ACCESS_GATED_COMPONENT_{PRIMARY_POPULATION_VARIANT}"
)
land_source_context["POPULATION_COMPONENT"] = land_source_context[
    primary_component_column
]
land_source_context["LAND_SOURCE_COMPOSITE_AVAILABLE"] = land_source_context[
    primary_available_column
]
land_source_context["LAND_SOURCE_COMPOSITE"] = land_source_context[
    primary_access_component_column
]
source_composite_available = land_source_context[primary_available_column]
population_cap = float(
    population_cap_summary.set_index("variant").loc[
        PRIMARY_POPULATION_VARIANT, "population_log1p_cap"
    ]
)
display(population_cap_summary)

## Population-weighted travel demand

The fixed-city experiment is now complemented by a gridded population-origin measure. Census-supported R7 population is aggregated to population-weighted H3 R4 regional origins, retaining the smallest high-population set that covers at least 99% of represented population plus low-population origins within 25 km of a land source. OSRM driving time is routed from each retained origin to every land-source centroid.

For source (s) and decay scale (	au), the uncalibrated demand is:

[
D_{s,	au} = sum_o P_o exp(-t_{o,s}/	au)
]

Missing routes are not treated as zero. Each source records routed-population coverage, and modeled demand is available only when at least 99% of the selected origin population is routed.

In [ ]:
population_origin_cells = population.loc[
    population["POPULATION_CONTEXT_AVAILABLE"].eq(True)
    & population["POPULATION"].notna()
    & population["POPULATION"].gt(0)
].copy()
population_origin_latlng = np.asarray(
    [h3.cell_to_latlng(cell) for cell in population_origin_cells["H3_INDEX"]],
    dtype="float64",
)
population_origin_cells["_LATITUDE"] = population_origin_latlng[:, 0]
population_origin_cells["_LONGITUDE"] = population_origin_latlng[:, 1]
population_origin_cells["_ORIGIN_H3"] = population_origin_cells["H3_INDEX"].map(
    lambda cell: h3.cell_to_parent(cell, POPULATION_TRAVEL_ORIGIN_RESOLUTION)
)
population_origin_cells["_WEIGHTED_LATITUDE"] = (
    population_origin_cells["POPULATION"] * population_origin_cells["_LATITUDE"]
)
population_origin_cells["_WEIGHTED_LONGITUDE"] = (
    population_origin_cells["POPULATION"] * population_origin_cells["_LONGITUDE"]
)

population_travel_origin_universe = (
    population_origin_cells.groupby("_ORIGIN_H3", as_index=False)
    .agg(
        POPULATION=("POPULATION", "sum"),
        POPULATION_US_2020=("POPULATION_US_2020", "sum"),
        POPULATION_CA_2021=("POPULATION_CA_2021", "sum"),
        SOURCE_R7_CELL_COUNT=("H3_INDEX", "size"),
        _WEIGHTED_LATITUDE=("_WEIGHTED_LATITUDE", "sum"),
        _WEIGHTED_LONGITUDE=("_WEIGHTED_LONGITUDE", "sum"),
    )
    .rename(columns={"_ORIGIN_H3": "origin_h3"})
)
population_travel_origin_universe["ORIGIN_LATITUDE"] = (
    population_travel_origin_universe["_WEIGHTED_LATITUDE"]
    / population_travel_origin_universe["POPULATION"]
)
population_travel_origin_universe["ORIGIN_LONGITUDE"] = (
    population_travel_origin_universe["_WEIGHTED_LONGITUDE"]
    / population_travel_origin_universe["POPULATION"]
)
population_travel_origin_universe = population_travel_origin_universe.drop(
    columns=["_WEIGHTED_LATITUDE", "_WEIGHTED_LONGITUDE"]
)
population_travel_total_population = float(
    population.loc[
        population["POPULATION_CONTEXT_AVAILABLE"].eq(True)
        & population["POPULATION"].notna(),
        "POPULATION",
    ].sum()
)
population_travel_origin_universe = population_travel_origin_universe.sort_values(
    ["POPULATION", "origin_h3"], ascending=[False, True]
).reset_index(drop=True)
population_travel_origin_universe["CUMULATIVE_POPULATION_BEFORE"] = (
    population_travel_origin_universe["POPULATION"].cumsum()
    - population_travel_origin_universe["POPULATION"]
)
population_travel_origin_universe["CUMULATIVE_POPULATION_SHARE"] = (
    population_travel_origin_universe["POPULATION"].cumsum()
    / population_travel_total_population
)
population_travel_origin_universe["RETAINED_BY_POPULATION_SHARE"] = (
    population_travel_origin_universe["CUMULATIVE_POPULATION_BEFORE"]
    / population_travel_total_population
).lt(POPULATION_TRAVEL_RETAINED_SHARE)


def unit_sphere_xyz(latitudes, longitudes):
    latitudes_radians = np.radians(np.asarray(latitudes, dtype="float64"))
    longitudes_radians = np.radians(np.asarray(longitudes, dtype="float64"))
    return np.column_stack(
        [
            np.cos(latitudes_radians) * np.cos(longitudes_radians),
            np.cos(latitudes_radians) * np.sin(longitudes_radians),
            np.sin(latitudes_radians),
        ]
    )


source_latlng = np.asarray(
    [h3.cell_to_latlng(cell) for cell in land_sources["source_h3"]],
    dtype="float64",
)
source_tree = cKDTree(unit_sphere_xyz(source_latlng[:, 0], source_latlng[:, 1]))
origin_chord_distance, _ = source_tree.query(
    unit_sphere_xyz(
        population_travel_origin_universe["ORIGIN_LATITUDE"],
        population_travel_origin_universe["ORIGIN_LONGITUDE"],
    ),
    k=1,
)
population_travel_origin_universe["NEAREST_LAND_SOURCE_DISTANCE_KM"] = (
    2.0
    * EARTH_MEAN_RADIUS_KM
    * np.arcsin(np.clip(origin_chord_distance / 2.0, 0.0, 1.0))
)
population_travel_origin_universe["RETAINED_AS_LOCAL_ORIGIN"] = (
    population_travel_origin_universe["NEAREST_LAND_SOURCE_DISTANCE_KM"]
    <= POPULATION_TRAVEL_LOCAL_ORIGIN_DISTANCE_KM
)
population_travel_origin_universe["ORIGIN_SELECTED"] = (
    population_travel_origin_universe["RETAINED_BY_POPULATION_SHARE"]
    | population_travel_origin_universe["RETAINED_AS_LOCAL_ORIGIN"]
)
population_travel_origin_universe["SELECTION_REASON"] = np.select(
    [
        population_travel_origin_universe["RETAINED_BY_POPULATION_SHARE"]
        & population_travel_origin_universe["RETAINED_AS_LOCAL_ORIGIN"],
        population_travel_origin_universe["RETAINED_BY_POPULATION_SHARE"],
        population_travel_origin_universe["RETAINED_AS_LOCAL_ORIGIN"],
    ],
    ["population_share_and_local", "population_share", "local_origin"],
    default="not_selected",
)
population_travel_origins = (
    population_travel_origin_universe.loc[
        population_travel_origin_universe["ORIGIN_SELECTED"]
    ]
    .copy()
    .sort_values("origin_h3")
    .reset_index(drop=True)
)
population_travel_selected_population = float(
    population_travel_origins["POPULATION"].sum()
)
population_travel_represented_fraction = (
    population_travel_selected_population / population_travel_total_population
)
if population_travel_represented_fraction < POPULATION_TRAVEL_RETAINED_SHARE:
    raise ValueError("Population-travel origins do not meet retained-population coverage.")
if not population_travel_origins["origin_h3"].is_unique:
    raise ValueError("Population-travel origins are not unique by origin_h3.")

population_travel_origin_summary = pd.DataFrame(
    [
        {
            "measure": "population_origin_resolution",
            "value": POPULATION_TRAVEL_ORIGIN_RESOLUTION,
            "unit": "h3_resolution",
        },
        {
            "measure": "population_origin_universe",
            "value": len(population_travel_origin_universe),
            "unit": "origin_cells",
        },
        {
            "measure": "selected_population_origins",
            "value": len(population_travel_origins),
            "unit": "origin_cells",
        },
        {
            "measure": "total_supported_population",
            "value": population_travel_total_population,
            "unit": "people",
        },
        {
            "measure": "selected_origin_population",
            "value": population_travel_selected_population,
            "unit": "people",
        },
        {
            "measure": "selected_population_fraction",
            "value": population_travel_represented_fraction,
            "unit": "fraction",
        },
        {
            "measure": "omitted_population",
            "value": (
                population_travel_total_population
                - population_travel_selected_population
            ),
            "unit": "people",
        },
    ]
)
display(population_travel_origin_summary)
display(
    population_travel_origins[
        [
            "origin_h3",
            "POPULATION",
            "ORIGIN_LATITUDE",
            "ORIGIN_LONGITUDE",
            "NEAREST_LAND_SOURCE_DISTANCE_KM",
            "SELECTION_REASON",
        ]
    ].head()
)

In [ ]:
POPULATION_TRAVEL_ORIGIN_BATCH_SIZE = 50
POPULATION_TRAVEL_DESTINATION_BATCH_SIZE = 50
POPULATION_TRAVEL_REQUEST_ATTEMPTS = 4
POPULATION_TRAVEL_REQUEST_TIMEOUT_SECONDS = 180
POPULATION_TRAVEL_REQUEST_INTERVAL_SECONDS = 0.25
REFRESH_POPULATION_TRAVEL_ROUTING = False

POPULATION_TRAVEL_DIR.mkdir(parents=True, exist_ok=True)
POPULATION_TRAVEL_ROUTING_CACHE_DIR.mkdir(parents=True, exist_ok=True)
population_travel_destinations = land_sources[["source_h3"]].copy().reset_index(drop=True)
destination_latlng = np.asarray(
    [h3.cell_to_latlng(cell) for cell in population_travel_destinations["source_h3"]],
    dtype="float64",
)
population_travel_destinations["DESTINATION_LATITUDE"] = destination_latlng[:, 0]
population_travel_destinations["DESTINATION_LONGITUDE"] = destination_latlng[:, 1]

destination_count = len(population_travel_destinations)
selected_origin_count = len(population_travel_origins)
routed_origin_count = np.zeros(destination_count, dtype="int64")
routed_population = np.zeros(destination_count, dtype="float64")
population_travel_demand = {
    minutes: np.zeros(destination_count, dtype="float64")
    for minutes in POPULATION_TRAVEL_DECAY_MINUTES
}
population_within_travel_time = {
    minutes: np.zeros(destination_count, dtype="float64")
    for minutes in POPULATION_TRAVEL_WITHIN_MINUTES
}
population_travel_cache_paths = []
population_origin_snap_distance_m = {}
population_destination_snap_distance_m = {}

for origin_batch_index, origin_start in enumerate(
    range(0, selected_origin_count, POPULATION_TRAVEL_ORIGIN_BATCH_SIZE)
):
    origin_batch = population_travel_origins.iloc[
        origin_start : origin_start + POPULATION_TRAVEL_ORIGIN_BATCH_SIZE
    ].copy()
    origin_coordinates = list(
        zip(
            origin_batch["ORIGIN_LONGITUDE"],
            origin_batch["ORIGIN_LATITUDE"],
            strict=True,
        )
    )
    for destination_batch_index, destination_start in enumerate(
        range(
            0,
            destination_count,
            POPULATION_TRAVEL_DESTINATION_BATCH_SIZE,
        )
    ):
        destination_batch = population_travel_destinations.iloc[
            destination_start : destination_start
            + POPULATION_TRAVEL_DESTINATION_BATCH_SIZE
        ].copy()
        destination_coordinates = list(
            zip(
                destination_batch["DESTINATION_LONGITUDE"],
                destination_batch["DESTINATION_LATITUDE"],
                strict=True,
            )
        )
        all_coordinates = origin_coordinates + destination_coordinates
        origin_batch_count = len(origin_batch)
        destination_batch_count = len(destination_batch)
        request_contract = {
            "endpoint": OSRM_TABLE_ENDPOINT,
            "profile": "driving",
            "annotations": ["duration"],
            "origin_resolution": POPULATION_TRAVEL_ORIGIN_RESOLUTION,
            "origins": [
                {
                    "origin_h3": row.origin_h3,
                    "population": round(float(row.POPULATION), 9),
                    "latitude": round(float(row.ORIGIN_LATITUDE), 7),
                    "longitude": round(float(row.ORIGIN_LONGITUDE), 7),
                }
                for row in origin_batch.itertuples(index=False)
            ],
            "destinations": [
                {
                    "source_h3": row.source_h3,
                    "latitude": round(float(row.DESTINATION_LATITUDE), 7),
                    "longitude": round(float(row.DESTINATION_LONGITUDE), 7),
                }
                for row in destination_batch.itertuples(index=False)
            ],
        }
        request_digest = hashlib.sha256(
            json.dumps(
                request_contract,
                sort_keys=True,
                separators=(",", ":"),
            ).encode("utf-8")
        ).hexdigest()
        cache_path = POPULATION_TRAVEL_ROUTING_CACHE_DIR / (
            f"osrm_population_table_o{origin_batch_index:03d}_"
            f"d{destination_batch_index:03d}_{request_digest[:16]}.json"
        )
        population_travel_cache_paths.append(cache_path)

        if cache_path.is_file() and not REFRESH_POPULATION_TRAVEL_ROUTING:
            cache_wrapper = json.loads(cache_path.read_text(encoding="utf-8"))
            if cache_wrapper.get("request_sha256") != request_digest:
                raise ValueError(
                    f"Population-travel cache contract mismatch: {cache_path}"
                )
            payload = cache_wrapper["response"]
        else:
            coordinate_string = ";".join(
                f"{float(longitude):.7f},{float(latitude):.7f}"
                for longitude, latitude in all_coordinates
            )
            source_indexes = ";".join(
                str(index) for index in range(origin_batch_count)
            )
            destination_indexes = ";".join(
                str(index)
                for index in range(
                    origin_batch_count,
                    origin_batch_count + destination_batch_count,
                )
            )
            request_url = f"{OSRM_TABLE_ENDPOINT}/{coordinate_string}"
            payload = None
            errors = []
            for attempt in range(POPULATION_TRAVEL_REQUEST_ATTEMPTS):
                try:
                    response = requests.get(
                        request_url,
                        params={
                            "sources": source_indexes,
                            "destinations": destination_indexes,
                            "annotations": "duration",
                        },
                        headers={
                            "User-Agent": (
                                "OrcaCastDataPrep/1.1 "
                                "(+https://github.com/orcacast)"
                            ),
                            "Accept": "application/json",
                        },
                        timeout=(
                            30,
                            POPULATION_TRAVEL_REQUEST_TIMEOUT_SECONDS,
                        ),
                    )
                    response.raise_for_status()
                    candidate_payload = response.json()
                    if candidate_payload.get("code") != "Ok":
                        raise RuntimeError(
                            "OSRM returned "
                            f"{candidate_payload.get('code')}: "
                            f"{candidate_payload.get('message')}"
                        )
                    payload = candidate_payload
                    time.sleep(POPULATION_TRAVEL_REQUEST_INTERVAL_SECONDS)
                    break
                except Exception as exc:
                    errors.append(
                        f"attempt {attempt + 1}: "
                        f"{type(exc).__name__}: {exc}"
                    )
                    if attempt + 1 < POPULATION_TRAVEL_REQUEST_ATTEMPTS:
                        time.sleep(min(2 ** (attempt + 1), 10))
            if payload is None:
                raise RuntimeError(
                    "OSRM population-travel batch failed: "
                    + "; ".join(errors)
                )
            cache_wrapper = {
                "schema_version": "0.1.0-prototype",
                "request_sha256": request_digest,
                "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
                "request_contract": request_contract,
                "response": payload,
            }
            temporary_cache_path = cache_path.with_suffix(".json.tmp")
            temporary_cache_path.write_text(
                json.dumps(
                    cache_wrapper,
                    separators=(",", ":"),
                    allow_nan=False,
                ),
                encoding="utf-8",
            )
            temporary_cache_path.replace(cache_path)

        if payload.get("code") != "Ok":
            raise ValueError(
                f"Population-travel routing failed in {cache_path.name}."
            )
        if len(payload.get("sources", [])) != origin_batch_count:
            raise ValueError(
                f"Population-travel origin count mismatch in {cache_path.name}."
            )
        if len(payload.get("destinations", [])) != destination_batch_count:
            raise ValueError(
                f"Population-travel destination count mismatch in {cache_path.name}."
            )
        if len(payload.get("durations", [])) != origin_batch_count:
            raise ValueError(
                f"Population-travel duration row mismatch in {cache_path.name}."
            )
        if any(
            len(row) != destination_batch_count
            for row in payload["durations"]
        ):
            raise ValueError(
                f"Population-travel duration column mismatch in {cache_path.name}."
            )

        for origin_position, origin_row in enumerate(
            origin_batch.itertuples(index=False)
        ):
            waypoint = payload["sources"][origin_position]
            snap_distance = waypoint.get("distance")
            if snap_distance is None:
                continue
            snap_distance = float(snap_distance)
            previous = population_origin_snap_distance_m.get(
                origin_row.origin_h3
            )
            if previous is not None and not np.isclose(
                previous, snap_distance, atol=0.01
            ):
                raise ValueError(
                    f"Population-origin snap drift for {origin_row.origin_h3}."
                )
            population_origin_snap_distance_m[
                origin_row.origin_h3
            ] = snap_distance

        for destination_position, destination_row in enumerate(
            destination_batch.itertuples(index=False)
        ):
            waypoint = payload["destinations"][destination_position]
            snap_distance = waypoint.get("distance")
            if snap_distance is None:
                continue
            snap_distance = float(snap_distance)
            previous = population_destination_snap_distance_m.get(
                destination_row.source_h3
            )
            if previous is not None and not np.isclose(
                previous, snap_distance, atol=0.01
            ):
                raise ValueError(
                    "Population-travel destination snap drift for "
                    f"{destination_row.source_h3}."
                )
            population_destination_snap_distance_m[
                destination_row.source_h3
            ] = snap_distance

        duration_seconds = np.asarray(
            [
                [
                    np.nan if value is None else float(value)
                    for value in row
                ]
                for row in payload["durations"]
            ],
            dtype="float64",
        )
        valid_route = np.isfinite(duration_seconds)
        origin_population = origin_batch["POPULATION"].to_numpy(
            dtype="float64"
        )[:, np.newaxis]
        destination_positions = np.arange(
            destination_start,
            destination_start + destination_batch_count,
        )
        routed_origin_count[destination_positions] += valid_route.sum(
            axis=0
        )
        routed_population[destination_positions] += np.where(
            valid_route,
            origin_population,
            0.0,
        ).sum(axis=0)
        safe_duration_seconds = np.where(
            valid_route,
            duration_seconds,
            0.0,
        )
        for minutes in POPULATION_TRAVEL_DECAY_MINUTES:
            population_travel_demand[minutes][
                destination_positions
            ] += np.where(
                valid_route,
                origin_population
                * np.exp(
                    -safe_duration_seconds / (60.0 * float(minutes))
                ),
                0.0,
            ).sum(axis=0)
        for minutes in POPULATION_TRAVEL_WITHIN_MINUTES:
            population_within_travel_time[minutes][
                destination_positions
            ] += np.where(
                valid_route
                & (duration_seconds <= float(minutes) * 60.0),
                origin_population,
                0.0,
            ).sum(axis=0)

population_travel_origins["ORIGIN_ROAD_SNAP_DISTANCE_M"] = (
    population_travel_origins["origin_h3"].map(
        population_origin_snap_distance_m
    )
)
source_population_travel = population_travel_destinations[
    ["source_h3"]
].copy()
source_population_travel["POPULATION_TRAVEL_SELECTED_ORIGIN_COUNT"] = (
    selected_origin_count
)
source_population_travel["POPULATION_TRAVEL_ROUTED_ORIGIN_COUNT"] = (
    routed_origin_count
)
source_population_travel["POPULATION_TRAVEL_TOTAL_SUPPORTED_POPULATION"] = (
    population_travel_total_population
)
source_population_travel["POPULATION_TRAVEL_SELECTED_ORIGIN_POPULATION"] = (
    population_travel_selected_population
)
source_population_travel["POPULATION_TRAVEL_ROUTED_SELECTED_POPULATION"] = (
    routed_population
)
source_population_travel[
    "POPULATION_TRAVEL_ORIGIN_REPRESENTATION_FRACTION"
] = population_travel_represented_fraction
source_population_travel[
    "POPULATION_TRAVEL_ROUTED_SELECTED_POPULATION_FRACTION"
] = routed_population / population_travel_selected_population
source_population_travel[
    "POPULATION_TRAVEL_ROUTED_TOTAL_POPULATION_FRACTION"
] = routed_population / population_travel_total_population
source_population_travel["POPULATION_TRAVEL_CONTEXT_AVAILABLE"] = (
    source_population_travel[
        "POPULATION_TRAVEL_ROUTED_SELECTED_POPULATION_FRACTION"
    ]
    >= POPULATION_TRAVEL_MIN_ROUTED_SHARE
)
source_population_travel["POPULATION_TRAVEL_MEASUREMENT_STATUS"] = np.select(
    [
        np.isclose(
            source_population_travel[
                "POPULATION_TRAVEL_ROUTED_SELECTED_POPULATION_FRACTION"
            ],
            1.0,
            rtol=0.0,
            atol=1e-12,
        ),
        source_population_travel[
            "POPULATION_TRAVEL_ROUTED_SELECTED_POPULATION_FRACTION"
        ]
        >= POPULATION_TRAVEL_MIN_ROUTED_SHARE,
    ],
    [
        "complete_selected_origin_routing",
        "coverage_gated_partial_selected_origin_routing",
    ],
    default="unavailable_insufficient_selected_origin_routing",
)
population_travel_available = source_population_travel[
    "POPULATION_TRAVEL_CONTEXT_AVAILABLE"
]
for minutes in POPULATION_TRAVEL_DECAY_MINUTES:
    source_population_travel[
        f"POPULATION_TRAVEL_DEMAND_{minutes}_MIN"
    ] = pd.Series(
        population_travel_demand[minutes],
        index=source_population_travel.index,
    ).where(population_travel_available)
for minutes in POPULATION_TRAVEL_WITHIN_MINUTES:
    source_population_travel[
        f"POPULATION_WITHIN_TRAVEL_TIME_{minutes}_MIN"
    ] = pd.Series(
        population_within_travel_time[minutes],
        index=source_population_travel.index,
    ).where(population_travel_available)
source_population_travel["ROUTING_PROFILE"] = "osrm_driving"
source_population_travel["POPULATION_TRAVEL_DESTINATION_BASIS"] = (
    "population_weighted_h3_r4_origins_to_source_h3_r7_centroid"
)
source_population_travel["POPULATION_TRAVEL_DECAY_KERNEL"] = (
    "origin_population * exp(-driving_time_minutes / decay_minutes)"
)

population_travel_numeric_columns = [
    column
    for column in source_population_travel.columns
    if column.startswith("POPULATION_TRAVEL_DEMAND_")
    or column.startswith("POPULATION_WITHIN_TRAVEL_TIME_")
]
if any(
    source_population_travel[column].dropna().lt(0).any()
    for column in population_travel_numeric_columns
):
    raise ValueError("Population-travel routing produced a negative measure.")
for column in [
    "POPULATION_TRAVEL_ORIGIN_REPRESENTATION_FRACTION",
    "POPULATION_TRAVEL_ROUTED_SELECTED_POPULATION_FRACTION",
    "POPULATION_TRAVEL_ROUTED_TOTAL_POPULATION_FRACTION",
]:
    if not source_population_travel[column].between(0.0, 1.0).all():
        raise ValueError(f"{column} is outside [0, 1].")
if not source_population_travel["source_h3"].is_unique:
    raise ValueError("Population-travel source output is not unique.")

population_travel_routing_cache_inventory = pd.DataFrame(
    [
        {
            "path": str(path.relative_to(REPO_ROOT)),
            "bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        }
        for path in population_travel_cache_paths
    ]
)
land_source_context = land_source_context.merge(
    source_population_travel,
    on="source_h3",
    how="left",
    validate="one_to_one",
)
if land_source_context[
    "POPULATION_TRAVEL_CONTEXT_AVAILABLE"
].isna().any():
    raise ValueError("A land source is missing its population-travel row.")

population_travel_source_summary_rows = (
    population_travel_origin_summary.to_dict(orient="records")
    + [
        {
            "measure": "routing_cache_batches",
            "value": len(population_travel_routing_cache_inventory),
            "unit": "batches",
        },
        {
            "measure": "land_sources",
            "value": len(source_population_travel),
            "unit": "source_cells",
        },
        {
            "measure": "population_travel_context_available",
            "value": int(population_travel_available.sum()),
            "unit": "source_cells",
        },
        {
            "measure": "median_routed_selected_population_fraction",
            "value": float(
                source_population_travel[
                    "POPULATION_TRAVEL_ROUTED_SELECTED_POPULATION_FRACTION"
                ].median()
            ),
            "unit": "fraction",
        },
    ]
)
for minutes in POPULATION_TRAVEL_DECAY_MINUTES:
    population_travel_source_summary_rows.append(
        {
            "measure": f"median_population_travel_demand_{minutes}_min",
            "value": float(
                source_population_travel[
                    f"POPULATION_TRAVEL_DEMAND_{minutes}_MIN"
                ].median()
            ),
            "unit": "decayed_people",
        }
    )
for minutes in POPULATION_TRAVEL_WITHIN_MINUTES:
    population_travel_source_summary_rows.append(
        {
            "measure": f"median_population_within_travel_time_{minutes}_min",
            "value": float(
                source_population_travel[
                    f"POPULATION_WITHIN_TRAVEL_TIME_{minutes}_MIN"
                ].median()
            ),
            "unit": "people",
        }
    )
population_travel_source_summary = pd.DataFrame(
    population_travel_source_summary_rows
)
display(population_travel_source_summary)

In [ ]:
road_routing_available = land_source_context["ROAD_ROUTING_AVAILABLE"].eq(True)
city_travel_available = land_source_context["CITY_TRAVEL_ROUTING_AVAILABLE"].eq(True)
transport_component_available = road_routing_available & city_travel_available
land_source_context["ROAD_PROXIMITY_COMPONENT"] = np.exp(
    -land_source_context["ROAD_SNAP_DISTANCE_FROM_SOURCE_CENTROID_M"]
    / (ROAD_DISTANCE_DECAY_KM * 1000.0)
).where(road_routing_available)
land_source_context["CITY_TRAVEL_ACCESS_COMPONENT"] = np.exp(
    -land_source_context["MIN_CITY_TRAVEL_TIME_MIN"]
    / CITY_TRAVEL_TIME_DECAY_MINUTES
).where(city_travel_available)
land_source_context["ROAD_AND_CITY_ACCESS_COMPONENT"] = (
    land_source_context["ROAD_PROXIMITY_COMPONENT"]
    * land_source_context["CITY_TRAVEL_ACCESS_COMPONENT"]
).where(transport_component_available)
land_source_context["EXACT_POPULATION_ROAD_AND_CITY_COMPONENT"] = (
    land_source_context["POPULATION_COMPONENT_EXACT_CELL"]
    * land_source_context["ROAD_AND_CITY_ACCESS_COMPONENT"]
).where(
    transport_component_available
    & land_source_context["POPULATION_SOURCE_AVAILABLE_EXACT_CELL"]
)
land_source_context["DECAYED_25_KM_POPULATION_ROAD_AND_CITY_COMPONENT"] = (
    land_source_context["POPULATION_COMPONENT_DECAYED_25_KM"]
    * land_source_context["ROAD_AND_CITY_ACCESS_COMPONENT"]
).where(
    transport_component_available
    & land_source_context["POPULATION_SOURCE_AVAILABLE_DECAYED_25_KM"]
)
land_source_context["ROAD_PROXIMITY_AVAILABLE"] = road_routing_available
land_source_context["CITY_TRAVEL_ACCESS_AVAILABLE"] = city_travel_available
land_source_context["ROAD_AND_CITY_ACCESS_AVAILABLE"] = transport_component_available
land_source_context["EXACT_POPULATION_ROAD_AND_CITY_AVAILABLE"] = (
    transport_component_available
    & land_source_context["POPULATION_SOURCE_AVAILABLE_EXACT_CELL"]
)
land_source_context["DECAYED_25_KM_POPULATION_ROAD_AND_CITY_AVAILABLE"] = (
    transport_component_available
    & land_source_context["POPULATION_SOURCE_AVAILABLE_DECAYED_25_KM"]
)

transport_pressure_specs = {
    "ROAD_PROXIMITY": {
        "component_column": "ROAD_PROXIMITY_COMPONENT",
        "availability_column": "ROAD_PROXIMITY_AVAILABLE",
        "source_scope": "transport_only",
        "formula": f"exp(-road_snap_distance_km / {ROAD_DISTANCE_DECAY_KM:g})",
    },
    "CITY_TRAVEL_ACCESS": {
        "component_column": "CITY_TRAVEL_ACCESS_COMPONENT",
        "availability_column": "CITY_TRAVEL_ACCESS_AVAILABLE",
        "source_scope": "transport_only",
        "formula": f"exp(-minimum_city_travel_time_minutes / {CITY_TRAVEL_TIME_DECAY_MINUTES:g})",
    },
    "ROAD_AND_CITY_ACCESS": {
        "component_column": "ROAD_AND_CITY_ACCESS_COMPONENT",
        "availability_column": "ROAD_AND_CITY_ACCESS_AVAILABLE",
        "source_scope": "transport_only",
        "formula": "ROAD_PROXIMITY_COMPONENT * CITY_TRAVEL_ACCESS_COMPONENT",
    },
    "EXACT_POPULATION_ROAD_AND_CITY": {
        "component_column": "EXACT_POPULATION_ROAD_AND_CITY_COMPONENT",
        "availability_column": "EXACT_POPULATION_ROAD_AND_CITY_AVAILABLE",
        "source_scope": "population_transport",
        "formula": "POPULATION_COMPONENT_EXACT_CELL * ROAD_AND_CITY_ACCESS_COMPONENT",
    },
    "DECAYED_25_KM_POPULATION_ROAD_AND_CITY": {
        "component_column": "DECAYED_25_KM_POPULATION_ROAD_AND_CITY_COMPONENT",
        "availability_column": "DECAYED_25_KM_POPULATION_ROAD_AND_CITY_AVAILABLE",
        "source_scope": "population_transport",
        "formula": "POPULATION_COMPONENT_DECAYED_25_KM * ROAD_AND_CITY_ACCESS_COMPONENT",
    },
}
for variant, spec in transport_pressure_specs.items():
    available = land_source_context[spec["availability_column"]].eq(True)
    component = land_source_context[spec["component_column"]]
    if not component.loc[available].between(0.0, 1.0).all():
        raise ValueError(f"{variant} transport components must be within [0, 1].")
    if not component.loc[~available].isna().all():
        raise ValueError(f"Unavailable {variant} components must remain null.")

transport_component_summary = pd.DataFrame(
    [
        {
            "variant": variant,
            "source_scope": spec["source_scope"],
            "formula": spec["formula"],
            "available_sources": int(
                land_source_context[spec["availability_column"]].eq(True).sum()
            ),
            "median_component": float(
                land_source_context[spec["component_column"]].median()
            ),
            "p10_component": float(
                land_source_context[spec["component_column"]].quantile(0.10)
            ),
            "p90_component": float(
                land_source_context[spec["component_column"]].quantile(0.90)
            ),
        }
        for variant, spec in transport_pressure_specs.items()
    ]
)
display(transport_component_summary)

In [ ]:
population_travel_cap_rows = []
population_travel_new_specs = {}
for minutes in POPULATION_TRAVEL_DECAY_MINUTES:
    raw_column = f"POPULATION_TRAVEL_DEMAND_{minutes}_MIN"
    component_column = (
        f"POPULATION_TRAVEL_DEMAND_COMPONENT_{minutes}_MIN"
    )
    availability_column = (
        f"POPULATION_TRAVEL_DEMAND_AVAILABLE_{minutes}_MIN"
    )
    available = (
        land_source_context["POPULATION_TRAVEL_CONTEXT_AVAILABLE"].eq(True)
        & land_source_context[raw_column].notna()
    )
    log_values = np.log1p(
        land_source_context.loc[available, raw_column].astype("float64")
    )
    cap = float(log_values.quantile(POPULATION_CAP_QUANTILE))
    if not np.isfinite(cap) or cap <= 0.0:
        raise ValueError(
            f"Invalid population-travel log1p cap for {minutes} minutes."
        )
    land_source_context[availability_column] = available
    land_source_context[component_column] = (
        np.log1p(land_source_context[raw_column]) / cap
    ).clip(0.0, 1.0).where(available)
    variant = f"POPULATION_TRAVEL_DEMAND_{minutes}_MIN"
    population_travel_new_specs[variant] = {
        "component_column": component_column,
        "availability_column": availability_column,
        "source_scope": "population_travel",
        "formula": (
            "clip(log1p(sum(origin_population * "
            f"exp(-driving_time_minutes / {minutes}))) / "
            f"q{int(POPULATION_CAP_QUANTILE * 100)}_source_log1p, 0, 1)"
        ),
    }
    population_travel_cap_rows.append(
        {
            "variant": variant,
            "decay_minutes": minutes,
            "population_log1p_cap_quantile": POPULATION_CAP_QUANTILE,
            "population_log1p_cap": cap,
            "available_sources": int(available.sum()),
        }
    )

primary_population_travel_component_column = (
    f"POPULATION_TRAVEL_DEMAND_COMPONENT_"
    f"{PRIMARY_POPULATION_TRAVEL_DECAY_MINUTES}_MIN"
)
primary_population_travel_available_column = (
    f"POPULATION_TRAVEL_DEMAND_AVAILABLE_"
    f"{PRIMARY_POPULATION_TRAVEL_DECAY_MINUTES}_MIN"
)
road_adjusted_population_travel_available = (
    land_source_context[primary_population_travel_available_column].eq(True)
    & land_source_context["ROAD_PROXIMITY_AVAILABLE"].eq(True)
)
land_source_context[
    "POPULATION_TRAVEL_DEMAND_120_MIN_ROAD_ADJUSTED_COMPONENT"
] = (
    land_source_context[primary_population_travel_component_column]
    * land_source_context["ROAD_PROXIMITY_COMPONENT"]
).where(road_adjusted_population_travel_available)
land_source_context[
    "POPULATION_TRAVEL_DEMAND_120_MIN_ROAD_ADJUSTED_AVAILABLE"
] = road_adjusted_population_travel_available
population_travel_new_specs[
    "POPULATION_TRAVEL_DEMAND_120_MIN_ROAD_ADJUSTED"
] = {
    "component_column": (
        "POPULATION_TRAVEL_DEMAND_120_MIN_ROAD_ADJUSTED_COMPONENT"
    ),
    "availability_column": (
        "POPULATION_TRAVEL_DEMAND_120_MIN_ROAD_ADJUSTED_AVAILABLE"
    ),
    "source_scope": "population_travel",
    "formula": (
        "POPULATION_TRAVEL_DEMAND_COMPONENT_120_MIN * "
        "ROAD_PROXIMITY_COMPONENT"
    ),
}

for variant, spec in population_travel_new_specs.items():
    available = land_source_context[spec["availability_column"]].eq(True)
    component = land_source_context[spec["component_column"]]
    if not component.loc[available].between(0.0, 1.0).all():
        raise ValueError(
            f"{variant} population-travel components must be within [0, 1]."
        )
    if not component.loc[~available].isna().all():
        raise ValueError(
            f"Unavailable {variant} population-travel components must be null."
        )
transport_pressure_specs.update(population_travel_new_specs)
population_travel_cap_summary = pd.DataFrame(
    population_travel_cap_rows
)
population_travel_component_summary = pd.DataFrame(
    [
        {
            "variant": variant,
            "source_scope": spec["source_scope"],
            "formula": spec["formula"],
            "available_sources": int(
                land_source_context[
                    spec["availability_column"]
                ].eq(True).sum()
            ),
            "median_component": float(
                land_source_context[spec["component_column"]].median()
            ),
            "p10_component": float(
                land_source_context[
                    spec["component_column"]
                ].quantile(0.10)
            ),
            "p90_component": float(
                land_source_context[
                    spec["component_column"]
                ].quantile(0.90)
            ),
        }
        for variant, spec in population_travel_new_specs.items()
    ]
)
transport_component_summary = pd.concat(
    [
        transport_component_summary,
        population_travel_component_summary,
    ],
    ignore_index=True,
)
display(population_travel_cap_summary)
display(population_travel_component_summary)

In [ ]:
if land_pairs["weight_static_viewability"].isna().any():
    raise ValueError("Static viewability contains null weights.")
if not land_pairs["weight_static_viewability"].between(0.0, 1.0).all():
    raise ValueError("Static viewability weights must be bounded to [0, 1].")

pair_context = land_pairs.merge(
    land_source_context[
        ["source_h3", "LAND_SOURCE_COMPOSITE_AVAILABLE", "LAND_SOURCE_COMPOSITE"]
    ],
    on="source_h3",
    how="left",
    validate="many_to_one",
)
if pair_context["LAND_SOURCE_COMPOSITE_AVAILABLE"].isna().any():
    raise ValueError("A viewshed source was missing from the source-context universe.")

pair_context["CONTEXT_STATIC_SUPPORT"] = pair_context[
    "weight_static_viewability"
].where(pair_context["LAND_SOURCE_COMPOSITE_AVAILABLE"])
pair_context["LAND_VIEWING_PRESSURE_CONTRIBUTION"] = (
    pair_context["weight_static_viewability"]
    * pair_context["LAND_SOURCE_COMPOSITE"]
)

target_pressure = (
    pair_context.groupby("target_h3", as_index=False)
    .agg(
        VIEWSHED_SOURCE_COUNT=("source_h3", "nunique"),
        VIEWSHED_STATIC_SUPPORT_SUM=("weight_static_viewability", "sum"),
        CONTEXT_SOURCE_COUNT=(
            "LAND_SOURCE_COMPOSITE_AVAILABLE", lambda values: int(values.eq(True).sum())
        ),
        CONTEXT_STATIC_SUPPORT_SUM=("CONTEXT_STATIC_SUPPORT", "sum"),
        LAND_VIEWING_PRESSURE_RAW=(
            "LAND_VIEWING_PRESSURE_CONTRIBUTION", lambda values: values.sum(min_count=1)
        ),
    )
    .sort_values("target_h3")
    .reset_index(drop=True)
)
target_pressure.insert(1, "H3_RESOLUTION", 7)
target_pressure["LAND_SOURCE_CONTEXT_COVERAGE"] = (
    target_pressure["CONTEXT_STATIC_SUPPORT_SUM"]
    / target_pressure["VIEWSHED_STATIC_SUPPORT_SUM"].replace(0.0, np.nan)
)
target_pressure["LAND_VIEWING_PRESSURE_PERCENTILE"] = target_pressure[
    "LAND_VIEWING_PRESSURE_RAW"
].rank(method="average", pct=True)
target_pressure["LAND_VIEWING_PRESSURE_STATUS"] = np.where(
    target_pressure["LAND_VIEWING_PRESSURE_RAW"].notna(),
    "derived_partial_context",
    "unavailable_no_usable_land_source_context",
)
target_pressure["MEASUREMENT_STATUS"] = "derived"

if not target_pressure["target_h3"].is_unique:
    raise ValueError("Target pressure is not unique by target_h3.")
if not target_pressure["target_h3"].map(h3.get_resolution).eq(7).all():
    raise ValueError("Target pressure contains a non-R7 H3 index.")
if not target_pressure["LAND_SOURCE_CONTEXT_COVERAGE"].dropna().between(0.0, 1.0).all():
    raise ValueError("Target source-context coverage must be bounded to [0, 1].")
if not target_pressure.loc[
    target_pressure["CONTEXT_SOURCE_COUNT"].eq(0), "LAND_VIEWING_PRESSURE_RAW"
].isna().all():
    raise ValueError("Targets without usable source context must retain null pressure.")

In [ ]:
source_variant_columns = ["source_h3"]
for variant in population_variant_specs:
    source_variant_columns.extend(
        [
            f"POPULATION_SOURCE_AVAILABLE_{variant}",
            f"POPULATION_COMPONENT_{variant}",
            f"LAND_SOURCE_ACCESS_GATED_AVAILABLE_{variant}",
            f"LAND_SOURCE_ACCESS_GATED_COMPONENT_{variant}",
        ]
    )
pair_context = land_pairs.merge(
    land_source_context[source_variant_columns],
    on="source_h3",
    how="left",
    validate="many_to_one",
)

target_pressure = (
    pair_context.groupby("target_h3", as_index=False)
    .agg(
        VIEWSHED_SOURCE_COUNT=("source_h3", "nunique"),
        VIEWSHED_STATIC_SUPPORT_SUM=("weight_static_viewability", "sum"),
    )
    .sort_values("target_h3")
    .reset_index(drop=True)
)
target_pressure.insert(1, "H3_RESOLUTION", 7)
target_variant_summary_rows = []
for variant in population_variant_specs:
    population_available_column = f"POPULATION_SOURCE_AVAILABLE_{variant}"
    population_component_column = f"POPULATION_COMPONENT_{variant}"
    access_available_column = f"LAND_SOURCE_ACCESS_GATED_AVAILABLE_{variant}"
    access_component_column = f"LAND_SOURCE_ACCESS_GATED_COMPONENT_{variant}"
    pair_context["_POPULATION_STATIC_SUPPORT"] = pair_context[
        "weight_static_viewability"
    ].where(pair_context[population_available_column])
    pair_context["_POPULATION_PRESSURE_CONTRIBUTION"] = (
        pair_context["weight_static_viewability"]
        * pair_context[population_component_column]
    )
    pair_context["_ACCESS_GATED_STATIC_SUPPORT"] = pair_context[
        "weight_static_viewability"
    ].where(pair_context[access_available_column])
    pair_context["_ACCESS_GATED_PRESSURE_CONTRIBUTION"] = (
        pair_context["weight_static_viewability"]
        * pair_context[access_component_column]
    )

    variant_target = (
        pair_context.groupby("target_h3", as_index=False)
        .agg(
            POPULATION_SOURCE_COUNT=(
                population_available_column,
                lambda values: int(values.eq(True).sum()),
            ),
            POPULATION_STATIC_SUPPORT_SUM=("_POPULATION_STATIC_SUPPORT", "sum"),
            POPULATION_PRESSURE_RAW=(
                "_POPULATION_PRESSURE_CONTRIBUTION",
                lambda values: values.sum(min_count=1),
            ),
            ACCESS_GATED_SOURCE_COUNT=(
                access_available_column,
                lambda values: int(values.eq(True).sum()),
            ),
            ACCESS_GATED_STATIC_SUPPORT_SUM=(
                "_ACCESS_GATED_STATIC_SUPPORT", "sum"
            ),
            ACCESS_GATED_PRESSURE_RAW=(
                "_ACCESS_GATED_PRESSURE_CONTRIBUTION",
                lambda values: values.sum(min_count=1),
            ),
        )
        .rename(
            columns={
                column: f"{column}_{variant}"
                for column in [
                    "POPULATION_SOURCE_COUNT",
                    "POPULATION_STATIC_SUPPORT_SUM",
                    "POPULATION_PRESSURE_RAW",
                    "ACCESS_GATED_SOURCE_COUNT",
                    "ACCESS_GATED_STATIC_SUPPORT_SUM",
                    "ACCESS_GATED_PRESSURE_RAW",
                ]
            }
        )
    )
    target_pressure = target_pressure.merge(
        variant_target, on="target_h3", how="left", validate="one_to_one"
    )
    population_raw_column = f"POPULATION_PRESSURE_RAW_{variant}"
    access_raw_column = f"ACCESS_GATED_PRESSURE_RAW_{variant}"
    target_pressure[f"POPULATION_CONTEXT_COVERAGE_{variant}"] = (
        target_pressure[f"POPULATION_STATIC_SUPPORT_SUM_{variant}"]
        / target_pressure["VIEWSHED_STATIC_SUPPORT_SUM"].replace(0.0, np.nan)
    )
    target_pressure[f"ACCESS_GATED_CONTEXT_COVERAGE_{variant}"] = (
        target_pressure[f"ACCESS_GATED_STATIC_SUPPORT_SUM_{variant}"]
        / target_pressure["VIEWSHED_STATIC_SUPPORT_SUM"].replace(0.0, np.nan)
    )
    target_pressure[f"POPULATION_PRESSURE_PERCENTILE_{variant}"] = (
        target_pressure[population_raw_column].rank(method="average", pct=True)
    )
    target_pressure[f"ACCESS_GATED_PRESSURE_PERCENTILE_{variant}"] = (
        target_pressure[access_raw_column].rank(method="average", pct=True)
    )
    target_variant_summary_rows.append(
        {
            "variant": variant,
            "population_pressure_available_targets": int(
                target_pressure[population_raw_column].notna().sum()
            ),
            "access_gated_pressure_available_targets": int(
                target_pressure[access_raw_column].notna().sum()
            ),
            "median_population_context_coverage": float(
                target_pressure[f"POPULATION_CONTEXT_COVERAGE_{variant}"].median()
            ),
            "median_access_gated_context_coverage": float(
                target_pressure[f"ACCESS_GATED_CONTEXT_COVERAGE_{variant}"].median()
            ),
        }
    )

target_variant_summary = pd.DataFrame(target_variant_summary_rows)
primary_access_raw_column = (
    f"ACCESS_GATED_PRESSURE_RAW_{PRIMARY_POPULATION_VARIANT}"
)
primary_access_percentile_column = (
    f"ACCESS_GATED_PRESSURE_PERCENTILE_{PRIMARY_POPULATION_VARIANT}"
)
target_pressure["CONTEXT_SOURCE_COUNT"] = target_pressure[
    f"ACCESS_GATED_SOURCE_COUNT_{PRIMARY_POPULATION_VARIANT}"
]
target_pressure["CONTEXT_STATIC_SUPPORT_SUM"] = target_pressure[
    f"ACCESS_GATED_STATIC_SUPPORT_SUM_{PRIMARY_POPULATION_VARIANT}"
]
target_pressure["LAND_SOURCE_CONTEXT_COVERAGE"] = target_pressure[
    f"ACCESS_GATED_CONTEXT_COVERAGE_{PRIMARY_POPULATION_VARIANT}"
]
target_pressure["LAND_VIEWING_PRESSURE_RAW"] = target_pressure[
    primary_access_raw_column
]
target_pressure["LAND_VIEWING_PRESSURE_PERCENTILE"] = target_pressure[
    primary_access_percentile_column
]
target_pressure["LAND_VIEWING_PRESSURE_STATUS"] = np.where(
    target_pressure["LAND_VIEWING_PRESSURE_RAW"].notna(),
    f"derived_partial_context_{PRIMARY_POPULATION_VARIANT.lower()}",
    "unavailable_no_usable_land_source_context",
)
target_pressure["PRIMARY_POPULATION_VARIANT"] = PRIMARY_POPULATION_VARIANT
target_pressure["MEASUREMENT_STATUS"] = "derived"

coverage_columns = [
    column
    for column in target_pressure.columns
    if "CONTEXT_COVERAGE_" in column or column == "LAND_SOURCE_CONTEXT_COVERAGE"
]
if not all(
    target_pressure[column].dropna().between(0.0, 1.0).all()
    for column in coverage_columns
):
    raise ValueError("A target pressure context-coverage value is outside [0, 1].")
if not target_pressure.loc[
    target_pressure["CONTEXT_SOURCE_COUNT"].eq(0), "LAND_VIEWING_PRESSURE_RAW"
].isna().all():
    raise ValueError("Primary targets without usable source context must remain null.")
display(target_variant_summary)

In [ ]:
transport_source_columns = ["source_h3"]
for spec in transport_pressure_specs.values():
    transport_source_columns.extend(
        [spec["component_column"], spec["availability_column"]]
    )
transport_source_columns = list(dict.fromkeys(transport_source_columns))
transport_pair_context = land_pairs.merge(
    land_source_context[transport_source_columns],
    on="source_h3",
    how="left",
    validate="many_to_one",
)
transport_target_summary_rows = []
for variant, spec in transport_pressure_specs.items():
    available_column = spec["availability_column"]
    component_column = spec["component_column"]
    transport_pair_context["_TRANSPORT_STATIC_SUPPORT"] = transport_pair_context[
        "weight_static_viewability"
    ].where(transport_pair_context[available_column])
    transport_pair_context["_TRANSPORT_PRESSURE_CONTRIBUTION"] = (
        transport_pair_context["weight_static_viewability"]
        * transport_pair_context[component_column]
    )
    variant_target = (
        transport_pair_context.groupby("target_h3", as_index=False)
        .agg(
            TRANSPORT_SOURCE_COUNT=(
                available_column, lambda values: int(values.eq(True).sum())
            ),
            TRANSPORT_STATIC_SUPPORT_SUM=("_TRANSPORT_STATIC_SUPPORT", "sum"),
            TRANSPORT_PRESSURE_RAW=(
                "_TRANSPORT_PRESSURE_CONTRIBUTION",
                lambda values: values.sum(min_count=1),
            ),
        )
        .rename(
            columns={
                "TRANSPORT_SOURCE_COUNT": f"TRANSPORT_SOURCE_COUNT_{variant}",
                "TRANSPORT_STATIC_SUPPORT_SUM": f"TRANSPORT_STATIC_SUPPORT_SUM_{variant}",
                "TRANSPORT_PRESSURE_RAW": f"TRANSPORT_PRESSURE_RAW_{variant}",
            }
        )
    )
    target_pressure = target_pressure.merge(
        variant_target, on="target_h3", how="left", validate="one_to_one"
    )
    target_pressure = target_pressure.copy()
    raw_column = f"TRANSPORT_PRESSURE_RAW_{variant}"
    coverage_column = f"TRANSPORT_CONTEXT_COVERAGE_{variant}"
    target_pressure[coverage_column] = (
        target_pressure[f"TRANSPORT_STATIC_SUPPORT_SUM_{variant}"]
        / target_pressure["VIEWSHED_STATIC_SUPPORT_SUM"].replace(0.0, np.nan)
    )
    target_pressure[f"TRANSPORT_PRESSURE_PERCENTILE_{variant}"] = (
        target_pressure[raw_column].rank(method="average", pct=True)
    )
    transport_target_summary_rows.append(
        {
            "variant": variant,
            "source_scope": spec["source_scope"],
            "available_targets": int(target_pressure[raw_column].notna().sum()),
            "median_context_coverage": float(target_pressure[coverage_column].median()),
            "maximum_pressure_raw": float(target_pressure[raw_column].max()),
        }
    )
transport_target_summary = pd.DataFrame(transport_target_summary_rows)
transport_coverage_columns = [
    f"TRANSPORT_CONTEXT_COVERAGE_{variant}"
    for variant in transport_pressure_specs
]
if not all(
    target_pressure[column].dropna().between(0.0, 1.0).all()
    for column in transport_coverage_columns
):
    raise ValueError("A transport target context-coverage value is outside [0, 1].")
display(transport_target_summary)

In [ ]:
target_pressure_summary = pd.DataFrame(
    [
        {
            "target_cells": len(target_pressure),
            "pressure_available": int(target_pressure["LAND_VIEWING_PRESSURE_RAW"].notna().sum()),
            "pressure_unavailable": int(target_pressure["LAND_VIEWING_PRESSURE_RAW"].isna().sum()),
            "median_context_coverage": float(target_pressure["LAND_SOURCE_CONTEXT_COVERAGE"].median()),
            "mean_context_coverage": float(target_pressure["LAND_SOURCE_CONTEXT_COVERAGE"].mean()),
            "maximum_pressure_raw": float(target_pressure["LAND_VIEWING_PRESSURE_RAW"].max()),
        }
    ]
)
display(target_pressure_summary)
display(
    target_pressure.sort_values("LAND_VIEWING_PRESSURE_RAW", ascending=False).head(12)
)

## Date-specific weather and daylight viewability

This section constructs two transparent daily variants before sightings are loaded. The **core** physical layer combines static source-target viewability with a pair-distance-aware mean-visibility support curve and daylight fraction. A minimum-visibility version is retained as a conservative sensitivity diagnostic. The **extended** layer adds soft wind and precipitation support. Calendar and the primary population/access source component enter only the reporting-opportunity contribution; they never alter physical visibility. None of these indices is a detection probability.

In [ ]:
DYNAMIC_SAMPLE_DATE = dynamic_context_end_date
dynamic_date_slug = DYNAMIC_SAMPLE_DATE.date().isoformat()
DYNAMIC_SOURCE_SAMPLE_PATH = (
    DYNAMIC_VIEWABILITY_DIR
    / f"land_source_dynamic_context_h3_r7_{dynamic_date_slug}_prototype.parquet"
)
DYNAMIC_TARGET_SAMPLE_PATH = (
    DYNAMIC_VIEWABILITY_DIR
    / f"target_land_viewability_h3_r7_{dynamic_date_slug}_prototype.parquet"
)
DYNAMIC_VIEWABILITY_METADATA_PATH = (
    DYNAMIC_VIEWABILITY_DIR
    / f"target_land_viewability_h3_r7_{dynamic_date_slug}_prototype.metadata.json"
)


def stable_sigmoid(values: pd.Series | np.ndarray) -> np.ndarray:
    array = np.asarray(values, dtype="float64")
    return 1.0 / (1.0 + np.exp(-np.clip(array, -60.0, 60.0)))


dynamic_source_modulators = load_land_source_dynamic_context_for_date(
    DYNAMIC_SAMPLE_DATE
).merge(
    land_source_context[
        [
            "source_h3",
            "LAND_SOURCE_COMPOSITE_AVAILABLE",
            "LAND_SOURCE_COMPOSITE",
        ]
    ],
    on="source_h3",
    how="left",
    validate="one_to_one",
)
dynamic_source_modulators["DAYLIGHT_SUPPORT"] = dynamic_source_modulators[
    "DAYLIGHT_FRACTION"
].clip(0.0, 1.0)
dynamic_source_modulators["CALENDAR_ACTIVITY_SUPPORT"] = (
    dynamic_source_modulators["calendar_effort_weight"].clip(0.0, 1.0)
)
wind_available = (
    dynamic_source_modulators["WEATHER_CONTEXT_AVAILABLE"]
    & dynamic_source_modulators["WIND_SPEED_10M_MS_MEAN"].notna()
)
precip_available = (
    dynamic_source_modulators["WEATHER_CONTEXT_AVAILABLE"]
    & dynamic_source_modulators["PRECIP_MM_DAY_ESTIMATE"].notna()
)
dynamic_source_modulators["WIND_SUPPORT"] = pd.Series(
    stable_sigmoid(
        (
            WIND_SUPPORT_MIDPOINT_MS
            - dynamic_source_modulators["WIND_SPEED_10M_MS_MEAN"]
        )
        / WIND_SUPPORT_SLOPE_MS
    ),
    index=dynamic_source_modulators.index,
).where(wind_available)
dynamic_source_modulators["PRECIP_SUPPORT"] = (
    1.0
    / (
        1.0
        + dynamic_source_modulators["PRECIP_MM_DAY_ESTIMATE"].clip(lower=0.0)
        / PRECIP_SUPPORT_HALF_MM_DAY
    )
).where(precip_available)
dynamic_source_modulators["CONDITIONS_SUPPORT"] = (
    dynamic_source_modulators["WIND_SUPPORT"] ** CONDITIONS_WIND_EXPONENT
    * dynamic_source_modulators["PRECIP_SUPPORT"]
    ** CONDITIONS_PRECIP_EXPONENT
)
dynamic_source_modulators["CORE_DYNAMIC_CONTEXT_AVAILABLE"] = (
    dynamic_source_modulators["WEATHER_CONTEXT_AVAILABLE"]
    & dynamic_source_modulators["DAYLIGHT_CONTEXT_AVAILABLE"]
    & dynamic_source_modulators["VISIBILITY_KM_MEAN"].notna()
    & dynamic_source_modulators["DAYLIGHT_SUPPORT"].notna()
)
dynamic_source_modulators["EXTENDED_DYNAMIC_CONTEXT_AVAILABLE"] = (
    dynamic_source_modulators["CORE_DYNAMIC_CONTEXT_AVAILABLE"]
    & dynamic_source_modulators["CONDITIONS_SUPPORT"].notna()
)
dynamic_source_modulators["CORE_REPORTING_CONTEXT_AVAILABLE"] = (
    dynamic_source_modulators["CORE_DYNAMIC_CONTEXT_AVAILABLE"]
    & dynamic_source_modulators["CALENDAR_CONTEXT_AVAILABLE"]
    & dynamic_source_modulators["LAND_SOURCE_COMPOSITE_AVAILABLE"]
)
dynamic_source_modulators["EXTENDED_REPORTING_CONTEXT_AVAILABLE"] = (
    dynamic_source_modulators["EXTENDED_DYNAMIC_CONTEXT_AVAILABLE"]
    & dynamic_source_modulators["CALENDAR_CONTEXT_AVAILABLE"]
    & dynamic_source_modulators["LAND_SOURCE_COMPOSITE_AVAILABLE"]
)

for column in [
    "DAYLIGHT_SUPPORT",
    "CALENDAR_ACTIVITY_SUPPORT",
    "WIND_SUPPORT",
    "PRECIP_SUPPORT",
    "CONDITIONS_SUPPORT",
]:
    values = dynamic_source_modulators[column].dropna()
    if not values.between(0.0, 1.0).all():
        raise ValueError(f"{column} must be bounded to [0, 1].")

dynamic_source_modulator_summary = pd.DataFrame(
    [
        {
            "sample_date": DYNAMIC_SAMPLE_DATE,
            "land_source_cells": len(dynamic_source_modulators),
            "core_dynamic_context_available": int(
                dynamic_source_modulators["CORE_DYNAMIC_CONTEXT_AVAILABLE"].sum()
            ),
            "extended_dynamic_context_available": int(
                dynamic_source_modulators[
                    "EXTENDED_DYNAMIC_CONTEXT_AVAILABLE"
                ].sum()
            ),
            "core_reporting_context_available": int(
                dynamic_source_modulators[
                    "CORE_REPORTING_CONTEXT_AVAILABLE"
                ].sum()
            ),
            "median_daylight_support": float(
                dynamic_source_modulators["DAYLIGHT_SUPPORT"].median()
            ),
            "median_wind_support": float(
                dynamic_source_modulators["WIND_SUPPORT"].median()
            ),
            "median_precip_support": float(
                dynamic_source_modulators["PRECIP_SUPPORT"].median()
            ),
        }
    ]
)
display(dynamic_source_modulator_summary)

In [ ]:
source_cell_order = pd.Index(land_sources["source_h3"].astype(str))
target_cell_order = pd.Index(target_pressure["target_h3"].astype(str))
source_codes = pd.Categorical(
    land_pairs["source_h3"].astype(str), categories=source_cell_order
).codes
target_codes = pd.Categorical(
    land_pairs["target_h3"].astype(str), categories=target_cell_order
).codes
if (source_codes < 0).any() or (target_codes < 0).any():
    raise ValueError("A viewshed pair was absent from the source or target universe.")

source_latlng = np.radians(
    np.asarray([h3.cell_to_latlng(cell) for cell in source_cell_order], dtype="float64")
)
target_latlng = np.radians(
    np.asarray([h3.cell_to_latlng(cell) for cell in target_cell_order], dtype="float64")
)
source_latitude = source_latlng[source_codes, 0]
source_longitude = source_latlng[source_codes, 1]
target_latitude = target_latlng[target_codes, 0]
target_longitude = target_latlng[target_codes, 1]
latitude_delta = target_latitude - source_latitude
longitude_delta = target_longitude - source_longitude
haversine_a = (
    np.sin(latitude_delta / 2.0) ** 2
    + np.cos(source_latitude)
    * np.cos(target_latitude)
    * np.sin(longitude_delta / 2.0) ** 2
)
source_target_distance_km = (
    2.0
    * EARTH_MEAN_RADIUS_KM
    * np.arcsin(np.sqrt(np.clip(haversine_a, 0.0, 1.0)))
)
if not np.isfinite(source_target_distance_km).all():
    raise ValueError("Source-target distance calculation produced non-finite values.")
centroid_distance_diagnostics = {
    "maximum_centroid_distance_km": float(source_target_distance_km.max()),
    "pairs_with_centroid_distance_over_30_km": int(
        (source_target_distance_km > 30.0).sum()
    ),
    "interpretation": (
        "The canonical lookup uses cell support; boundary-intersecting H3 pairs can "
        "have centroid distances slightly above the configured 30 km support."
    ),
}

dynamic_pair_context = land_pairs.copy()
dynamic_pair_context["SOURCE_TARGET_DISTANCE_KM"] = (
    source_target_distance_km.astype("float32")
)
dynamic_pair_context = dynamic_pair_context.merge(
    dynamic_source_modulators[
        [
            "source_h3",
            "DATE",
            "VISIBILITY_KM_MEAN",
            "VISIBILITY_KM_MIN",
            "DAYLIGHT_SUPPORT",
            "CALENDAR_ACTIVITY_SUPPORT",
            "CONDITIONS_SUPPORT",
            "LAND_SOURCE_COMPOSITE",
            "CORE_DYNAMIC_CONTEXT_AVAILABLE",
            "EXTENDED_DYNAMIC_CONTEXT_AVAILABLE",
            "CORE_REPORTING_CONTEXT_AVAILABLE",
            "EXTENDED_REPORTING_CONTEXT_AVAILABLE",
        ]
    ],
    on="source_h3",
    how="left",
    validate="many_to_one",
)


def pair_visibility_support(
    visibility_km: pd.Series,
    distance_km: pd.Series,
    available: pd.Series,
) -> pd.Series:
    visibility_values = pd.to_numeric(visibility_km, errors="coerce").to_numpy(
        dtype="float64"
    )
    distance_values = pd.to_numeric(distance_km, errors="coerce").to_numpy(
        dtype="float64"
    )
    valid = (
        available.eq(True).to_numpy()
        & np.isfinite(visibility_values)
        & np.isfinite(distance_values)
    )
    output = np.full(len(visibility_values), np.nan, dtype="float64")
    nonnegative_visibility = np.maximum(visibility_values[valid], 0.0)
    transition_km = np.maximum(
        VISIBILITY_MINIMUM_TRANSITION_KM,
        VISIBILITY_TRANSITION_FRACTION * nonnegative_visibility,
    )
    output[valid] = stable_sigmoid(
        (nonnegative_visibility - distance_values[valid]) / transition_km
    )
    output[valid & (visibility_values <= 0.0)] = 0.0
    return pd.Series(output, index=visibility_km.index)


dynamic_pair_context["VISIBILITY_MEAN_SUPPORT"] = pair_visibility_support(
    dynamic_pair_context["VISIBILITY_KM_MEAN"],
    dynamic_pair_context["SOURCE_TARGET_DISTANCE_KM"],
    dynamic_pair_context["CORE_DYNAMIC_CONTEXT_AVAILABLE"],
)
dynamic_pair_context["VISIBILITY_MIN_SUPPORT"] = pair_visibility_support(
    dynamic_pair_context["VISIBILITY_KM_MIN"],
    dynamic_pair_context["SOURCE_TARGET_DISTANCE_KM"],
    dynamic_pair_context["CORE_DYNAMIC_CONTEXT_AVAILABLE"],
)
for column in ["VISIBILITY_MEAN_SUPPORT", "VISIBILITY_MIN_SUPPORT"]:
    values = dynamic_pair_context[column].dropna()
    if not values.between(0.0, 1.0).all():
        raise ValueError(f"{column} must be bounded to [0, 1].")

dynamic_pair_context["CORE_PHYSICAL_STATIC_SUPPORT"] = dynamic_pair_context[
    "weight_static_viewability"
].where(dynamic_pair_context["CORE_DYNAMIC_CONTEXT_AVAILABLE"])
dynamic_pair_context["EXTENDED_PHYSICAL_STATIC_SUPPORT"] = dynamic_pair_context[
    "weight_static_viewability"
].where(dynamic_pair_context["EXTENDED_DYNAMIC_CONTEXT_AVAILABLE"])
dynamic_pair_context["CORE_REPORTING_STATIC_SUPPORT"] = dynamic_pair_context[
    "weight_static_viewability"
].where(dynamic_pair_context["CORE_REPORTING_CONTEXT_AVAILABLE"])
dynamic_pair_context["EXTENDED_REPORTING_STATIC_SUPPORT"] = dynamic_pair_context[
    "weight_static_viewability"
].where(dynamic_pair_context["EXTENDED_REPORTING_CONTEXT_AVAILABLE"])
dynamic_pair_context["CORE_PHYSICAL_VIEWABILITY_CONTRIBUTION"] = (
    dynamic_pair_context["weight_static_viewability"]
    * dynamic_pair_context["VISIBILITY_MEAN_SUPPORT"]
    * dynamic_pair_context["DAYLIGHT_SUPPORT"]
)
dynamic_pair_context["CORE_MIN_VISIBILITY_PHYSICAL_CONTRIBUTION"] = (
    dynamic_pair_context["weight_static_viewability"]
    * dynamic_pair_context["VISIBILITY_MIN_SUPPORT"]
    * dynamic_pair_context["DAYLIGHT_SUPPORT"]
)
dynamic_pair_context["EXTENDED_PHYSICAL_VIEWABILITY_CONTRIBUTION"] = (
    dynamic_pair_context["CORE_PHYSICAL_VIEWABILITY_CONTRIBUTION"]
    * dynamic_pair_context["CONDITIONS_SUPPORT"]
)
dynamic_pair_context["CORE_LAND_REPORTING_OPPORTUNITY_CONTRIBUTION"] = (
    dynamic_pair_context["CORE_PHYSICAL_VIEWABILITY_CONTRIBUTION"]
    * dynamic_pair_context["LAND_SOURCE_COMPOSITE"]
    * dynamic_pair_context["CALENDAR_ACTIVITY_SUPPORT"]
)
dynamic_pair_context["EXTENDED_LAND_REPORTING_OPPORTUNITY_CONTRIBUTION"] = (
    dynamic_pair_context["EXTENDED_PHYSICAL_VIEWABILITY_CONTRIBUTION"]
    * dynamic_pair_context["LAND_SOURCE_COMPOSITE"]
    * dynamic_pair_context["CALENDAR_ACTIVITY_SUPPORT"]
)

dynamic_target_viewability = (
    dynamic_pair_context.groupby("target_h3", as_index=False)
    .agg(
        VIEWSHED_SOURCE_COUNT=("source_h3", "nunique"),
        VIEWSHED_STATIC_SUPPORT_SUM=("weight_static_viewability", "sum"),
        CORE_PHYSICAL_STATIC_SUPPORT_SUM=("CORE_PHYSICAL_STATIC_SUPPORT", "sum"),
        EXTENDED_PHYSICAL_STATIC_SUPPORT_SUM=(
            "EXTENDED_PHYSICAL_STATIC_SUPPORT", "sum"
        ),
        CORE_REPORTING_STATIC_SUPPORT_SUM=("CORE_REPORTING_STATIC_SUPPORT", "sum"),
        EXTENDED_REPORTING_STATIC_SUPPORT_SUM=(
            "EXTENDED_REPORTING_STATIC_SUPPORT", "sum"
        ),
        CORE_PHYSICAL_VIEWABILITY_RAW=(
            "CORE_PHYSICAL_VIEWABILITY_CONTRIBUTION",
            lambda values: values.sum(min_count=1),
        ),
        CORE_MIN_VISIBILITY_PHYSICAL_RAW=(
            "CORE_MIN_VISIBILITY_PHYSICAL_CONTRIBUTION",
            lambda values: values.sum(min_count=1),
        ),
        EXTENDED_PHYSICAL_VIEWABILITY_RAW=(
            "EXTENDED_PHYSICAL_VIEWABILITY_CONTRIBUTION",
            lambda values: values.sum(min_count=1),
        ),
        CORE_LAND_REPORTING_OPPORTUNITY_RAW=(
            "CORE_LAND_REPORTING_OPPORTUNITY_CONTRIBUTION",
            lambda values: values.sum(min_count=1),
        ),
        EXTENDED_LAND_REPORTING_OPPORTUNITY_RAW=(
            "EXTENDED_LAND_REPORTING_OPPORTUNITY_CONTRIBUTION",
            lambda values: values.sum(min_count=1),
        ),
    )
    .sort_values("target_h3")
    .reset_index(drop=True)
)
dynamic_target_viewability.insert(1, "H3_RESOLUTION", 7)
dynamic_target_viewability.insert(2, "DATE", DYNAMIC_SAMPLE_DATE)
for numerator, output_column in [
    ("CORE_PHYSICAL_STATIC_SUPPORT_SUM", "CORE_PHYSICAL_CONTEXT_COVERAGE"),
    ("EXTENDED_PHYSICAL_STATIC_SUPPORT_SUM", "EXTENDED_PHYSICAL_CONTEXT_COVERAGE"),
    ("CORE_REPORTING_STATIC_SUPPORT_SUM", "CORE_REPORTING_CONTEXT_COVERAGE"),
    (
        "EXTENDED_REPORTING_STATIC_SUPPORT_SUM",
        "EXTENDED_REPORTING_CONTEXT_COVERAGE",
    ),
]:
    dynamic_target_viewability[output_column] = (
        dynamic_target_viewability[numerator]
        / dynamic_target_viewability["VIEWSHED_STATIC_SUPPORT_SUM"].replace(
            0.0, np.nan
        )
    )
for raw_column in [
    "CORE_PHYSICAL_VIEWABILITY_RAW",
    "CORE_MIN_VISIBILITY_PHYSICAL_RAW",
    "EXTENDED_PHYSICAL_VIEWABILITY_RAW",
    "CORE_LAND_REPORTING_OPPORTUNITY_RAW",
    "EXTENDED_LAND_REPORTING_OPPORTUNITY_RAW",
]:
    dynamic_target_viewability[f"{raw_column}_PERCENTILE"] = (
        dynamic_target_viewability[raw_column].rank(method="average", pct=True)
    )
dynamic_target_viewability["DYNAMIC_VIEWABILITY_STATUS"] = np.where(
    dynamic_target_viewability["CORE_PHYSICAL_VIEWABILITY_RAW"].notna(),
    "derived_supported_context",
    "unavailable_dynamic_context",
)
dynamic_target_viewability["REPORTING_OPPORTUNITY_STATUS"] = np.where(
    dynamic_target_viewability["CORE_LAND_REPORTING_OPPORTUNITY_RAW"].notna(),
    "derived_partial_source_context",
    "unavailable_no_usable_land_source_context",
)
dynamic_target_viewability["MEASUREMENT_STATUS"] = "derived"

if not dynamic_target_viewability["target_h3"].is_unique:
    raise ValueError("Dynamic target viewability is not unique by target_h3.")
if len(dynamic_target_viewability) != len(target_pressure):
    raise ValueError("Dynamic viewability changed the canonical target universe.")
coverage_columns = [
    column
    for column in dynamic_target_viewability.columns
    if column.endswith("_CONTEXT_COVERAGE")
]
for column in coverage_columns:
    if not dynamic_target_viewability[column].dropna().between(0.0, 1.0).all():
        raise ValueError(f"{column} must be bounded to [0, 1].")
if not (
    dynamic_target_viewability["EXTENDED_PHYSICAL_VIEWABILITY_RAW"]
    <= dynamic_target_viewability["CORE_PHYSICAL_VIEWABILITY_RAW"] + 1e-12
).all():
    raise ValueError("Extended physical viewability cannot exceed the core value.")

dynamic_target_viewability_summary = pd.DataFrame(
    [
        {
            "sample_date": DYNAMIC_SAMPLE_DATE,
            "target_cells": len(dynamic_target_viewability),
            "core_physical_available": int(
                dynamic_target_viewability["CORE_PHYSICAL_VIEWABILITY_RAW"]
                .notna()
                .sum()
            ),
            "extended_physical_available": int(
                dynamic_target_viewability["EXTENDED_PHYSICAL_VIEWABILITY_RAW"]
                .notna()
                .sum()
            ),
            "core_reporting_available": int(
                dynamic_target_viewability[
                    "CORE_LAND_REPORTING_OPPORTUNITY_RAW"
                ]
                .notna()
                .sum()
            ),
            "median_core_physical_coverage": float(
                dynamic_target_viewability[
                    "CORE_PHYSICAL_CONTEXT_COVERAGE"
                ].median()
            ),
            "median_core_reporting_coverage": float(
                dynamic_target_viewability[
                    "CORE_REPORTING_CONTEXT_COVERAGE"
                ].median()
            ),
        }
    ]
)
display(dynamic_target_viewability_summary)
display(
    dynamic_target_viewability.sort_values(
        "CORE_LAND_REPORTING_OPPORTUNITY_RAW", ascending=False
    ).head(12)
)

## Daily H3 R6 land reporting-opportunity streams

This section publishes the model-facing **relative land-based effort proxy** before sightings are loaded. Native H3 R7 target results are averaged within each modeled H3 R6 target so edge cells do not receive less opportunity merely because fewer R7 water children are represented. The streams form an inspectable cascade rather than additive terms: static viewability; weather/daylight physical viewability; population-backed opportunity; a road/city transport challenger; and a mapped-public-shore challenger. The broad-coverage population stream is the primary proxy. Mapped access remains partial and never hard-gates the primary value.

Every stream retains raw support, a fixed-reference relative index, and coverage. A zero is emitted only when supported inputs imply zero opportunity; missing dynamic or source context remains null with an explicit status. This is reporting opportunity, not observer-hours, detection probability, whale absence, or human disturbance.

In [ ]:
for frame_name in ("pair_context", "dynamic_pair_context"):
    if frame_name in globals():
        del globals()[frame_name]

land_effort_source_columns = [
    "source_h3",
    "POPULATION_SOURCE_AVAILABLE_DECAYED_25_KM",
    "POPULATION_COMPONENT",
    "DECAYED_25_KM_POPULATION_ROAD_AND_CITY_AVAILABLE",
    "DECAYED_25_KM_POPULATION_ROAD_AND_CITY_COMPONENT",
    "LAND_SOURCE_COMPOSITE_AVAILABLE",
    "LAND_SOURCE_COMPOSITE",
]
missing_land_effort_columns = set(land_effort_source_columns) - set(
    land_source_context.columns
)
if missing_land_effort_columns:
    raise ValueError(
        "Land source context is missing effort-stream fields: "
        f"{sorted(missing_land_effort_columns)}"
    )
land_effort_source_context = land_source_context[
    land_effort_source_columns
].copy()

land_effort_pair_basis = land_pairs[
    ["source_h3", "target_h3", "weight_static_viewability"]
].copy()
land_effort_pair_basis["H3_INDEX"] = land_effort_pair_basis[
    "target_h3"
].map(lambda cell: h3.cell_to_parent(cell, LAND_EFFORT_TARGET_RESOLUTION))
target_r7_child_counts = (
    land_effort_pair_basis[["target_h3", "H3_INDEX"]]
    .drop_duplicates()
    .groupby("H3_INDEX")
    .size()
)
land_effort_pair_basis["TARGET_R7_CHILD_COUNT"] = (
    land_effort_pair_basis["H3_INDEX"].map(target_r7_child_counts)
)
land_effort_pair_basis["WEATHER_H3_R5"] = land_effort_pair_basis[
    "source_h3"
].map(lambda cell: h3.cell_to_parent(cell, 5))
land_effort_pair_basis["DISTANCE_BIN_INDEX"] = np.rint(
    source_target_distance_km / DAILY_DISTANCE_BIN_KM
).astype("int16")
land_effort_pair_basis = land_effort_pair_basis.merge(
    land_effort_source_context,
    on="source_h3",
    how="left",
    validate="many_to_one",
)
if land_effort_pair_basis["POPULATION_COMPONENT"].isna().any():
    raise ValueError("Primary population context must cover every land source.")
if land_effort_pair_basis[
    "DECAYED_25_KM_POPULATION_ROAD_AND_CITY_COMPONENT"
].isna().any():
    raise ValueError("Transport challenger context must cover every land source.")

land_effort_pair_basis["_BASE_STATIC_WEIGHT"] = (
    land_effort_pair_basis["weight_static_viewability"]
    / land_effort_pair_basis["TARGET_R7_CHILD_COUNT"]
)
land_effort_pair_basis["_STATIC_WEIGHT"] = land_effort_pair_basis[
    "_BASE_STATIC_WEIGHT"
]
land_effort_pair_basis["_POPULATION_WEIGHT"] = (
    land_effort_pair_basis["_BASE_STATIC_WEIGHT"]
    * land_effort_pair_basis["POPULATION_COMPONENT"]
)
land_effort_pair_basis["_TRANSPORT_WEIGHT"] = (
    land_effort_pair_basis["_BASE_STATIC_WEIGHT"]
    * land_effort_pair_basis[
        "DECAYED_25_KM_POPULATION_ROAD_AND_CITY_COMPONENT"
    ]
)
land_effort_pair_basis["_MAPPED_ACCESS_WEIGHT"] = (
    land_effort_pair_basis["_BASE_STATIC_WEIGHT"]
    * land_effort_pair_basis["LAND_SOURCE_COMPOSITE"]
)
land_effort_pair_basis["_POPULATION_CONTEXT_WEIGHT"] = (
    land_effort_pair_basis["_BASE_STATIC_WEIGHT"].where(
        land_effort_pair_basis[
            "POPULATION_SOURCE_AVAILABLE_DECAYED_25_KM"
        ].eq(True)
    )
)
land_effort_pair_basis["_TRANSPORT_CONTEXT_WEIGHT"] = (
    land_effort_pair_basis["_BASE_STATIC_WEIGHT"].where(
        land_effort_pair_basis[
            "DECAYED_25_KM_POPULATION_ROAD_AND_CITY_AVAILABLE"
        ].eq(True)
    )
)
land_effort_pair_basis["_MAPPED_ACCESS_CONTEXT_WEIGHT"] = (
    land_effort_pair_basis["_BASE_STATIC_WEIGHT"].where(
        land_effort_pair_basis["LAND_SOURCE_COMPOSITE_AVAILABLE"].eq(True)
    )
)

land_effort_grid_basis = (
    land_effort_pair_basis.groupby(
        ["H3_INDEX", "WEATHER_H3_R5", "DISTANCE_BIN_INDEX"],
        as_index=False,
    )
    .agg(
        STATIC_WEIGHT=("_STATIC_WEIGHT", "sum"),
        POPULATION_WEIGHT=("_POPULATION_WEIGHT", "sum"),
        TRANSPORT_WEIGHT=("_TRANSPORT_WEIGHT", "sum"),
        MAPPED_ACCESS_WEIGHT=(
            "_MAPPED_ACCESS_WEIGHT",
            lambda values: values.sum(min_count=1),
        ),
        STATIC_CONTEXT_WEIGHT=("_BASE_STATIC_WEIGHT", "sum"),
        POPULATION_CONTEXT_WEIGHT=(
            "_POPULATION_CONTEXT_WEIGHT",
            lambda values: values.sum(min_count=1),
        ),
        TRANSPORT_CONTEXT_WEIGHT=(
            "_TRANSPORT_CONTEXT_WEIGHT",
            lambda values: values.sum(min_count=1),
        ),
        MAPPED_ACCESS_CONTEXT_WEIGHT=(
            "_MAPPED_ACCESS_CONTEXT_WEIGHT",
            lambda values: values.sum(min_count=1),
        ),
        PAIR_COUNT=("source_h3", "size"),
    )
    .sort_values(["WEATHER_H3_R5", "H3_INDEX", "DISTANCE_BIN_INDEX"])
    .reset_index(drop=True)
)
if int(land_effort_grid_basis["PAIR_COUNT"].sum()) != len(land_pairs):
    raise ValueError("Land effort grid basis did not conserve source-target pairs.")
if land_effort_grid_basis.duplicated(
    ["H3_INDEX", "WEATHER_H3_R5", "DISTANCE_BIN_INDEX"]
).any():
    raise ValueError("Land effort grid basis keys are not unique.")

land_effort_target_order = pd.Index(
    sorted(land_effort_grid_basis["H3_INDEX"].unique()),
    name="H3_INDEX",
)
land_effort_target_code = {
    cell: index for index, cell in enumerate(land_effort_target_order)
}
land_effort_stream_order = (
    "STATIC",
    "POPULATION",
    "TRANSPORT",
    "MAPPED_ACCESS",
)
land_effort_weight_columns = {
    "STATIC": "STATIC_WEIGHT",
    "POPULATION": "POPULATION_WEIGHT",
    "TRANSPORT": "TRANSPORT_WEIGHT",
    "MAPPED_ACCESS": "MAPPED_ACCESS_WEIGHT",
}
land_effort_context_columns = {
    "STATIC": "STATIC_CONTEXT_WEIGHT",
    "POPULATION": "POPULATION_CONTEXT_WEIGHT",
    "TRANSPORT": "TRANSPORT_CONTEXT_WEIGHT",
    "MAPPED_ACCESS": "MAPPED_ACCESS_CONTEXT_WEIGHT",
}

land_effort_target_totals = (
    land_effort_grid_basis.groupby("H3_INDEX")[
        [*land_effort_weight_columns.values(), *land_effort_context_columns.values()]
    ]
    .sum(min_count=1)
    .reindex(land_effort_target_order)
)
land_effort_target_static = pd.DataFrame(
    {
        "H3_INDEX": land_effort_target_order,
        "H3_RESOLUTION": LAND_EFFORT_TARGET_RESOLUTION,
        "TARGET_R7_CHILD_COUNT": target_r7_child_counts.reindex(
            land_effort_target_order
        ).to_numpy(),
        "LAND_STATIC_VIEWABILITY_RAW": land_effort_target_totals[
            "STATIC_WEIGHT"
        ].to_numpy(),
        "POPULATION_STATIC_POTENTIAL_RAW": land_effort_target_totals[
            "POPULATION_WEIGHT"
        ].to_numpy(),
        "TRANSPORT_STATIC_POTENTIAL_RAW": land_effort_target_totals[
            "TRANSPORT_WEIGHT"
        ].to_numpy(),
        "MAPPED_ACCESS_STATIC_POTENTIAL_RAW": land_effort_target_totals[
            "MAPPED_ACCESS_WEIGHT"
        ].to_numpy(),
        "STATIC_CONTEXT_SUPPORT": land_effort_target_totals[
            "STATIC_CONTEXT_WEIGHT"
        ].to_numpy(),
        "POPULATION_CONTEXT_STATIC_SUPPORT": land_effort_target_totals[
            "POPULATION_CONTEXT_WEIGHT"
        ].to_numpy(),
        "TRANSPORT_CONTEXT_STATIC_SUPPORT": land_effort_target_totals[
            "TRANSPORT_CONTEXT_WEIGHT"
        ].to_numpy(),
        "MAPPED_ACCESS_CONTEXT_STATIC_SUPPORT": land_effort_target_totals[
            "MAPPED_ACCESS_CONTEXT_WEIGHT"
        ].to_numpy(),
    }
)
land_effort_target_static["MAPPED_ACCESS_STATIC_CONTEXT_FRACTION"] = (
    land_effort_target_static["MAPPED_ACCESS_CONTEXT_STATIC_SUPPORT"]
    / land_effort_target_static["STATIC_CONTEXT_SUPPORT"].replace(0.0, np.nan)
)
target_source_counts = land_effort_pair_basis.groupby("H3_INDEX")[
    "source_h3"
].nunique()
land_effort_target_static["LAND_SOURCE_COUNT"] = (
    land_effort_target_static["H3_INDEX"].map(target_source_counts).astype("int32")
)

land_effort_distance_bin_count = (
    int(land_effort_grid_basis["DISTANCE_BIN_INDEX"].max()) + 1
)
land_effort_distance_km = (
    np.arange(land_effort_distance_bin_count, dtype="float64")
    * DAILY_DISTANCE_BIN_KM
)
land_effort_weather_bases = []
target_count = len(land_effort_target_order)
for weather_cell, weather_basis in land_effort_grid_basis.groupby(
    "WEATHER_H3_R5", sort=False
):
    row_codes = weather_basis["DISTANCE_BIN_INDEX"].to_numpy()
    column_codes = weather_basis["H3_INDEX"].map(
        land_effort_target_code
    ).to_numpy()
    opportunity_matrices = []
    for stream in land_effort_stream_order:
        values = weather_basis[land_effort_weight_columns[stream]]
        valid = values.notna() & values.ne(0.0)
        matrix = sparse.csr_matrix(
            (
                values.loc[valid].to_numpy(dtype="float64"),
                (row_codes[valid], column_codes[valid]),
            ),
            shape=(land_effort_distance_bin_count, target_count),
        )
        matrix.eliminate_zeros()
        opportunity_matrices.append(matrix)
    opportunity_matrix_transpose = sparse.hstack(
        opportunity_matrices, format="csr"
    ).transpose().tocsr()
    context_totals = []
    for stream in land_effort_stream_order:
        context_total = np.zeros(target_count, dtype="float64")
        context_values = weather_basis[
            land_effort_context_columns[stream]
        ].fillna(0.0).to_numpy(dtype="float64")
        np.add.at(context_total, column_codes, context_values)
        context_totals.append(context_total)
    land_effort_weather_bases.append(
        {
            "weather_h3_r5": str(weather_cell),
            "daylight_h3_r4": h3.cell_to_parent(str(weather_cell), 4),
            "opportunity_matrix_transpose": opportunity_matrix_transpose,
            "context_totals": np.concatenate(context_totals),
        }
    )

land_effort_weather_daily = {
    str(cell): frame.set_index("DATE").sort_index()
    for cell, frame in surface_weather_daily.groupby("H3_INDEX")
}
land_effort_daylight_daily = {
    str(cell): frame.set_index("DATE").sort_index()
    for cell, frame in daylight_daily.groupby("H3_INDEX")
}
land_effort_calendar_daily = calendar_dynamic.set_index("DATE").sort_index()
land_effort_dates = pd.date_range(
    dynamic_context_start_date, dynamic_context_end_date, freq="D"
)
land_effort_reference_start_date = land_effort_dates.min()
land_effort_reference_end_date = min(
    land_effort_dates.max(),
    land_effort_reference_start_date
    + pd.Timedelta(weeks=LAND_EFFORT_REFERENCE_WEEKS)
    - pd.Timedelta(days=1),
)
land_effort_reference_dates = pd.date_range(
    land_effort_reference_start_date, land_effort_reference_end_date, freq="D"
)
land_effort_basis_summary = pd.DataFrame(
    [
        {
            "native_pairs": len(land_pairs),
            "grouped_basis_rows": len(land_effort_grid_basis),
            "target_h3_r6_cells": target_count,
            "dates": len(land_effort_dates),
            "expected_output_rows": len(land_effort_dates) * target_count,
            "weather_parent_cells": len(land_effort_weather_bases),
            "reference_start": land_effort_reference_start_date,
            "reference_end": land_effort_reference_end_date,
        }
    ]
)
display(land_effort_basis_summary)
display(
    land_effort_target_static[
        [
            "LAND_STATIC_VIEWABILITY_RAW",
            "MAPPED_ACCESS_STATIC_CONTEXT_FRACTION",
            "TARGET_R7_CHILD_COUNT",
            "LAND_SOURCE_COUNT",
        ]
    ].describe()
)

In [ ]:
def compute_land_effort_grid_chunk(
    date_index: pd.DatetimeIndex,
) -> dict[str, np.ndarray]:
    date_index = pd.DatetimeIndex(date_index).normalize()
    date_count = len(date_index)
    packed_width = len(land_effort_stream_order) * target_count
    physical_core_raw = np.zeros(
        (date_count, target_count), dtype="float64"
    )
    physical_core_context = np.zeros_like(physical_core_raw)
    extended_raw = np.zeros((date_count, packed_width), dtype="float64")
    extended_context = np.zeros_like(extended_raw)

    for weather_basis in land_effort_weather_bases:
        weather_frame = land_effort_weather_daily.get(
            weather_basis["weather_h3_r5"]
        )
        daylight_frame = land_effort_daylight_daily.get(
            weather_basis["daylight_h3_r4"]
        )
        if weather_frame is None or daylight_frame is None:
            continue
        weather_for_dates = weather_frame.reindex(date_index)
        daylight_for_dates = daylight_frame.reindex(date_index)
        visibility = weather_for_dates["VISIBILITY_KM_MEAN"].to_numpy(
            dtype="float64"
        )
        daylight_fraction = daylight_for_dates[
            "DAYLIGHT_FRACTION"
        ].to_numpy(dtype="float64")
        wind_speed = weather_for_dates[
            "WIND_SPEED_10M_MS_MEAN"
        ].to_numpy(dtype="float64")
        precipitation = weather_for_dates[
            "PRECIP_MM_DAY_ESTIMATE"
        ].to_numpy(dtype="float64")
        core_available = (
            np.isfinite(visibility)
            & np.isfinite(daylight_fraction)
            & weather_for_dates["SAMPLE_COVERAGE_FRAC"].eq(1.0).to_numpy()
            & weather_for_dates["QC_STATE"].eq("COMPLETE").to_numpy()
        )
        extended_available = (
            core_available
            & np.isfinite(wind_speed)
            & np.isfinite(precipitation)
        )

        nonnegative_visibility = np.maximum(visibility, 0.0)
        transition_km = np.maximum(
            VISIBILITY_MINIMUM_TRANSITION_KM,
            VISIBILITY_TRANSITION_FRACTION * nonnegative_visibility,
        )
        visibility_support = stable_sigmoid(
            (
                nonnegative_visibility[:, None]
                - land_effort_distance_km[None, :]
            )
            / transition_km[:, None]
        )
        visibility_support[visibility <= 0.0, :] = 0.0
        visibility_support[~core_available, :] = 0.0
        core_support = visibility_support * daylight_fraction[:, None]
        core_support[~core_available, :] = 0.0

        wind_support = stable_sigmoid(
            (WIND_SUPPORT_MIDPOINT_MS - wind_speed)
            / WIND_SUPPORT_SLOPE_MS
        )
        precipitation_support = 1.0 / (
            1.0
            + np.maximum(precipitation, 0.0)
            / PRECIP_SUPPORT_HALF_MM_DAY
        )
        conditions_support = (
            wind_support**CONDITIONS_WIND_EXPONENT
            * precipitation_support**CONDITIONS_PRECIP_EXPONENT
        )
        extended_support = core_support * conditions_support[:, None]
        extended_support[~extended_available, :] = 0.0

        opportunity_matrix_transpose = weather_basis[
            "opportunity_matrix_transpose"
        ]
        physical_core_raw += (
            opportunity_matrix_transpose[:target_count] @ core_support.T
        ).T
        extended_raw += (
            opportunity_matrix_transpose @ extended_support.T
        ).T
        context_totals = weather_basis["context_totals"]
        physical_core_context += (
            core_available[:, None] * context_totals[:target_count][None, :]
        )
        extended_context += (
            extended_available[:, None] * context_totals[None, :]
        )

    calendar_values = land_effort_calendar_daily.reindex(date_index)[
        "calendar_effort_weight"
    ].to_numpy(dtype="float64")
    calendar_available = np.isfinite(calendar_values)
    raw_blocks = {
        stream: extended_raw[
            :, index * target_count : (index + 1) * target_count
        ]
        for index, stream in enumerate(land_effort_stream_order)
    }
    context_blocks = {
        stream: extended_context[
            :, index * target_count : (index + 1) * target_count
        ]
        for index, stream in enumerate(land_effort_stream_order)
    }
    for stream in ("POPULATION", "TRANSPORT", "MAPPED_ACCESS"):
        raw_blocks[stream] *= calendar_values[:, None]
        raw_blocks[stream][~calendar_available, :] = np.nan
        context_blocks[stream][~calendar_available, :] = 0.0

    static_context_totals = {
        "STATIC": land_effort_target_static[
            "STATIC_CONTEXT_SUPPORT"
        ].to_numpy(dtype="float64"),
        "POPULATION": land_effort_target_static[
            "POPULATION_CONTEXT_STATIC_SUPPORT"
        ].to_numpy(dtype="float64"),
        "TRANSPORT": land_effort_target_static[
            "TRANSPORT_CONTEXT_STATIC_SUPPORT"
        ].to_numpy(dtype="float64"),
        "MAPPED_ACCESS": land_effort_target_static[
            "MAPPED_ACCESS_CONTEXT_STATIC_SUPPORT"
        ].to_numpy(dtype="float64"),
    }

    def coverage_ratio(
        available_context: np.ndarray, total_context: np.ndarray
    ) -> np.ndarray:
        return np.clip(
            np.divide(
                available_context,
                total_context[None, :],
                out=np.full_like(available_context, np.nan),
                where=total_context[None, :] > 0.0,
            ),
            0.0,
            1.0,
        )

    physical_core_coverage = coverage_ratio(
        physical_core_context, static_context_totals["STATIC"]
    )
    physical_extended_coverage = coverage_ratio(
        context_blocks["STATIC"], static_context_totals["STATIC"]
    )
    population_coverage = coverage_ratio(
        context_blocks["POPULATION"],
        static_context_totals["POPULATION"],
    )
    transport_coverage = coverage_ratio(
        context_blocks["TRANSPORT"],
        static_context_totals["TRANSPORT"],
    )
    mapped_access_coverage = coverage_ratio(
        context_blocks["MAPPED_ACCESS"],
        static_context_totals["MAPPED_ACCESS"],
    )

    zero_static_viewability = land_effort_target_static[
        "STATIC_CONTEXT_SUPPORT"
    ].to_numpy(dtype="float64") <= 0.0
    physical_core_raw[physical_core_context <= 0.0] = np.nan
    for stream in land_effort_stream_order:
        raw_blocks[stream][context_blocks[stream] <= 0.0] = np.nan
    physical_core_raw[:, zero_static_viewability] = 0.0
    for stream in land_effort_stream_order:
        raw_blocks[stream][:, zero_static_viewability] = 0.0
    if np.nanmax(raw_blocks["STATIC"] - physical_core_raw) > 1e-10:
        raise ValueError(
            "Extended physical opportunity exceeded core physical opportunity."
        )

    result = {
        "DATE": date_index.to_numpy(dtype="datetime64[ns]"),
        "CALENDAR_EFFORT_WEIGHT": calendar_values,
        "PHYSICAL_VIEWABILITY_CORE_RAW": physical_core_raw,
        "PHYSICAL_VIEWABILITY_RAW": raw_blocks["STATIC"],
        "POPULATION_OPPORTUNITY_RAW": raw_blocks["POPULATION"],
        "TRANSPORT_OPPORTUNITY_RAW": raw_blocks["TRANSPORT"],
        "MAPPED_ACCESS_OPPORTUNITY_RAW": raw_blocks["MAPPED_ACCESS"],
        "PHYSICAL_CORE_CONTEXT_COVERAGE": physical_core_coverage,
        "PHYSICAL_CONTEXT_COVERAGE": physical_extended_coverage,
        "POPULATION_CONTEXT_COVERAGE": population_coverage,
        "TRANSPORT_CONTEXT_COVERAGE": transport_coverage,
        "MAPPED_ACCESS_DYNAMIC_CONTEXT_COVERAGE": mapped_access_coverage,
    }
    for column, values in result.items():
        if column.endswith("_COVERAGE"):
            finite = values[np.isfinite(values)]
            if finite.size and ((finite < 0.0).any() or (finite > 1.0).any()):
                raise ValueError(f"{column} is outside [0, 1].")
    return result


land_effort_sample_chunk = compute_land_effort_grid_chunk(
    pd.DatetimeIndex([DYNAMIC_SAMPLE_DATE])
)
exact_sample_r6 = (
    dynamic_target_viewability.assign(
        H3_INDEX=dynamic_target_viewability["target_h3"].map(
            lambda cell: h3.cell_to_parent(
                cell, LAND_EFFORT_TARGET_RESOLUTION
            )
        )
    )
    .groupby("H3_INDEX")[[
        "EXTENDED_PHYSICAL_VIEWABILITY_RAW",
        "EXTENDED_LAND_REPORTING_OPPORTUNITY_RAW",
    ]]
    .mean()
    .reindex(land_effort_target_order)
)
land_effort_approximation_rows = []
for stream, exact_column, approximate_column in [
    (
        "PHYSICAL_VIEWABILITY",
        "EXTENDED_PHYSICAL_VIEWABILITY_RAW",
        "PHYSICAL_VIEWABILITY_RAW",
    ),
    (
        "MAPPED_ACCESS_OPPORTUNITY",
        "EXTENDED_LAND_REPORTING_OPPORTUNITY_RAW",
        "MAPPED_ACCESS_OPPORTUNITY_RAW",
    ),
]:
    exact_values = exact_sample_r6[exact_column].to_numpy(dtype="float64")
    approximate_values = land_effort_sample_chunk[approximate_column][0]
    valid = np.isfinite(exact_values) & np.isfinite(approximate_values)
    normalized_absolute_error = float(
        np.abs(approximate_values[valid] - exact_values[valid]).sum()
        / np.abs(exact_values[valid]).sum()
    )
    aggregate_relative_error = float(
        abs(approximate_values[valid].sum() - exact_values[valid].sum())
        / abs(exact_values[valid].sum())
    )
    land_effort_approximation_rows.append(
        {
            "stream": stream,
            "target_cells_compared": int(valid.sum()),
            "normalized_absolute_error": normalized_absolute_error,
            "aggregate_relative_error": aggregate_relative_error,
        }
    )
land_effort_approximation_validation = pd.DataFrame(
    land_effort_approximation_rows
)
if land_effort_approximation_validation[
    "normalized_absolute_error"
].max() > 0.01:
    raise ValueError(
        "H3 R6 distance-bin approximation exceeded 1% normalized error."
    )
display(land_effort_approximation_validation)

In [ ]:
def iter_date_chunks(
    date_index: pd.DatetimeIndex, chunk_days: int
):
    for start in range(0, len(date_index), chunk_days):
        yield date_index[start : start + chunk_days]


land_effort_scale_sources = {
    "LAND_STATIC_VIEWABILITY": "LAND_STATIC_VIEWABILITY_RAW",
    "PHYSICAL_VIEWABILITY": "PHYSICAL_VIEWABILITY_RAW",
    "POPULATION_OPPORTUNITY": "POPULATION_OPPORTUNITY_RAW",
    "TRANSPORT_OPPORTUNITY": "TRANSPORT_OPPORTUNITY_RAW",
    "MAPPED_ACCESS_OPPORTUNITY": "MAPPED_ACCESS_OPPORTUNITY_RAW",
}
reference_stream_values = {
    stream: []
    for stream in land_effort_scale_sources
    if stream != "LAND_STATIC_VIEWABILITY"
}
for reference_chunk_dates in iter_date_chunks(
    land_effort_reference_dates, LAND_EFFORT_DATE_CHUNK_DAYS
):
    reference_chunk = compute_land_effort_grid_chunk(reference_chunk_dates)
    for stream, raw_column in land_effort_scale_sources.items():
        if stream == "LAND_STATIC_VIEWABILITY":
            continue
        reference_stream_values[stream].append(
            reference_chunk[raw_column].ravel()
        )

land_effort_scale_caps = {}
for stream, raw_column in land_effort_scale_sources.items():
    if stream == "LAND_STATIC_VIEWABILITY":
        raw_values = land_effort_target_static[raw_column].to_numpy(
            dtype="float64"
        )
    else:
        raw_values = np.concatenate(reference_stream_values[stream])
    valid = raw_values[np.isfinite(raw_values) & (raw_values >= 0.0)]
    log_cap = float(
        np.quantile(np.log1p(valid), LAND_EFFORT_SCALE_QUANTILE)
    )
    if not np.isfinite(log_cap) or log_cap <= 0.0:
        raise ValueError(f"{stream} did not produce a positive scale cap.")
    land_effort_scale_caps[stream] = log_cap


def scale_land_effort_stream(values: np.ndarray, stream: str) -> np.ndarray:
    scaled = np.log1p(np.clip(values, 0.0, None)) / land_effort_scale_caps[
        stream
    ]
    scaled[~np.isfinite(values)] = np.nan
    return np.clip(scaled, 0.0, 1.0)


land_effort_static_index = scale_land_effort_stream(
    land_effort_target_static["LAND_STATIC_VIEWABILITY_RAW"].to_numpy(
        dtype="float64"
    ),
    "LAND_STATIC_VIEWABILITY",
)
land_effort_static_arrays = {
    column: land_effort_target_static[column].to_numpy()
    for column in [
        "TARGET_R7_CHILD_COUNT",
        "LAND_SOURCE_COUNT",
        "LAND_STATIC_VIEWABILITY_RAW",
        "POPULATION_STATIC_POTENTIAL_RAW",
        "TRANSPORT_STATIC_POTENTIAL_RAW",
        "MAPPED_ACCESS_STATIC_POTENTIAL_RAW",
        "MAPPED_ACCESS_STATIC_CONTEXT_FRACTION",
    ]
}

DYNAMIC_VIEWABILITY_DIR.mkdir(parents=True, exist_ok=True)
land_effort_temp_path = LAND_EFFORT_GRID_PATH.with_name(
    f"{LAND_EFFORT_GRID_PATH.stem}.tmp{LAND_EFFORT_GRID_PATH.suffix}"
)
land_effort_temp_path.unlink(missing_ok=True)
land_effort_writer = None
land_effort_rows_written = 0
land_effort_status_counts: dict[str, int] = {}
mapped_access_status_counts: dict[str, int] = {}
coverage_tracking = {
    column: {"minimum": np.inf, "maximum": -np.inf}
    for column in [
        "PHYSICAL_CONTEXT_COVERAGE",
        "POPULATION_CONTEXT_COVERAGE",
        "TRANSPORT_CONTEXT_COVERAGE",
        "MAPPED_ACCESS_DYNAMIC_CONTEXT_COVERAGE",
    ]
}

try:
    for output_chunk_dates in iter_date_chunks(
        land_effort_dates, LAND_EFFORT_DATE_CHUNK_DAYS
    ):
        chunk = compute_land_effort_grid_chunk(output_chunk_dates)
        chunk_date_count = len(output_chunk_dates)
        population_index = scale_land_effort_stream(
            chunk["POPULATION_OPPORTUNITY_RAW"],
            "POPULATION_OPPORTUNITY",
        )
        stream_indices = {
            "PHYSICAL_VIEWABILITY_STREAM_INDEX": scale_land_effort_stream(
                chunk["PHYSICAL_VIEWABILITY_RAW"],
                "PHYSICAL_VIEWABILITY",
            ),
            "POPULATION_OPPORTUNITY_STREAM_INDEX": population_index,
            "TRANSPORT_OPPORTUNITY_STREAM_INDEX": scale_land_effort_stream(
                chunk["TRANSPORT_OPPORTUNITY_RAW"],
                "TRANSPORT_OPPORTUNITY",
            ),
            "MAPPED_ACCESS_OPPORTUNITY_STREAM_INDEX": scale_land_effort_stream(
                chunk["MAPPED_ACCESS_OPPORTUNITY_RAW"],
                "MAPPED_ACCESS_OPPORTUNITY",
            ),
        }
        for column, values in stream_indices.items():
            finite = values[np.isfinite(values)]
            if finite.size and ((finite < 0.0).any() or (finite > 1.0).any()):
                raise ValueError(f"{column} is outside [0, 1].")

        primary_coverage = chunk["POPULATION_CONTEXT_COVERAGE"]
        primary_raw = chunk["POPULATION_OPPORTUNITY_RAW"]
        zero_static_viewability = np.tile(
            land_effort_static_arrays["LAND_STATIC_VIEWABILITY_RAW"] <= 0.0,
            (chunk_date_count, 1),
        )
        primary_status = np.select(
            [
                zero_static_viewability,
                ~np.isfinite(primary_raw) | ~np.isfinite(primary_coverage),
                primary_coverage >= 0.95,
                primary_coverage > 0.0,
            ],
            [
                "derived_zero_static_viewability",
                "unavailable_required_context",
                "derived_supported_context",
                "derived_partial_dynamic_context",
            ],
            default="unavailable_required_context",
        )
        access_context_fraction = np.tile(
            land_effort_static_arrays[
                "MAPPED_ACCESS_STATIC_CONTEXT_FRACTION"
            ],
            (chunk_date_count, 1),
        )
        mapped_access_status = np.select(
            [
                zero_static_viewability,
                np.isfinite(chunk["MAPPED_ACCESS_OPPORTUNITY_RAW"])
                & (access_context_fraction > 0.0),
            ],
            [
                "derived_zero_static_viewability",
                "derived_partial_mapped_context",
            ],
            default="unavailable_no_mapped_context",
        )

        output_frame = pd.DataFrame(
            {
                "DATE": np.repeat(chunk["DATE"], target_count),
                "H3_INDEX": np.tile(
                    land_effort_target_order.to_numpy(), chunk_date_count
                ),
                "H3_RESOLUTION": np.int8(LAND_EFFORT_TARGET_RESOLUTION),
                "TARGET_R7_CHILD_COUNT": np.tile(
                    land_effort_static_arrays["TARGET_R7_CHILD_COUNT"],
                    chunk_date_count,
                ).astype("int8"),
                "LAND_SOURCE_COUNT": np.tile(
                    land_effort_static_arrays["LAND_SOURCE_COUNT"],
                    chunk_date_count,
                ).astype("int32"),
                "LAND_STATIC_VIEWABILITY_RAW": np.tile(
                    land_effort_static_arrays[
                        "LAND_STATIC_VIEWABILITY_RAW"
                    ],
                    chunk_date_count,
                ).astype("float32"),
                "LAND_STATIC_VIEWABILITY_STREAM_INDEX": np.tile(
                    land_effort_static_index, chunk_date_count
                ).astype("float32"),
                "POPULATION_STATIC_POTENTIAL_RAW": np.tile(
                    land_effort_static_arrays[
                        "POPULATION_STATIC_POTENTIAL_RAW"
                    ],
                    chunk_date_count,
                ).astype("float32"),
                "TRANSPORT_STATIC_POTENTIAL_RAW": np.tile(
                    land_effort_static_arrays[
                        "TRANSPORT_STATIC_POTENTIAL_RAW"
                    ],
                    chunk_date_count,
                ).astype("float32"),
                "MAPPED_ACCESS_STATIC_POTENTIAL_RAW": np.tile(
                    land_effort_static_arrays[
                        "MAPPED_ACCESS_STATIC_POTENTIAL_RAW"
                    ],
                    chunk_date_count,
                ).astype("float32"),
                "MAPPED_ACCESS_STATIC_CONTEXT_FRACTION": (
                    access_context_fraction.ravel().astype("float32")
                ),
                "CALENDAR_EFFORT_WEIGHT": np.repeat(
                    chunk["CALENDAR_EFFORT_WEIGHT"], target_count
                ).astype("float32"),
                "PHYSICAL_VIEWABILITY_CORE_RAW": chunk[
                    "PHYSICAL_VIEWABILITY_CORE_RAW"
                ].ravel().astype("float32"),
                "PHYSICAL_VIEWABILITY_RAW": chunk[
                    "PHYSICAL_VIEWABILITY_RAW"
                ].ravel().astype("float32"),
                "PHYSICAL_VIEWABILITY_STREAM_INDEX": stream_indices[
                    "PHYSICAL_VIEWABILITY_STREAM_INDEX"
                ].ravel().astype("float32"),
                "PHYSICAL_CORE_CONTEXT_COVERAGE": chunk[
                    "PHYSICAL_CORE_CONTEXT_COVERAGE"
                ].ravel().astype("float32"),
                "PHYSICAL_CONTEXT_COVERAGE": chunk[
                    "PHYSICAL_CONTEXT_COVERAGE"
                ].ravel().astype("float32"),
                "POPULATION_OPPORTUNITY_RAW": primary_raw.ravel().astype(
                    "float32"
                ),
                "POPULATION_OPPORTUNITY_STREAM_INDEX": population_index.ravel().astype(
                    "float32"
                ),
                "POPULATION_CONTEXT_COVERAGE": primary_coverage.ravel().astype(
                    "float32"
                ),
                "TRANSPORT_OPPORTUNITY_RAW": chunk[
                    "TRANSPORT_OPPORTUNITY_RAW"
                ].ravel().astype("float32"),
                "TRANSPORT_OPPORTUNITY_STREAM_INDEX": stream_indices[
                    "TRANSPORT_OPPORTUNITY_STREAM_INDEX"
                ].ravel().astype("float32"),
                "TRANSPORT_CONTEXT_COVERAGE": chunk[
                    "TRANSPORT_CONTEXT_COVERAGE"
                ].ravel().astype("float32"),
                "MAPPED_ACCESS_OPPORTUNITY_RAW": chunk[
                    "MAPPED_ACCESS_OPPORTUNITY_RAW"
                ].ravel().astype("float32"),
                "MAPPED_ACCESS_OPPORTUNITY_STREAM_INDEX": stream_indices[
                    "MAPPED_ACCESS_OPPORTUNITY_STREAM_INDEX"
                ].ravel().astype("float32"),
                "MAPPED_ACCESS_DYNAMIC_CONTEXT_COVERAGE": chunk[
                    "MAPPED_ACCESS_DYNAMIC_CONTEXT_COVERAGE"
                ].ravel().astype("float32"),
                "LAND_BASED_EFFORT_PROXY_RAW": primary_raw.ravel().astype(
                    "float32"
                ),
                "LAND_BASED_EFFORT_PROXY_INDEX": population_index.ravel().astype(
                    "float32"
                ),
                "LAND_BASED_EFFORT_PROXY_STATUS": primary_status.ravel(),
                "MAPPED_ACCESS_STREAM_STATUS": mapped_access_status.ravel(),
                "MEASUREMENT_STATUS": "derived_relative_proxy",
            }
        )
        if output_frame.duplicated(["DATE", "H3_INDEX"]).any():
            raise ValueError("Land effort output chunk contains duplicate keys.")
        table = pa.Table.from_pandas(output_frame, preserve_index=False)
        if land_effort_writer is None:
            land_effort_writer = pq.ParquetWriter(
                land_effort_temp_path,
                table.schema,
                compression="zstd",
                use_dictionary=True,
            )
        land_effort_writer.write_table(table, row_group_size=len(output_frame))
        land_effort_rows_written += len(output_frame)

        for status, count in output_frame[
            "LAND_BASED_EFFORT_PROXY_STATUS"
        ].value_counts().items():
            land_effort_status_counts[str(status)] = (
                land_effort_status_counts.get(str(status), 0) + int(count)
            )
        for status, count in output_frame[
            "MAPPED_ACCESS_STREAM_STATUS"
        ].value_counts().items():
            mapped_access_status_counts[str(status)] = (
                mapped_access_status_counts.get(str(status), 0) + int(count)
            )
        for column in coverage_tracking:
            values = output_frame[column].to_numpy(dtype="float64")
            finite = values[np.isfinite(values)]
            if finite.size:
                coverage_tracking[column]["minimum"] = min(
                    coverage_tracking[column]["minimum"], float(finite.min())
                )
                coverage_tracking[column]["maximum"] = max(
                    coverage_tracking[column]["maximum"], float(finite.max())
                )
finally:
    if land_effort_writer is not None:
        land_effort_writer.close()

expected_land_effort_rows = len(land_effort_dates) * target_count
if land_effort_rows_written != expected_land_effort_rows:
    raise ValueError(
        f"Expected {expected_land_effort_rows} effort rows, "
        f"wrote {land_effort_rows_written}."
    )
land_effort_temp_path.replace(LAND_EFFORT_GRID_PATH)
land_effort_parquet = pq.ParquetFile(LAND_EFFORT_GRID_PATH)
if land_effort_parquet.metadata.num_rows != expected_land_effort_rows:
    raise ValueError("Published land effort Parquet row count is incorrect.")
required_land_effort_columns = {
    "DATE",
    "H3_INDEX",
    "H3_RESOLUTION",
    "LAND_STATIC_VIEWABILITY_STREAM_INDEX",
    "PHYSICAL_VIEWABILITY_STREAM_INDEX",
    "POPULATION_OPPORTUNITY_STREAM_INDEX",
    "TRANSPORT_OPPORTUNITY_STREAM_INDEX",
    "MAPPED_ACCESS_OPPORTUNITY_STREAM_INDEX",
    "CALENDAR_EFFORT_WEIGHT",
    "LAND_BASED_EFFORT_PROXY_RAW",
    "LAND_BASED_EFFORT_PROXY_INDEX",
    "LAND_BASED_EFFORT_PROXY_STATUS",
}
missing_output_columns = required_land_effort_columns - set(
    land_effort_parquet.schema.names
)
if missing_output_columns:
    raise ValueError(
        f"Published land effort output is missing {sorted(missing_output_columns)}"
    )

land_effort_artifact = {
    "path": str(LAND_EFFORT_GRID_PATH.relative_to(REPO_ROOT)),
    "bytes": LAND_EFFORT_GRID_PATH.stat().st_size,
    "sha256": sha256_file(LAND_EFFORT_GRID_PATH),
}
land_effort_metadata = {
    "schema_version": "0.1.0-prototype",
    "product": "human.land_based_effort_proxy_daily_h3_r6",
    "status": "prototype_uncalibrated_relative_reporting_opportunity",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "grain": ["DATE", "H3_INDEX", "H3_RESOLUTION"],
    "rows": expected_land_effort_rows,
    "columns": land_effort_parquet.schema.names,
    "date_coverage": {
        "start_date": land_effort_dates.min().date().isoformat(),
        "end_date": land_effort_dates.max().date().isoformat(),
        "dates": len(land_effort_dates),
    },
    "spatial_support": {
        "native_source_resolution": 7,
        "native_target_resolution": 7,
        "output_target_resolution": LAND_EFFORT_TARGET_RESOLUTION,
        "target_cells": target_count,
        "native_pairs": len(land_pairs),
        "grouped_basis_rows": len(land_effort_grid_basis),
        "target_aggregation": (
            "mean across modeled H3 R7 target children within each H3 R6 cell"
        ),
        "distance_bin_km": DAILY_DISTANCE_BIN_KM,
    },
    "streams": {
        "LAND_STATIC_VIEWABILITY": (
            "terrain, vegetation, and distance viewability; static over date"
        ),
        "PHYSICAL_VIEWABILITY": (
            "static viewability * distance-aware visibility * daylight * "
            "wind support * precipitation support"
        ),
        "POPULATION_OPPORTUNITY": (
            "physical viewability * decayed 25 km population component * "
            "calendar effort weight"
        ),
        "TRANSPORT_OPPORTUNITY": (
            "physical viewability * decayed 25 km population * road proximity * "
            "city travel access * calendar effort weight; challenger"
        ),
        "MAPPED_ACCESS_OPPORTUNITY": (
            "physical viewability * decayed 25 km population * mapped public-shore "
            "fraction * calendar effort weight; partial challenger"
        ),
        "CALENDAR": "explicit date-only calendar effort weight",
    },
    "primary_proxy": {
        "column": "LAND_BASED_EFFORT_PROXY_INDEX",
        "raw_column": "LAND_BASED_EFFORT_PROXY_RAW",
        "source_stream": "POPULATION_OPPORTUNITY",
        "interpretation": (
            "relative land reporting opportunity, not observer-hours or detection probability"
        ),
    },
    "scaling": {
        "formula": "clip(log1p(raw) / q99_reference_log1p_raw, 0, 1)",
        "quantile": LAND_EFFORT_SCALE_QUANTILE,
        "reference_start_date": (
            land_effort_reference_start_date.date().isoformat()
        ),
        "reference_end_date": land_effort_reference_end_date.date().isoformat(),
        "reference_weeks": LAND_EFFORT_REFERENCE_WEEKS,
        "log1p_caps": land_effort_scale_caps,
    },
    "coverage": {
        "dynamic_ranges": coverage_tracking,
        "primary_status_counts": land_effort_status_counts,
        "mapped_access_status_counts": mapped_access_status_counts,
        "mapped_access_static_context_fraction": {
            "target_cells_with_mapped_context": int(
                land_effort_target_static[
                    "MAPPED_ACCESS_STATIC_CONTEXT_FRACTION"
                ].notna().sum()
            ),
            "target_cells_total": target_count,
            "summary_scope": "targets with mapped public-shore context only",
            "minimum": float(
                land_effort_target_static[
                    "MAPPED_ACCESS_STATIC_CONTEXT_FRACTION"
                ].min()
            ),
            "median": float(
                land_effort_target_static[
                    "MAPPED_ACCESS_STATIC_CONTEXT_FRACTION"
                ].median()
            ),
            "maximum": float(
                land_effort_target_static[
                    "MAPPED_ACCESS_STATIC_CONTEXT_FRACTION"
                ].max()
            ),
        },
    },
    "approximation_validation": land_effort_approximation_validation.to_dict(
        orient="records"
    ),
    "input_inventory": input_inventory.to_dict(orient="records"),
    "sightings_used_to_construct_proxy": False,
    "measurement_contract": {
        "direct_observer_effort_available": False,
        "unavailable_context_is_zero": False,
        "disturbance_combined_with_reporting_opportunity": False,
    },
    "known_limitations": [
        "The index is relative reporting opportunity, not observed observer effort or detection probability.",
        "Static population and transport evidence are proxies for potential visitation, not observed daily visitation.",
        "Mapped public-shore context is partial in Washington and unavailable in British Columbia.",
        "Daily weather summaries do not identify conditions during actual viewing hours.",
        "Wind is a proxy for viewing conditions; Beaufort state and swell are not observed.",
        "The transport stream uses prototype public OSRM routing and remains a challenger.",
        "Reporting adoption and observer skill/equipment remain unmeasured.",
    ],
    "artifact": land_effort_artifact,
}
LAND_EFFORT_GRID_METADATA_PATH.write_text(
    json.dumps(land_effort_metadata, indent=2), encoding="utf-8"
)
land_effort_output_summary = pd.DataFrame(
    [
        {
            "path": land_effort_artifact["path"],
            "rows": expected_land_effort_rows,
            "dates": len(land_effort_dates),
            "target_cells": target_count,
            "bytes": land_effort_artifact["bytes"],
            "primary_available_rows": sum(
                count
                for status, count in land_effort_status_counts.items()
                if status != "unavailable_required_context"
            ),
        }
    ]
)
land_effort_output_sample = land_effort_parquet.read_row_group(0).slice(
    0, 12
).to_pandas()
display(pd.DataFrame(land_effort_scale_caps.items(), columns=["stream", "log1p_q99_cap"]))
display(land_effort_output_summary)
display(land_effort_output_sample)
print(f"Wrote {LAND_EFFORT_GRID_PATH.relative_to(REPO_ROOT)}")
print(f"Wrote {LAND_EFFORT_GRID_METADATA_PATH.relative_to(REPO_ROOT)}")

## Retrospective sightings comparison

Sightings are loaded only after target pressure is finalized. The comparison uses canonical observed, public-release-eligible records during the configured viewability window, with non-overlapping 2020–2022 and 2023–2026 H1 temporal holdouts for stability checks. Sighting counts and days are retrospective response summaries only—they are not inputs to the pressure formula. A zero means no reported record in this dataset/window, not confirmed whale absence.

In [ ]:
if not SIGHTINGS_PATH.is_file():
    raise FileNotFoundError(f"Missing observed sightings artifact: {SIGHTINGS_PATH}")
sightings_input = {
    "input": "observed_sightings_for_retrospective_comparison_only",
    "path": str(SIGHTINGS_PATH.relative_to(REPO_ROOT)),
    "bytes": SIGHTINGS_PATH.stat().st_size,
    "sha256": sha256_file(SIGHTINGS_PATH),
}
sightings = pd.read_parquet(
    SIGHTINGS_PATH,
    columns=[
        "OBSERVATION_ID",
        "SIGHTING_DATE",
        "LATITUDE",
        "LONGITUDE",
        "PUBLIC_RELEASE_ELIGIBLE",
    ],
)
if not sightings["OBSERVATION_ID"].is_unique:
    raise ValueError("Observed sightings are not unique by OBSERVATION_ID.")
sightings["SIGHTING_DATE"] = pd.to_datetime(sightings["SIGHTING_DATE"])
sightings_window = sightings.loc[
    sightings["PUBLIC_RELEASE_ELIGIBLE"]
    & sightings["SIGHTING_DATE"].between(SIGHTINGS_START_DATE, SIGHTINGS_END_DATE)
].copy()
sightings_window["target_h3"] = [
    h3.latlng_to_cell(latitude, longitude, 7)
    for latitude, longitude in zip(
        sightings_window["LATITUDE"], sightings_window["LONGITUDE"], strict=True
    )
]
target_universe = set(target_pressure["target_h3"])
sightings_in_target_universe = sightings_window.loc[
    sightings_window["target_h3"].isin(target_universe)
].copy()
sighting_counts = (
    sightings_in_target_universe.groupby("target_h3", as_index=False)
    .agg(
        REPORTED_SIGHTING_COUNT=("OBSERVATION_ID", "nunique"),
        REPORTED_SIGHTING_DAYS=("SIGHTING_DATE", "nunique"),
    )
)
target_comparison = target_pressure.merge(
    sighting_counts, on="target_h3", how="left", validate="one_to_one"
)
for column in ["REPORTED_SIGHTING_COUNT", "REPORTED_SIGHTING_DAYS"]:
    target_comparison[column] = target_comparison[column].fillna(0).astype("int64")
target_comparison["HAS_REPORTED_SIGHTING"] = target_comparison[
    "REPORTED_SIGHTING_COUNT"
].gt(0)
target_comparison["SIGHTINGS_COMPARISON_START_DATE"] = SIGHTINGS_START_DATE.date().isoformat()
target_comparison["SIGHTINGS_COMPARISON_END_DATE"] = SIGHTINGS_END_DATE.date().isoformat()

## Daily weather, daylight, viewshed, and population opportunity

This section builds one regional row per date over the shared sightings and dynamic-context window. Static viewshed support is retained alone, population-weighted, and population-plus-mapped-access weighted. Daily visibility remains distance-aware, daylight is mapped at the source cell, wind and precipitation form the extended conditions variant, and calendar remains an explicit activity factor. A 0.25 km distance-bin approximation makes the complete time series tractable; it is checked against the exact two-million-pair sample-date result. Daily sighting counts are joined only after all opportunity features are complete. Zero sightings means no public-release-eligible record in the canonical dataset on that date, not confirmed whale absence.

In [ ]:
daily_source_basis = land_source_context[
    [
        "source_h3",
        "POPULATION_COMPONENT",
        "LAND_SOURCE_COMPOSITE",
    ]
].copy()
daily_source_basis["WEATHER_H3_R5"] = daily_source_basis["source_h3"].map(
    lambda cell: h3.cell_to_parent(cell, 5)
)
daily_source_basis["DAYLIGHT_H3_R4"] = daily_source_basis["source_h3"].map(
    lambda cell: h3.cell_to_parent(cell, 4)
)
daily_pair_basis = land_pairs[
    ["source_h3", "weight_static_viewability"]
].merge(
    daily_source_basis,
    on="source_h3",
    how="left",
    validate="many_to_one",
)
daily_pair_basis["DISTANCE_BIN_INDEX"] = np.rint(
    source_target_distance_km / DAILY_DISTANCE_BIN_KM
).astype("int16")
daily_pair_basis["STATIC_PAIR_WEIGHT"] = daily_pair_basis[
    "weight_static_viewability"
].astype("float64")
daily_pair_basis["POPULATION_PAIR_WEIGHT"] = (
    daily_pair_basis["STATIC_PAIR_WEIGHT"]
    * daily_pair_basis["POPULATION_COMPONENT"]
)
daily_pair_basis["ACCESS_GATED_PAIR_WEIGHT"] = (
    daily_pair_basis["STATIC_PAIR_WEIGHT"]
    * daily_pair_basis["LAND_SOURCE_COMPOSITE"]
)
daily_opportunity_basis = (
    daily_pair_basis.groupby(
        ["WEATHER_H3_R5", "DAYLIGHT_H3_R4", "DISTANCE_BIN_INDEX"],
        as_index=False,
    )
    .agg(
        STATIC_WEIGHT=("STATIC_PAIR_WEIGHT", "sum"),
        POPULATION_WEIGHT=(
            "POPULATION_PAIR_WEIGHT", lambda values: values.sum(min_count=1)
        ),
        ACCESS_GATED_WEIGHT=(
            "ACCESS_GATED_PAIR_WEIGHT",
            lambda values: values.sum(min_count=1),
        ),
        PAIR_COUNT=("source_h3", "size"),
    )
    .sort_values(
        ["WEATHER_H3_R5", "DAYLIGHT_H3_R4", "DISTANCE_BIN_INDEX"]
    )
    .reset_index(drop=True)
)
if int(daily_opportunity_basis["PAIR_COUNT"].sum()) != len(land_pairs):
    raise ValueError("Daily opportunity basis did not conserve source-target pairs.")
if daily_opportunity_basis.duplicated(
    ["WEATHER_H3_R5", "DAYLIGHT_H3_R4", "DISTANCE_BIN_INDEX"]
).any():
    raise ValueError("Daily opportunity basis keys are not unique.")

daily_static_scope_totals = {
    "PHYSICAL": float(daily_opportunity_basis["STATIC_WEIGHT"].sum()),
    "POPULATION": float(
        daily_opportunity_basis["POPULATION_WEIGHT"].sum(min_count=1)
    ),
    "ACCESS": float(
        daily_opportunity_basis["ACCESS_GATED_WEIGHT"].sum(min_count=1)
    ),
}
daily_variant_scopes = {
    "CORE_PHYSICAL": "PHYSICAL",
    "MIN_VIS_PHYSICAL": "PHYSICAL",
    "EXTENDED_PHYSICAL": "PHYSICAL",
    "CORE_POPULATION": "POPULATION",
    "EXTENDED_POPULATION": "POPULATION",
    "CORE_ACCESS": "ACCESS",
    "EXTENDED_ACCESS": "ACCESS",
}
weather_daily_by_cell = {
    str(cell): frame.set_index("DATE").sort_index()
    for cell, frame in surface_weather_daily.groupby("H3_INDEX")
}
daylight_daily_by_cell = {
    str(cell): frame.set_index("DATE").sort_index()
    for cell, frame in daylight_daily.groupby("H3_INDEX")
}
calendar_daily_by_date = calendar_dynamic.set_index("DATE").sort_index()


def compute_grouped_daily_opportunity(date_index: pd.DatetimeIndex) -> pd.DataFrame:
    date_index = pd.DatetimeIndex(date_index).normalize()
    result = pd.DataFrame({"DATE": date_index})
    numerators = {
        name: np.zeros(len(date_index), dtype="float64")
        for name in daily_variant_scopes
    }
    denominators = {
        name: np.zeros(len(date_index), dtype="float64")
        for name in daily_variant_scopes
    }
    calendar_values = calendar_daily_by_date.reindex(date_index)[
        "calendar_effort_weight"
    ].to_numpy(dtype="float64")

    for (weather_cell, daylight_cell), basis in daily_opportunity_basis.groupby(
        ["WEATHER_H3_R5", "DAYLIGHT_H3_R4"], sort=False
    ):
        weather_frame = weather_daily_by_cell.get(str(weather_cell))
        daylight_frame = daylight_daily_by_cell.get(str(daylight_cell))
        if weather_frame is None or daylight_frame is None:
            continue
        weather_for_dates = weather_frame.reindex(date_index)
        daylight_for_dates = daylight_frame.reindex(date_index)
        visibility_mean = weather_for_dates["VISIBILITY_KM_MEAN"].to_numpy(
            dtype="float64"
        )
        visibility_min = weather_for_dates["VISIBILITY_KM_MIN"].to_numpy(
            dtype="float64"
        )
        wind_mean = weather_for_dates["WIND_SPEED_10M_MS_MEAN"].to_numpy(
            dtype="float64"
        )
        precipitation = weather_for_dates["PRECIP_MM_DAY_ESTIMATE"].to_numpy(
            dtype="float64"
        )
        daylight_fraction = daylight_for_dates["DAYLIGHT_FRACTION"].to_numpy(
            dtype="float64"
        )
        core_available = (
            np.isfinite(visibility_mean)
            & np.isfinite(daylight_fraction)
            & weather_for_dates["SAMPLE_COVERAGE_FRAC"].eq(1.0).to_numpy()
            & weather_for_dates["QC_STATE"].eq("COMPLETE").to_numpy()
        )
        minimum_visibility_available = core_available & np.isfinite(visibility_min)
        extended_available = (
            core_available
            & np.isfinite(wind_mean)
            & np.isfinite(precipitation)
        )
        distance_km = (
            basis["DISTANCE_BIN_INDEX"].to_numpy(dtype="float64")
            * DAILY_DISTANCE_BIN_KM
        )

        nonnegative_visibility = np.maximum(visibility_mean, 0.0)
        transition_km = np.maximum(
            VISIBILITY_MINIMUM_TRANSITION_KM,
            VISIBILITY_TRANSITION_FRACTION * nonnegative_visibility,
        )
        visibility_support = stable_sigmoid(
            (nonnegative_visibility[:, None] - distance_km[None, :])
            / transition_km[:, None]
        )
        visibility_support[visibility_mean <= 0.0, :] = 0.0
        visibility_support[~core_available, :] = np.nan

        nonnegative_visibility_min = np.maximum(visibility_min, 0.0)
        transition_min_km = np.maximum(
            VISIBILITY_MINIMUM_TRANSITION_KM,
            VISIBILITY_TRANSITION_FRACTION * nonnegative_visibility_min,
        )
        minimum_visibility_support = stable_sigmoid(
            (nonnegative_visibility_min[:, None] - distance_km[None, :])
            / transition_min_km[:, None]
        )
        minimum_visibility_support[visibility_min <= 0.0, :] = 0.0
        minimum_visibility_support[~minimum_visibility_available, :] = np.nan

        wind_support = stable_sigmoid(
            (WIND_SUPPORT_MIDPOINT_MS - wind_mean) / WIND_SUPPORT_SLOPE_MS
        )
        precip_support = 1.0 / (
            1.0
            + np.maximum(precipitation, 0.0) / PRECIP_SUPPORT_HALF_MM_DAY
        )
        conditions_support = (
            wind_support**CONDITIONS_WIND_EXPONENT
            * precip_support**CONDITIONS_PRECIP_EXPONENT
        )
        conditions_support[~extended_available] = np.nan

        for weight_column, scope in [
            ("STATIC_WEIGHT", "PHYSICAL"),
            ("POPULATION_WEIGHT", "POPULATION"),
            ("ACCESS_GATED_WEIGHT", "ACCESS"),
        ]:
            weights = basis[weight_column].fillna(0.0).to_numpy(dtype="float64")
            available_weight = float(weights.sum())
            if available_weight <= 0.0:
                continue
            core_values = (
                np.nansum(visibility_support * weights[None, :], axis=1)
                * daylight_fraction
            )
            core_values[~core_available] = np.nan
            if scope != "PHYSICAL":
                core_values = core_values * calendar_values
            extended_values = core_values * conditions_support
            core_name = f"CORE_{scope}"
            extended_name = f"EXTENDED_{scope}"
            core_valid = np.isfinite(core_values)
            extended_valid = np.isfinite(extended_values)
            numerators[core_name][core_valid] += core_values[core_valid]
            denominators[core_name][core_valid] += available_weight
            numerators[extended_name][extended_valid] += extended_values[
                extended_valid
            ]
            denominators[extended_name][extended_valid] += available_weight

            if scope == "PHYSICAL":
                minimum_values = (
                    np.nansum(
                        minimum_visibility_support * weights[None, :], axis=1
                    )
                    * daylight_fraction
                )
                minimum_values[~minimum_visibility_available] = np.nan
                minimum_valid = np.isfinite(minimum_values)
                numerators["MIN_VIS_PHYSICAL"][minimum_valid] += minimum_values[
                    minimum_valid
                ]
                denominators["MIN_VIS_PHYSICAL"][minimum_valid] += available_weight

    for name, scope in daily_variant_scopes.items():
        result[f"{name}_RAW"] = numerators[name]
        result[f"{name}_AVAILABLE_STATIC_SUPPORT"] = denominators[name]
        result[f"{name}_INDEX"] = np.divide(
            numerators[name],
            denominators[name],
            out=np.full(len(date_index), np.nan, dtype="float64"),
            where=denominators[name] > 0.0,
        )
        result[f"{name}_COVERAGE"] = np.clip(
            denominators[name] / daily_static_scope_totals[scope], 0.0, 1.0
        )
    result["STATIC_VIEWABILITY_TOTAL"] = daily_static_scope_totals["PHYSICAL"]
    result["STATIC_POPULATION_OPPORTUNITY_TOTAL"] = daily_static_scope_totals[
        "POPULATION"
    ]
    result["STATIC_ACCESS_GATED_OPPORTUNITY_TOTAL"] = daily_static_scope_totals[
        "ACCESS"
    ]
    result["CALENDAR_EFFORT_WEIGHT"] = calendar_values
    result["MEASUREMENT_STATUS"] = "derived"
    return result


daily_opportunity_approximation_sample = compute_grouped_daily_opportunity(
    pd.DatetimeIndex([DYNAMIC_SAMPLE_DATE])
)
exact_dynamic_sample_totals = {
    "CORE_PHYSICAL": float(
        dynamic_target_viewability["CORE_PHYSICAL_VIEWABILITY_RAW"].sum()
    ),
    "EXTENDED_PHYSICAL": float(
        dynamic_target_viewability["EXTENDED_PHYSICAL_VIEWABILITY_RAW"].sum()
    ),
    "CORE_ACCESS": float(
        dynamic_target_viewability["CORE_LAND_REPORTING_OPPORTUNITY_RAW"].sum()
    ),
    "EXTENDED_ACCESS": float(
        dynamic_target_viewability[
            "EXTENDED_LAND_REPORTING_OPPORTUNITY_RAW"
        ].sum()
    ),
}
daily_approximation_validation_rows = []
for variant, exact_value in exact_dynamic_sample_totals.items():
    approximate_value = float(
        daily_opportunity_approximation_sample[f"{variant}_RAW"].iloc[0]
    )
    relative_error = abs(approximate_value - exact_value) / exact_value
    daily_approximation_validation_rows.append(
        {
            "variant": variant,
            "approximate_value": approximate_value,
            "exact_value": exact_value,
            "relative_error": relative_error,
        }
    )
daily_approximation_validation = pd.DataFrame(
    daily_approximation_validation_rows
)
if daily_approximation_validation["relative_error"].max() > 0.001:
    raise ValueError("Daily distance-bin approximation exceeds 0.1% relative error.")

daily_opportunity_dates = pd.date_range(
    max(SIGHTINGS_START_DATE, dynamic_context_start_date),
    min(SIGHTINGS_END_DATE, dynamic_context_end_date),
    freq="D",
)
daily_opportunity = compute_grouped_daily_opportunity(daily_opportunity_dates)
daily_sighting_counts = (
    sightings_in_target_universe.assign(
        DATE=sightings_in_target_universe["SIGHTING_DATE"].dt.normalize()
    )
    .groupby("DATE", as_index=False)
    .agg(
        REPORTED_SIGHTING_COUNT=("OBSERVATION_ID", "nunique"),
        REPORTED_TARGET_CELL_COUNT=("target_h3", "nunique"),
    )
)
daily_opportunity = daily_opportunity.merge(
    daily_sighting_counts, on="DATE", how="left", validate="one_to_one"
)
for column in ["REPORTED_SIGHTING_COUNT", "REPORTED_TARGET_CELL_COUNT"]:
    daily_opportunity[column] = daily_opportunity[column].fillna(0).astype("int64")
daily_opportunity["HAS_REPORTED_SIGHTING"] = daily_opportunity[
    "REPORTED_SIGHTING_COUNT"
].gt(0)
daily_opportunity["SIGHTING_ZERO_SEMANTICS"] = (
    "no_public_release_eligible_record_not_confirmed_absence"
)
if not daily_opportunity["DATE"].is_unique:
    raise ValueError("Daily opportunity output is not unique by DATE.")
if daily_opportunity["DATE"].isna().any():
    raise ValueError("Daily opportunity output contains a null DATE.")
for column in [
    column for column in daily_opportunity.columns if column.endswith("_INDEX")
]:
    if not daily_opportunity[column].dropna().between(0.0, 1.0).all():
        raise ValueError(f"{column} must be bounded to [0, 1].")
for column in [
    column for column in daily_opportunity.columns if column.endswith("_COVERAGE")
]:
    if not daily_opportunity[column].dropna().between(0.0, 1.0).all():
        raise ValueError(f"{column} must be bounded to [0, 1].")

daily_opportunity_summary = pd.DataFrame(
    [
        {
            "start_date": daily_opportunity["DATE"].min(),
            "end_date": daily_opportunity["DATE"].max(),
            "dates": len(daily_opportunity),
            "reported_sightings": int(
                daily_opportunity["REPORTED_SIGHTING_COUNT"].sum()
            ),
            "dates_with_reported_sighting": int(
                daily_opportunity["HAS_REPORTED_SIGHTING"].sum()
            ),
            "dates_without_reported_sighting": int(
                (~daily_opportunity["HAS_REPORTED_SIGHTING"]).sum()
            ),
            "minimum_physical_context_coverage": float(
                daily_opportunity["CORE_PHYSICAL_COVERAGE"].min()
            ),
            "minimum_population_context_coverage": float(
                daily_opportunity["CORE_POPULATION_COVERAGE"].min()
            ),
            "minimum_access_context_coverage": float(
                daily_opportunity["CORE_ACCESS_COVERAGE"].min()
            ),
        }
    ]
)
display(daily_opportunity_summary)
display(daily_approximation_validation)

In [ ]:
daily_opportunity_feature_columns = [
    "CORE_PHYSICAL_INDEX",
    "MIN_VIS_PHYSICAL_INDEX",
    "EXTENDED_PHYSICAL_INDEX",
    "CORE_POPULATION_INDEX",
    "EXTENDED_POPULATION_INDEX",
    "CORE_ACCESS_INDEX",
    "EXTENDED_ACCESS_INDEX",
    "CALENDAR_EFFORT_WEIGHT",
]
daily_response_columns = [
    "REPORTED_SIGHTING_COUNT",
    "REPORTED_TARGET_CELL_COUNT",
    "HAS_REPORTED_SIGHTING",
]
daily_correlation_windows = {
    "FULL": (daily_opportunity_dates.min(), daily_opportunity_dates.max()),
    "2020_2022": (pd.Timestamp("2020-01-01"), pd.Timestamp("2022-12-31")),
    "2023_2026_H1": (pd.Timestamp("2023-01-01"), daily_opportunity_dates.max()),
}
daily_correlation_rows = []
for window_name, (window_start, window_end) in daily_correlation_windows.items():
    window = daily_opportunity.loc[
        daily_opportunity["DATE"].between(window_start, window_end)
    ].copy()
    year_month = window["DATE"].dt.to_period("M")
    anomaly_columns = daily_opportunity_feature_columns + [
        "REPORTED_SIGHTING_COUNT",
        "REPORTED_TARGET_CELL_COUNT",
    ]
    for column in anomaly_columns:
        window[f"{column}_MONTH_ANOMALY"] = (
            window[column] - window.groupby(year_month)[column].transform("mean")
        )
    for feature in daily_opportunity_feature_columns:
        for response in daily_response_columns:
            for method in ["spearman", "pearson"]:
                raw_correlation = window[[feature, response]].corr(method=method).iloc[
                    0, 1
                ]
                daily_correlation_rows.append(
                    {
                        "window_name": window_name,
                        "evaluation_scope": "raw_daily",
                        "feature": feature,
                        "response": response,
                        "method": method,
                        "dates": len(window),
                        "correlation": float(raw_correlation),
                    }
                )
                if response != "HAS_REPORTED_SIGHTING":
                    anomaly_feature = f"{feature}_MONTH_ANOMALY"
                    anomaly_response = f"{response}_MONTH_ANOMALY"
                    anomaly_correlation = window[
                        [anomaly_feature, anomaly_response]
                    ].corr(method=method).iloc[0, 1]
                    daily_correlation_rows.append(
                        {
                            "window_name": window_name,
                            "evaluation_scope": "within_month_anomaly",
                            "feature": feature,
                            "response": response,
                            "method": method,
                            "dates": len(window),
                            "correlation": float(anomaly_correlation),
                        }
                    )
daily_correlation_summary = pd.DataFrame(daily_correlation_rows)
daily_primary_correlations = daily_correlation_summary.loc[
    daily_correlation_summary["window_name"].eq("FULL")
    & daily_correlation_summary["response"].eq("REPORTED_SIGHTING_COUNT")
    & daily_correlation_summary["method"].eq("spearman")
].pivot(
    index="feature", columns="evaluation_scope", values="correlation"
).sort_values("raw_daily")
display(daily_primary_correlations)

DYNAMIC_VIEWABILITY_DIR.mkdir(parents=True, exist_ok=True)
daily_plot_frame = daily_opportunity.set_index("DATE").copy()
rolling_columns = [
    "REPORTED_SIGHTING_COUNT",
    "CORE_POPULATION_INDEX",
    "EXTENDED_POPULATION_INDEX",
    "EXTENDED_ACCESS_INDEX",
]
rolling = daily_plot_frame[rolling_columns].rolling(
    DAILY_ROLLING_WINDOW_DAYS, min_periods=7
).mean()
rolling_standardized = (rolling - rolling.mean()) / rolling.std(ddof=0)
daily_figure, daily_axes = plt.subplots(2, 1, figsize=(14, 10))
for column, color in [
    ("REPORTED_SIGHTING_COUNT", "#222222"),
    ("CORE_POPULATION_INDEX", "#457b9d"),
    ("EXTENDED_POPULATION_INDEX", "#e76f51"),
    ("EXTENDED_ACCESS_INDEX", "#2a9d8f"),
]:
    daily_axes[0].plot(
        rolling_standardized.index,
        rolling_standardized[column],
        label=column,
        color=color,
        linewidth=1.4,
    )
daily_axes[0].axhline(0.0, color="#777777", linewidth=0.7)
daily_axes[0].set_ylabel("Standardized 28-day rolling mean")
daily_axes[0].set_title("Daily land opportunity and reported sightings")
daily_axes[0].legend(loc="upper left", ncol=2)

bar_frame = daily_primary_correlations.reset_index()
bar_y = np.arange(len(bar_frame))
daily_axes[1].barh(
    bar_y - 0.18,
    bar_frame["raw_daily"],
    height=0.36,
    color="#457b9d",
    label="Raw daily",
)
daily_axes[1].barh(
    bar_y + 0.18,
    bar_frame["within_month_anomaly"],
    height=0.36,
    color="#e76f51",
    label="Within-month anomaly",
)
daily_axes[1].set_yticks(bar_y, bar_frame["feature"])
daily_axes[1].axvline(0.0, color="#333333", linewidth=0.8)
daily_axes[1].set_xlabel("Spearman correlation with daily reported sighting count")
daily_axes[1].set_title("Full-window daily correlations")
daily_axes[1].legend(loc="lower right")
daily_figure.tight_layout()
daily_figure.savefig(DAILY_CORRELATION_PLOT_PATH, dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
temporal_holdout_windows = {
    "2020_2022": (pd.Timestamp("2020-01-01"), pd.Timestamp("2022-12-31")),
    "2023_2026_H1": (pd.Timestamp("2023-01-01"), SIGHTINGS_END_DATE),
}
temporal_holdout_inventory_rows = []
for window_name, (window_start, window_end) in temporal_holdout_windows.items():
    window_sightings = sightings_in_target_universe.loc[
        sightings_in_target_universe["SIGHTING_DATE"].between(
            window_start, window_end
        )
    ]
    window_counts = (
        window_sightings.groupby("target_h3", as_index=False)
        .agg(
            REPORTED_SIGHTING_COUNT=("OBSERVATION_ID", "nunique"),
            REPORTED_SIGHTING_DAYS=("SIGHTING_DATE", "nunique"),
        )
        .rename(
            columns={
                "REPORTED_SIGHTING_COUNT": f"REPORTED_SIGHTING_COUNT_{window_name}",
                "REPORTED_SIGHTING_DAYS": f"REPORTED_SIGHTING_DAYS_{window_name}",
            }
        )
    )
    target_comparison = target_comparison.merge(
        window_counts, on="target_h3", how="left", validate="one_to_one"
    )
    count_column = f"REPORTED_SIGHTING_COUNT_{window_name}"
    days_column = f"REPORTED_SIGHTING_DAYS_{window_name}"
    observed_column = f"HAS_REPORTED_SIGHTING_{window_name}"
    target_comparison[count_column] = target_comparison[count_column].fillna(0).astype("int64")
    target_comparison[days_column] = target_comparison[days_column].fillna(0).astype("int64")
    target_comparison[observed_column] = target_comparison[count_column].gt(0)
    temporal_holdout_inventory_rows.append(
        {
            "window_name": window_name,
            "start_date": window_start.date().isoformat(),
            "end_date": window_end.date().isoformat(),
            "reported_sightings_in_target_universe": len(window_sightings),
            "target_cells_with_reported_sighting": int(
                target_comparison[observed_column].sum()
            ),
        }
    )

temporal_holdout_inventory = pd.DataFrame(temporal_holdout_inventory_rows)
if temporal_holdout_inventory["reported_sightings_in_target_universe"].sum() != len(
    sightings_in_target_universe
):
    raise ValueError("Temporal holdouts do not partition the comparison sightings.")
display(temporal_holdout_inventory)

In [ ]:
comparison_supported = target_comparison.loc[
    target_comparison["LAND_VIEWING_PRESSURE_RAW"].notna()
].copy()
comparison_supported["PRESSURE_DECILE"] = pd.Series(
    pd.NA, index=comparison_supported.index, dtype="Int64"
)
zero_pressure = comparison_supported["LAND_VIEWING_PRESSURE_RAW"].eq(0.0)
positive_pressure = comparison_supported["LAND_VIEWING_PRESSURE_RAW"].gt(0.0)
comparison_supported.loc[zero_pressure, "PRESSURE_DECILE"] = 0
comparison_supported.loc[positive_pressure, "PRESSURE_DECILE"] = (
    pd.qcut(
        comparison_supported.loc[positive_pressure, "LAND_VIEWING_PRESSURE_RAW"],
        q=10,
        labels=False,
        duplicates="raise",
    )
    .add(1)
    .astype("Int64")
)
target_comparison["PRESSURE_DECILE"] = pd.Series(pd.NA, index=target_comparison.index, dtype="Int64")
target_comparison.loc[comparison_supported.index, "PRESSURE_DECILE"] = comparison_supported[
    "PRESSURE_DECILE"
]

association_rows = []
for method in ["spearman", "pearson"]:
    correlations = comparison_supported[
        [
            "LAND_VIEWING_PRESSURE_RAW",
            "REPORTED_SIGHTING_COUNT",
            "REPORTED_SIGHTING_DAYS",
            "HAS_REPORTED_SIGHTING",
        ]
    ].corr(method=method)
    for response in [
        "REPORTED_SIGHTING_COUNT",
        "REPORTED_SIGHTING_DAYS",
        "HAS_REPORTED_SIGHTING",
    ]:
        association_rows.append(
            {
                "method": method,
                "response": response,
                "correlation": float(
                    correlations.loc["LAND_VIEWING_PRESSURE_RAW", response]
                ),
            }
        )
association_summary = pd.DataFrame(association_rows)
pressure_decile_summary = (
    comparison_supported.groupby("PRESSURE_DECILE", as_index=False, observed=True)
    .agg(
        TARGET_CELLS=("target_h3", "size"),
        MEAN_PRESSURE_PERCENTILE=("LAND_VIEWING_PRESSURE_PERCENTILE", "mean"),
        MEAN_CONTEXT_COVERAGE=("LAND_SOURCE_CONTEXT_COVERAGE", "mean"),
        REPORTED_SIGHTING_COUNT=("REPORTED_SIGHTING_COUNT", "sum"),
        MEAN_REPORTED_SIGHTING_DAYS=("REPORTED_SIGHTING_DAYS", "mean"),
        FRACTION_WITH_REPORTED_SIGHTING=("HAS_REPORTED_SIGHTING", "mean"),
    )
)

comparison_summary = pd.DataFrame(
    [
        {
            "observed_public_release_sightings_in_window": len(sightings_window),
            "sightings_in_target_universe": len(sightings_in_target_universe),
            "target_cells_with_pressure": len(comparison_supported),
            "target_cells_with_pressure_and_reported_sighting": int(comparison_supported["HAS_REPORTED_SIGHTING"].sum()),
        }
    ]
)
display(comparison_summary)
display(association_summary)
display(pressure_decile_summary)

In [ ]:
pressure_evaluation_specs = []
for variant in population_variant_specs:
    pressure_evaluation_specs.extend(
        [
            {
                "pressure_variant": variant,
                "source_scope": "population_only",
                "pressure_column": f"POPULATION_PRESSURE_RAW_{variant}",
            },
            {
                "pressure_variant": variant,
                "source_scope": "public_shore_access_gated",
                "pressure_column": f"ACCESS_GATED_PRESSURE_RAW_{variant}",
            },
        ]
    )
for variant, transport_spec in transport_pressure_specs.items():
    pressure_evaluation_specs.append(
        {
            "pressure_variant": variant,
            "source_scope": transport_spec["source_scope"],
            "pressure_column": f"TRANSPORT_PRESSURE_RAW_{variant}",
        }
    )
pressure_columns = [spec["pressure_column"] for spec in pressure_evaluation_specs]
shared_support = target_comparison[pressure_columns].notna().all(axis=1)
association_rows = []
for evaluation_scope in ["native_available", "shared_all_variants"]:
    for spec in pressure_evaluation_specs:
        if evaluation_scope == "native_available":
            evaluation_mask = target_comparison[spec["pressure_column"]].notna()
        else:
            evaluation_mask = shared_support
        evaluation = target_comparison.loc[evaluation_mask]
        for method in ["spearman", "pearson"]:
            correlations = evaluation[
                [
                    spec["pressure_column"],
                    "REPORTED_SIGHTING_COUNT",
                    "REPORTED_SIGHTING_DAYS",
                    "HAS_REPORTED_SIGHTING",
                ]
            ].corr(method=method)
            for response in [
                "REPORTED_SIGHTING_COUNT",
                "REPORTED_SIGHTING_DAYS",
                "HAS_REPORTED_SIGHTING",
            ]:
                association_rows.append(
                    {
                        "pressure_variant": spec["pressure_variant"],
                        "source_scope": spec["source_scope"],
                        "pressure_column": spec["pressure_column"],
                        "evaluation_scope": evaluation_scope,
                        "supported_target_cells": len(evaluation),
                        "method": method,
                        "response": response,
                        "correlation": float(
                            correlations.loc[spec["pressure_column"], response]
                        ),
                    }
                )
association_summary = pd.DataFrame(association_rows)
variant_spearman_summary = association_summary.loc[
    association_summary["evaluation_scope"].eq("shared_all_variants")
    & association_summary["method"].eq("spearman")
    & association_summary["response"].eq("REPORTED_SIGHTING_DAYS")
].sort_values("correlation")
display(variant_spearman_summary)

PROTOTYPE_DIR.mkdir(parents=True, exist_ok=True)
variant_figure, variant_axis = plt.subplots(
    figsize=(11, max(6, 0.42 * len(variant_spearman_summary)))
)
variant_labels = (
    variant_spearman_summary["source_scope"]
    + " | "
    + variant_spearman_summary["pressure_variant"]
)
scope_colors = {
    "population_only": "#457b9d",
    "public_shore_access_gated": "#2a9d8f",
    "transport_only": "#6d597a",
    "population_transport": "#e9c46a",
    "population_travel": "#f4a261",
}
variant_colors = [
    (
        "#e76f51"
        if scope == "public_shore_access_gated" and variant == PRIMARY_POPULATION_VARIANT
        else scope_colors.get(scope, "#777777")
    )
    for scope, variant in zip(
        variant_spearman_summary["source_scope"],
        variant_spearman_summary["pressure_variant"],
        strict=True,
    )
]
variant_axis.barh(variant_labels, variant_spearman_summary["correlation"], color=variant_colors)
variant_axis.axvline(0.0, color="#333333", linewidth=0.8)
variant_axis.set_xlabel("Spearman correlation with reported sighting days")
variant_axis.set_ylabel("Pressure construction")
variant_axis.set_title(
    f"Static-pressure sensitivity on shared target support (n={int(shared_support.sum()):,})"
)
variant_figure.tight_layout()
variant_figure.savefig(SIGHTINGS_VARIANTS_PLOT_PATH, dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
temporal_holdout_association_rows = []
for window_name, (window_start, window_end) in temporal_holdout_windows.items():
    response_columns = {
        "REPORTED_SIGHTING_COUNT": f"REPORTED_SIGHTING_COUNT_{window_name}",
        "REPORTED_SIGHTING_DAYS": f"REPORTED_SIGHTING_DAYS_{window_name}",
        "HAS_REPORTED_SIGHTING": f"HAS_REPORTED_SIGHTING_{window_name}",
    }
    for evaluation_scope in ["native_available", "shared_all_variants"]:
        for spec in pressure_evaluation_specs:
            if evaluation_scope == "native_available":
                evaluation_mask = target_comparison[spec["pressure_column"]].notna()
            else:
                evaluation_mask = shared_support
            evaluation = target_comparison.loc[evaluation_mask]
            for method in ["spearman", "pearson"]:
                correlation_columns = [
                    spec["pressure_column"], *response_columns.values()
                ]
                correlations = evaluation[correlation_columns].corr(method=method)
                for response, response_column in response_columns.items():
                    temporal_holdout_association_rows.append(
                        {
                            "window_name": window_name,
                            "window_start_date": window_start.date().isoformat(),
                            "window_end_date": window_end.date().isoformat(),
                            "pressure_variant": spec["pressure_variant"],
                            "source_scope": spec["source_scope"],
                            "pressure_column": spec["pressure_column"],
                            "evaluation_scope": evaluation_scope,
                            "supported_target_cells": len(evaluation),
                            "method": method,
                            "response": response,
                            "correlation": float(
                                correlations.loc[spec["pressure_column"], response_column]
                            ),
                        }
                    )
temporal_holdout_association_summary = pd.DataFrame(
    temporal_holdout_association_rows
)
temporal_holdout_spearman = temporal_holdout_association_summary.loc[
    temporal_holdout_association_summary["evaluation_scope"].eq("shared_all_variants")
    & temporal_holdout_association_summary["method"].eq("spearman")
    & temporal_holdout_association_summary["response"].eq("REPORTED_SIGHTING_DAYS")
]
display(
    temporal_holdout_spearman.pivot_table(
        index=["source_scope", "pressure_variant"],
        columns="window_name",
        values="correlation",
    ).sort_index()
)

In [ ]:
PROTOTYPE_DIR.mkdir(parents=True, exist_ok=True)
figure, axes = plt.subplots(1, 2, figsize=(14, 5.5))
hexbin = axes[0].hexbin(
    comparison_supported["LAND_VIEWING_PRESSURE_PERCENTILE"],
    np.log1p(comparison_supported["REPORTED_SIGHTING_DAYS"]),
    gridsize=35,
    mincnt=1,
    cmap="viridis",
)
axes[0].set_xlabel(f"Access-gated {PRIMARY_POPULATION_VARIANT} pressure percentile")
axes[0].set_ylabel("log1p(reported sighting days)")
axes[0].set_title("Target-cell sightings vs. primary pressure")
figure.colorbar(hexbin, ax=axes[0], label="Target cells per hex")

axes[1].bar(
    pressure_decile_summary["PRESSURE_DECILE"].astype(int),
    pressure_decile_summary["MEAN_REPORTED_SIGHTING_DAYS"],
    color="#2a9d8f",
    alpha=0.85,
)
axes[1].set_xlabel("Pressure group (0 = zero; 1–10 = positive-pressure deciles)")
axes[1].set_ylabel("Mean reported sighting days per target", color="#1d746b")
axes[1].set_title("Reported sightings by pressure group")
axes[1].set_xticks(range(0, 11))
fraction_axis = axes[1].twinx()
fraction_axis.plot(
    pressure_decile_summary["PRESSURE_DECILE"].astype(int),
    pressure_decile_summary["FRACTION_WITH_REPORTED_SIGHTING"],
    color="#e76f51",
    marker="o",
    linewidth=2,
)
fraction_axis.set_ylabel("Fraction with ≥1 reported sighting", color="#b94f37")
fraction_axis.set_ylim(0.0, 1.0)

figure.suptitle(
    f"Observed public-release sightings, {SIGHTINGS_START_DATE.date()} to {SIGHTINGS_END_DATE.date()}",
    y=1.02,
)
figure.tight_layout()
figure.savefig(SIGHTINGS_PLOT_PATH, dpi=180, bbox_inches="tight")
plt.show()

## Write prototype artifacts

These outputs are exploratory evidence products. They are intentionally outside `data/processed/` and must not be consumed as production model features.

In [ ]:
PROTOTYPE_DIR.mkdir(parents=True, exist_ok=True)
TRANSPORT_ACCESS_DIR.mkdir(parents=True, exist_ok=True)
POPULATION_TRAVEL_DIR.mkdir(parents=True, exist_ok=True)
DYNAMIC_VIEWABILITY_DIR.mkdir(parents=True, exist_ok=True)
land_source_context.to_parquet(PROTOTYPE_PATH, index=False)
source_transport_access.to_parquet(SOURCE_TRANSPORT_ACCESS_PATH, index=False)
transport_source_summary.to_csv(SOURCE_TRANSPORT_SUMMARY_PATH, index=False)
population_travel_origins.to_parquet(POPULATION_TRAVEL_ORIGINS_PATH, index=False)
source_population_travel.to_parquet(SOURCE_POPULATION_TRAVEL_PATH, index=False)
population_travel_source_summary.to_csv(SOURCE_POPULATION_TRAVEL_SUMMARY_PATH, index=False)
coverage_summary.to_csv(COVERAGE_PATH, index=False)
target_pressure.to_parquet(TARGET_PRESSURE_PATH, index=False)
dynamic_source_modulators.to_parquet(DYNAMIC_SOURCE_SAMPLE_PATH, index=False)
dynamic_target_viewability.to_parquet(DYNAMIC_TARGET_SAMPLE_PATH, index=False)
daily_opportunity.to_parquet(DAILY_OPPORTUNITY_PATH, index=False)
daily_correlation_summary.to_csv(DAILY_CORRELATIONS_PATH, index=False)
target_comparison.to_parquet(SIGHTINGS_COMPARISON_PATH, index=False)
pressure_decile_summary.to_csv(SIGHTINGS_DECILES_PATH, index=False)
population_catchment_summary.to_csv(POPULATION_CATCHMENT_SUMMARY_PATH, index=False)
association_summary.to_csv(PRESSURE_VARIANT_ASSOCIATIONS_PATH, index=False)
temporal_holdout_association_summary.to_csv(
    TEMPORAL_HOLDOUT_ASSOCIATIONS_PATH, index=False
)

population_variant_cap_records = population_cap_summary.to_dict(orient="records")
for record in population_variant_cap_records:
    if pd.isna(record["radius_km"]):
        record["radius_km"] = None

prototype_metadata = {
    "schema_version": "0.3.0-prototype",
    "product": "human.land_source_context_h3_r7_prototype",
    "status": "prototype_not_model_ready",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "grain": ["source_h3"],
    "h3_resolution": 7,
    "rows": len(land_source_context),
    "columns": list(land_source_context.columns),
    "inputs": input_inventory.to_dict(orient="records"),
    "coverage": coverage_summary.to_dict(orient="records"),
    "population_catchments": {
        "radii_km": list(POPULATION_CATCHMENT_RADII_KM),
        "decay_radii_km": list(POPULATION_DECAY_RADII_KM),
        "distance_basis": "H3 R7 centroid great-circle distance",
        "decay_kernel": "population * max(1 - distance_km / radius_km, 0)",
        "support_semantics": (
            "A supported zero is zero; no population-support cell within the radius is null."
        ),
    },
    "population_travel_demand": {
        "origin_resolution": POPULATION_TRAVEL_ORIGIN_RESOLUTION,
        "selected_population_fraction": population_travel_represented_fraction,
        "decay_minutes": list(POPULATION_TRAVEL_DECAY_MINUTES),
        "minimum_routed_selected_population_fraction": POPULATION_TRAVEL_MIN_ROUTED_SHARE,
        "source_artifact": str(SOURCE_POPULATION_TRAVEL_PATH.relative_to(REPO_ROOT)),
    },
    "known_limitations": [
        "This is source-side evidence, not direct observer effort or detection probability.",
        "Authoritative public-shore coverage is partial in Washington and unavailable in British Columbia.",
        "Unmatched context rows are null and must not be converted to zero.",
        "Calendar remains a standalone date table and joins only the date-specific dynamic sample.",
        "The source composite is an explicit prototype formula, not a calibrated effort model.",
        "Catchments use population-cell and source-cell centroids; boundary membership is approximate at H3 R7 scale.",
        "Washington uses the 2020 census and British Columbia uses the 2021 census; cross-border catchments mix vintages.",
        "Road proximity and city travel use H3 source centroids snapped to a routable driving graph; they do not establish legal or safe shore access.",
        "Population travel demand uses population-weighted H3 R4 origins covering at least 99 percent of represented population plus origins within 25 km of a land source.",
        "Weather and daylight modifiers are evaluated for one sample date only; full daily or weekly aggregation and detection-probability calibration are not applied.",
    ],
}
METADATA_PATH.write_text(
    json.dumps(prototype_metadata, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)

transport_metadata = {
    "schema_version": "0.1.0-prototype",
    "product": "human.land_source_transport_access_h3_r7_prototype",
    "status": "prototype_external_routing_snapshot",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "grain": ["source_h3"],
    "h3_resolution": 7,
    "rows": len(source_transport_access),
    "columns": list(source_transport_access.columns),
    "routing_service": {
        "endpoint": OSRM_TABLE_ENDPOINT,
        "profile": "driving",
        "provider": "OSRM public demo service using OpenStreetMap data",
        "attribution": "OpenStreetMap contributors; ODbL 1.0",
        "service_data_version": None,
        "cached_response_batches": routing_cache_inventory.to_dict(orient="records"),
    },
    "city_origins": transport_city_origins.to_dict(orient="records"),
    "road_distance_basis": "source_h3_centroid_to_osrm_snapped_driving_segment",
    "summary": transport_source_summary.to_dict(orient="records"),
    "known_limitations": [
        "This routing evidence does not establish legal public access, parking, trail access, safety, or viewpoint quality.",
        "H3 R7 centroid snapping is approximate for irregular coastal cells and islands.",
        "The public OSRM response does not report a reconstructable graph data version; cached checksum-pinned responses preserve this prototype run only.",
        "Driving travel may include or omit ferry links according to the service graph; ferry route semantics were not independently validated.",
        "Configured city origins are explicit approximate centers, not a population-complete origin universe.",
    ],
}
SOURCE_TRANSPORT_METADATA_PATH.write_text(
    json.dumps(transport_metadata, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)

population_travel_metadata = {
    "schema_version": "0.1.0-prototype",
    "product": "human.land_source_population_travel_demand_h3_r7_prototype",
    "status": "prototype_external_routing_snapshot",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "grain": ["source_h3"],
    "h3_resolution": 7,
    "rows": len(source_population_travel),
    "columns": list(source_population_travel.columns),
    "origin_artifact": {
        "path": str(POPULATION_TRAVEL_ORIGINS_PATH.relative_to(REPO_ROOT)),
        "sha256": sha256_file(POPULATION_TRAVEL_ORIGINS_PATH),
    },
    "source_artifact": {
        "path": str(SOURCE_POPULATION_TRAVEL_PATH.relative_to(REPO_ROOT)),
        "sha256": sha256_file(SOURCE_POPULATION_TRAVEL_PATH),
    },
    "origin_selection": {
        "origin_resolution": POPULATION_TRAVEL_ORIGIN_RESOLUTION,
        "retained_population_share_threshold": POPULATION_TRAVEL_RETAINED_SHARE,
        "local_origin_distance_km": POPULATION_TRAVEL_LOCAL_ORIGIN_DISTANCE_KM,
        "origin_universe_count": len(population_travel_origin_universe),
        "selected_origin_count": len(population_travel_origins),
        "total_supported_population": population_travel_total_population,
        "selected_origin_population": population_travel_selected_population,
        "selected_population_fraction": population_travel_represented_fraction,
        "omitted_population": population_travel_total_population - population_travel_selected_population,
    },
    "routing_service": {
        "endpoint": OSRM_TABLE_ENDPOINT,
        "profile": "driving",
        "provider": "OSRM public demo service using OpenStreetMap data",
        "attribution": "OpenStreetMap contributors; ODbL 1.0",
        "service_data_version": None,
        "minimum_routed_selected_population_fraction": POPULATION_TRAVEL_MIN_ROUTED_SHARE,
        "cached_response_batches": population_travel_routing_cache_inventory.to_dict(orient="records"),
    },
    "decay_minutes": list(POPULATION_TRAVEL_DECAY_MINUTES),
    "within_minutes": list(POPULATION_TRAVEL_WITHIN_MINUTES),
    "component_caps": population_travel_cap_summary.to_dict(orient="records"),
    "measurement_status_counts": {
        str(status): int(count)
        for status, count in source_population_travel[
            "POPULATION_TRAVEL_MEASUREMENT_STATUS"
        ].value_counts(dropna=False).items()
    },
    "summary": population_travel_source_summary.to_dict(orient="records"),
    "known_limitations": [
        "H3 R4 origins are regional population-weighted clusters, not individual household or neighborhood origins.",
        "The selected origins retain at least 99 percent of supported population plus nearby low-population origins; omitted population is reported, not represented as zero.",
        "Routes terminate at H3 R7 source centroids snapped to the driving graph, not mapped parking, trailheads, or legal viewing sites.",
        "The public OSRM response does not identify a reconstructable graph version; checksum-pinned cache responses preserve this prototype only.",
        "Driving travel may include or omit ferry links according to the service graph; ferry semantics were not independently validated.",
        "The 60, 120, and 240 minute decay scales and 99th-percentile component caps are uncalibrated sensitivity variants.",
    ],
}
SOURCE_POPULATION_TRAVEL_METADATA_PATH.write_text(
    json.dumps(population_travel_metadata, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)

pressure_input_names = {"land_viewshed", "population_context", "public_shore_access"}
target_pressure_metadata = {
    "schema_version": "0.3.0-prototype",
    "product": "human.target_land_viewing_pressure_h3_r7_prototype",
    "status": "prototype_uncalibrated_partial_context",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "grain": ["target_h3"],
    "h3_resolution": 7,
    "rows": len(target_pressure),
    "columns": list(target_pressure.columns),
    "primary_population_variant": PRIMARY_POPULATION_VARIANT,
    "pressure_formula": (
        "sum(weight_static_viewability * "
        "clip(log1p(POPULATION_DECAYED_25_KM) / q99_source_log1p, 0, 1) * "
        "ACCESSIBLE_WATERFRONT_FRACTION)"
    ),
    "population_cap_quantile": POPULATION_CAP_QUANTILE,
    "population_log1p_cap": float(population_cap),
    "population_variant_caps": population_variant_cap_records,
    "inputs": input_inventory.loc[
        input_inventory["input"].isin(pressure_input_names)
    ].to_dict(orient="records"),
    "sightings_used_to_construct_pressure": False,
    "target_summary": target_pressure_summary.iloc[0].to_dict(),
    "target_variant_summary": target_variant_summary.to_dict(orient="records"),
    "transport_pressure_variants": [
        {"variant": variant, **spec}
        for variant, spec in transport_pressure_specs.items()
    ],
    "transport_target_summary": transport_target_summary.to_dict(orient="records"),
    "known_limitations": [
        "Pressure is a partial-evidence reporting-opportunity index, not observer counts or detection probability.",
        "Public-shore evidence is partial in Washington and unavailable in British Columbia.",
        "Targets without usable source context retain null pressure.",
        "The 99th-percentile population cap and multiplicative formula require sensitivity analysis before promotion.",
        "Transport components are exploratory exponential decays with uncalibrated 5 km and 120 minute scales.",
        "Population-travel components use uncalibrated 60, 120, and 240 minute decay scales over selected regional origins.",
    ],
}
TARGET_PRESSURE_METADATA_PATH.write_text(
    json.dumps(target_pressure_metadata, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)

dynamic_input_names = {
    "land_viewshed",
    "population_context",
    "public_shore_access",
    "calendar_context",
    "surface_weather_manifest",
    "daylight_manifest",
}
dynamic_viewability_metadata = {
    "schema_version": "0.1.0-prototype",
    "product": "human.target_land_viewability_h3_r7_date_sample_prototype",
    "status": "prototype_uncalibrated_date_specific_partial_context",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "grain": ["target_h3", "DATE"],
    "h3_resolution": 7,
    "sample_date": DYNAMIC_SAMPLE_DATE.date().isoformat(),
    "shared_temporal_support": {
        "start_date": dynamic_context_start_date.date().isoformat(),
        "end_date": dynamic_context_end_date.date().isoformat(),
    },
    "rows": len(dynamic_target_viewability),
    "columns": list(dynamic_target_viewability.columns),
    "inputs": input_inventory.loc[
        input_inventory["input"].isin(dynamic_input_names)
    ].to_dict(orient="records"),
    "source_weather_join": {
        "weather_resolution": 5,
        "daylight_resolution": 4,
        "observer_source_resolution": 7,
        "weather_join": "source_h3 R7 parent at R5 plus DATE",
        "daylight_join": "source_h3 R7 parent at R4 plus DATE",
        "calendar_join": "DATE only",
    },
    "formulas": {
        "visibility_transition_km": (
            "max(1.0, 0.20 * visibility_km)"
        ),
        "visibility_support": (
            "sigmoid((visibility_km - source_target_centroid_distance_km) / "
            "visibility_transition_km); zero when visibility_km <= 0"
        ),
        "wind_support": "sigmoid((5.5 - wind_speed_10m_mean_ms) / 1.5)",
        "precip_support": "1 / (1 + precip_mm_day_estimate / 10)",
        "conditions_support": "wind_support ** 0.70 * precip_support ** 0.30",
        "core_physical_viewability": (
            "weight_static_viewability * visibility_mean_support * daylight_fraction"
        ),
        "minimum_visibility_sensitivity": (
            "weight_static_viewability * visibility_min_support * daylight_fraction"
        ),
        "extended_physical_viewability": (
            "core_physical_viewability * conditions_support"
        ),
        "core_land_reporting_opportunity": (
            "core_physical_viewability * land_source_composite * calendar_effort_weight"
        ),
        "extended_land_reporting_opportunity": (
            "extended_physical_viewability * land_source_composite * calendar_effort_weight"
        ),
    },
    "parameters": {
        "visibility_transition_fraction": VISIBILITY_TRANSITION_FRACTION,
        "visibility_minimum_transition_km": VISIBILITY_MINIMUM_TRANSITION_KM,
        "wind_support_midpoint_ms": WIND_SUPPORT_MIDPOINT_MS,
        "wind_support_slope_ms": WIND_SUPPORT_SLOPE_MS,
        "precip_support_half_mm_day": PRECIP_SUPPORT_HALF_MM_DAY,
        "conditions_wind_exponent": CONDITIONS_WIND_EXPONENT,
        "conditions_precip_exponent": CONDITIONS_PRECIP_EXPONENT,
    },
    "source_coverage": {
        "land_source_cells": len(dynamic_source_modulators),
        "core_dynamic_context_available": int(
            dynamic_source_modulators["CORE_DYNAMIC_CONTEXT_AVAILABLE"].sum()
        ),
        "extended_dynamic_context_available": int(
            dynamic_source_modulators["EXTENDED_DYNAMIC_CONTEXT_AVAILABLE"].sum()
        ),
        "core_reporting_context_available": int(
            dynamic_source_modulators["CORE_REPORTING_CONTEXT_AVAILABLE"].sum()
        ),
    },
    "target_coverage": {
        "target_cells": len(dynamic_target_viewability),
        "core_physical_available": int(
            dynamic_target_viewability["CORE_PHYSICAL_VIEWABILITY_RAW"].notna().sum()
        ),
        "extended_physical_available": int(
            dynamic_target_viewability[
                "EXTENDED_PHYSICAL_VIEWABILITY_RAW"
            ].notna().sum()
        ),
        "core_reporting_available": int(
            dynamic_target_viewability[
                "CORE_LAND_REPORTING_OPPORTUNITY_RAW"
            ].notna().sum()
        ),
        "median_core_physical_context_coverage": float(
            dynamic_target_viewability["CORE_PHYSICAL_CONTEXT_COVERAGE"].median()
        ),
        "median_core_reporting_context_coverage": float(
            dynamic_target_viewability["CORE_REPORTING_CONTEXT_COVERAGE"].median()
        ),
    },
    "centroid_distance_diagnostics": centroid_distance_diagnostics,
    "artifacts": [
        {
            "path": str(DYNAMIC_SOURCE_SAMPLE_PATH.relative_to(REPO_ROOT)),
            "sha256": sha256_file(DYNAMIC_SOURCE_SAMPLE_PATH),
        },
        {
            "path": str(DYNAMIC_TARGET_SAMPLE_PATH.relative_to(REPO_ROOT)),
            "sha256": sha256_file(DYNAMIC_TARGET_SAMPLE_PATH),
        },
    ],
    "sightings_used_to_construct_viewability": False,
    "known_limitations": [
        "This is a date-specific reporting-opportunity index, not direct observer effort or detection probability.",
        "Daily weather summaries do not identify conditions during actual viewing hours.",
        "HRRR source-cell wind is a proxy for sea state; Beaufort state and swell are not observed.",
        "Weather is sampled at the observer/source parent cell rather than continuously along each sightline.",
        "Public-shore evidence is partial in Washington and unavailable in British Columbia.",
        "This notebook materializes one sample date; a streaming daily-to-weekly builder is still required.",
    ],
}
DYNAMIC_VIEWABILITY_METADATA_PATH.write_text(
    json.dumps(dynamic_viewability_metadata, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)

daily_opportunity_metadata = {
    "schema_version": "0.1.0-prototype",
    "product": "analysis.daily_land_reporting_opportunity_vs_sightings",
    "status": "retrospective_daily_association_prototype",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "grain": ["DATE"],
    "rows": len(daily_opportunity),
    "columns": list(daily_opportunity.columns),
    "temporal_coverage": {
        "start_date": daily_opportunity["DATE"].min().date().isoformat(),
        "end_date": daily_opportunity["DATE"].max().date().isoformat(),
    },
    "spatial_aggregation": {
        "source_resolution": 7,
        "target_resolution": 7,
        "canonical_source_target_pairs": len(land_pairs),
        "grouped_basis_rows": len(daily_opportunity_basis),
        "weather_resolution": 5,
        "daylight_resolution": 4,
        "distance_bin_km": DAILY_DISTANCE_BIN_KM,
        "distance_basis": "H3 R7 source-target centroid great-circle distance",
    },
    "static_scope_totals": daily_static_scope_totals,
    "dynamic_variants": {
        "CORE_PHYSICAL": "static viewshed * mean visibility support * daylight fraction",
        "MIN_VIS_PHYSICAL": "static viewshed * minimum visibility support * daylight fraction",
        "EXTENDED_PHYSICAL": "core physical * wind support * precipitation support",
        "CORE_POPULATION": "core physical * primary population component * calendar effort weight",
        "EXTENDED_POPULATION": "extended physical * primary population component * calendar effort weight",
        "CORE_ACCESS": "core physical * population/access composite * calendar effort weight",
        "EXTENDED_ACCESS": "extended physical * population/access composite * calendar effort weight",
    },
    "parameters": {
        "visibility_transition_fraction": VISIBILITY_TRANSITION_FRACTION,
        "visibility_minimum_transition_km": VISIBILITY_MINIMUM_TRANSITION_KM,
        "wind_support_midpoint_ms": WIND_SUPPORT_MIDPOINT_MS,
        "wind_support_slope_ms": WIND_SUPPORT_SLOPE_MS,
        "precip_support_half_mm_day": PRECIP_SUPPORT_HALF_MM_DAY,
        "conditions_wind_exponent": CONDITIONS_WIND_EXPONENT,
        "conditions_precip_exponent": CONDITIONS_PRECIP_EXPONENT,
    },
    "approximation_validation": daily_approximation_validation.to_dict(
        orient="records"
    ),
    "coverage": {
        "minimum_core_physical": float(
            daily_opportunity["CORE_PHYSICAL_COVERAGE"].min()
        ),
        "minimum_core_population": float(
            daily_opportunity["CORE_POPULATION_COVERAGE"].min()
        ),
        "minimum_core_access": float(
            daily_opportunity["CORE_ACCESS_COVERAGE"].min()
        ),
    },
    "sightings": {
        **sightings_input,
        "reported_sightings": int(
            daily_opportunity["REPORTED_SIGHTING_COUNT"].sum()
        ),
        "dates_with_reported_sighting": int(
            daily_opportunity["HAS_REPORTED_SIGHTING"].sum()
        ),
        "zero_semantics": (
            "No public-release-eligible canonical record on the date; "
            "not confirmed whale absence."
        ),
        "used_to_construct_opportunity": False,
    },
    "correlation_contract": {
        "raw_daily": "correlation across daily regional rows",
        "within_month_anomaly": (
            "feature and response minus their mean within each year-month"
        ),
        "static_feature_treatment": (
            "Static totals have zero temporal variance and are retained as weighting "
            "contracts, not assigned a daily correlation."
        ),
        "full_window_spearman_sighting_count": (
            daily_primary_correlations.reset_index().to_dict(orient="records")
        ),
    },
    "artifacts": [
        {
            "path": str(DAILY_OPPORTUNITY_PATH.relative_to(REPO_ROOT)),
            "sha256": sha256_file(DAILY_OPPORTUNITY_PATH),
        },
        {
            "path": str(DAILY_CORRELATIONS_PATH.relative_to(REPO_ROOT)),
            "sha256": sha256_file(DAILY_CORRELATIONS_PATH),
        },
        {
            "path": str(DAILY_CORRELATION_PLOT_PATH.relative_to(REPO_ROOT)),
            "sha256": sha256_file(DAILY_CORRELATION_PLOT_PATH),
        },
    ],
    "known_limitations": [
        "The product is one regional row per date and does not estimate target-cell daily detection probability.",
        "Daily sighting zero means no eligible report in the canonical dataset, not confirmed whale absence.",
        "Sightings are downstream responses only; correlations do not establish causal observer effort or whale presence effects.",
        "Daily weather summaries do not identify conditions during actual viewing hours.",
        "HRRR source-cell wind is a proxy for sea state; Beaufort state and swell are not observed.",
        "Static viewshed and population totals have zero temporal variance; they enter daily features as spatial weights.",
        "Population and mapped-access weights are static census/access evidence, not observed daily visitation.",
        "Reporting adoption, direct observer hours, and ecological drivers of whale occurrence remain unmeasured.",
    ],
}
DAILY_OPPORTUNITY_METADATA_PATH.write_text(
    json.dumps(daily_opportunity_metadata, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)

comparison_metadata = {
    "schema_version": "0.3.0-prototype",
    "product": "analysis.sightings_vs_land_viewing_pressure_h3_r7",
    "status": "retrospective_association_only",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "pressure_artifact": {
        "path": str(TARGET_PRESSURE_PATH.relative_to(REPO_ROOT)),
        "sha256": sha256_file(TARGET_PRESSURE_PATH),
    },
    "sightings_input": sightings_input,
    "comparison_window": {
        "start_date": SIGHTINGS_START_DATE.date().isoformat(),
        "end_date": SIGHTINGS_END_DATE.date().isoformat(),
    },
    "primary_population_variant": PRIMARY_POPULATION_VARIANT,
    "shared_all_variants_target_cells": int(shared_support.sum()),
    "association_summary": association_summary.to_dict(orient="records"),
    "temporal_holdouts": temporal_holdout_inventory.to_dict(orient="records"),
    "temporal_holdout_association_summary": (
        temporal_holdout_association_summary.to_dict(orient="records")
    ),
    "interpretation": (
        "Associations describe reported sightings versus a land-viewing opportunity proxy; "
        "they do not estimate whale preference or causal presence effects."
    ),
}
SIGHTINGS_COMPARISON_METADATA_PATH.write_text(
    json.dumps(comparison_metadata, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)

print(f"Wrote {PROTOTYPE_PATH.relative_to(REPO_ROOT)}")
print(f"Wrote {SOURCE_TRANSPORT_ACCESS_PATH.relative_to(REPO_ROOT)}")
print(f"Wrote {SOURCE_TRANSPORT_SUMMARY_PATH.relative_to(REPO_ROOT)}")
print(f"Wrote {SOURCE_TRANSPORT_METADATA_PATH.relative_to(REPO_ROOT)}")
print(f"Wrote {POPULATION_TRAVEL_ORIGINS_PATH.relative_to(REPO_ROOT)}")
print(f"Wrote {SOURCE_POPULATION_TRAVEL_PATH.relative_to(REPO_ROOT)}")
print(f"Wrote {SOURCE_POPULATION_TRAVEL_SUMMARY_PATH.relative_to(REPO_ROOT)}")
print(f"Wrote {SOURCE_POPULATION_TRAVEL_METADATA_PATH.relative_to(REPO_ROOT)}")
print(f"Wrote {COVERAGE_PATH.relative_to(REPO_ROOT)}")
print(f"Wrote {METADATA_PATH.relative_to(REPO_ROOT)}")
print(f"Wrote {TARGET_PRESSURE_PATH.relative_to(REPO_ROOT)}")
print(f"Wrote {TARGET_PRESSURE_METADATA_PATH.relative_to(REPO_ROOT)}")
print(f"Wrote {DYNAMIC_SOURCE_SAMPLE_PATH.relative_to(REPO_ROOT)}")
print(f"Wrote {DYNAMIC_TARGET_SAMPLE_PATH.relative_to(REPO_ROOT)}")
print(f"Wrote {DYNAMIC_VIEWABILITY_METADATA_PATH.relative_to(REPO_ROOT)}")
print(f"Wrote {DAILY_OPPORTUNITY_PATH.relative_to(REPO_ROOT)}")
print(f"Wrote {DAILY_CORRELATIONS_PATH.relative_to(REPO_ROOT)}")
print(f"Wrote {DAILY_OPPORTUNITY_METADATA_PATH.relative_to(REPO_ROOT)}")
print(f"Wrote {DAILY_CORRELATION_PLOT_PATH.relative_to(REPO_ROOT)}")
print(f"Wrote {SIGHTINGS_COMPARISON_PATH.relative_to(REPO_ROOT)}")
print(f"Wrote {SIGHTINGS_COMPARISON_METADATA_PATH.relative_to(REPO_ROOT)}")
print(f"Wrote {SIGHTINGS_DECILES_PATH.relative_to(REPO_ROOT)}")
print(f"Wrote {SIGHTINGS_PLOT_PATH.relative_to(REPO_ROOT)}")
print(f"Wrote {POPULATION_CATCHMENT_SUMMARY_PATH.relative_to(REPO_ROOT)}")
print(f"Wrote {PRESSURE_VARIANT_ASSOCIATIONS_PATH.relative_to(REPO_ROOT)}")
print(f"Wrote {TEMPORAL_HOLDOUT_ASSOCIATIONS_PATH.relative_to(REPO_ROOT)}")
print(f"Wrote {SIGHTINGS_VARIANTS_PLOT_PATH.relative_to(REPO_ROOT)}")

## Next decisions

Before promotion into a package builder, replace the public OSRM demo snapshot with a versioned regional routing graph, refine the prototype H3 R4 population origins toward a validated spatial resolution, measure routing to mapped parking/access sites rather than H3 centroids, and validate ferry behavior. Then validate the visibility, wind, precipitation, road, and travel-time curves with target-cell and temporal holdouts, test additive versus access-gated formulas, promote the grouped daily calculation into a package builder, aggregate weekly at H3 R6 with reference-window scaling, and validate null propagation. The retrospective sightings association must remain downstream evaluation; population, population-weighted travel demand, road proximity, city travel, mapped public access, weather, daylight, and calendar must remain separately inspectable components throughout.